In [ ]:
# ============================================================
# 重要 BUG 记录（MERRA2 pressure-level O3 柱含量积分）
# ------------------------------------------------------------
# 旧版本代码在 get_lev_weights() 里存在一个关键错位 bug：
#
#   1）先对 lev 做了排序，用排序后的 lev 去计算 layer edges 和 overlap
#   2）但最后构造 DataArray 时，overlap 数值没有恢复到原始 lev 顺序，
#      却直接绑定回原始 lev 坐标
#
# 这样会导致：
#   - overlap 数值对应的是“排序后的层”
#   - lev 标签对应的是“原始文件层顺序”
#   - xarray 按 lev 标签对齐后，层权重会贴到错误的 pressure level 上
#
# 后果：
#   名义上的“30–70 hPa 积分”实际上没有用对 70/50/40/30 hPa 的厚度，
#   而是把部分权重错配到了 20/10 hPa 等错误层，导致 partial O3 column
#   被系统性高估。
#
# 这个 bug 的典型表现是：
#   - 30/40/50/70 hPa 各层 O3 climatology 看起来和 WACCM 差别不大
#   - 但 30–70 hPa partial column 却离谱地偏高（可高出 ~40–50 DU）
#
# 正确修复方法：
#   - lev 只在内部临时排序用于计算 overlap
#   - overlap 计算完成后，必须恢复到原始 lev 顺序
#   - 保证 overlap 数值与 lev 坐标标签严格一一对应
#
# 另外：
#   - MERRA2 原始 O3 这里按 kg/kg 处理
#   - 在做 DU 积分前，必须先乘 (M_air / M_O3) 转成 mol/mol
#
# 经验教训：
#   任何涉及 xarray 坐标对齐的权重计算，只要中途做过 sort/reindex，
#   都必须检查“权重数值”和“坐标标签”是否仍然严格对应。
#   否则结果可能数值上看起来合理，但物理上完全错层。
# ============================================================
import os
import glob
import numpy as np
import xarray as xr
from tqdm import tqdm
from joblib import Parallel, delayed

# ============================================================
# 1. 基础配置与路径
# ============================================================
DATA_DIR_O3 = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/O3"
OUT_ROOT    = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
EXP_NAME    = "MERRA2"

N_CORES = 10  # 建议根据磁盘 I/O 适当调整，通常 8-12 较好

PRESSURE_RANGES = [
    (30, 70,  "30_70hPa"),
    (1,  100, "1_100hPa"),
]

LAT_BAND = (60, 90)
AUTO_ENSURE_DAILY = True

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# ============================================================
# 2. 物理常数
# ============================================================
G      = 9.80665
M_AIR  = 28.964e-3
M_O3   = 47.9982e-3
NA     = 6.02214e23
DU_DEN = 2.687e20

# MERRA2 原始 O3: kg/kg -> mol/mol
KGKG_TO_MOLMOL = M_AIR / M_O3

# mol/mol -> DU
DU_FACTOR = NA / (G * M_AIR * DU_DEN)

# ============================================================
# 3. 修复后的 pressure-level overlap 权重
# ============================================================
def get_lev_weights(lev_hpa, p_top_hpa, p_bot_hpa):
    """
    对固定 pressure-level 数据：
    用 level center 构造 layer edges，
    再计算每层与目标 pressure range 的 overlap (Pa)。

    修复点：
    先按排序后的 lev 算 overlap，再还原回原始 lev 顺序，
    保证 overlap 数值与 lev 坐标严格一一对应。
    """
    lev_orig = np.asarray(lev_hpa, dtype=float)

    # 升序排序用于计算 layer edges
    sort_idx = np.argsort(lev_orig)
    lev_sorted = lev_orig[sort_idx]

    logp = np.log(lev_sorted)
    mids = np.exp(0.5 * (logp[:-1] + logp[1:]))

    edges = np.empty(lev_sorted.size + 1)
    edges[1:-1] = mids
    edges[0]  = lev_sorted[0]**2 / mids[0]
    edges[-1] = lev_sorted[-1]**2 / mids[-1]

    # hPa -> Pa
    p_layer_top = edges[:-1] * 100.0
    p_layer_bot = edges[1:]  * 100.0
    pT = float(p_top_hpa) * 100.0
    pB = float(p_bot_hpa) * 100.0

    overlap_sorted = np.maximum(
        0.0,
        np.minimum(p_layer_bot, pB) - np.maximum(p_layer_top, pT)
    )

    # 还原到原始 lev 顺序
    overlap_orig = np.empty_like(overlap_sorted)
    overlap_orig[sort_idx] = overlap_sorted

    return xr.DataArray(overlap_orig, coords={"lev": lev_hpa}, dims=("lev",))

# ============================================================
# 4. 单文件处理
# ============================================================
def process_single_year_file(file_path, weight_dict):
    try:
        with xr.open_dataset(file_path, decode_times=True) as ds:
            # 空间裁剪；不再强制 sortby("lev")，因为权重已和原始 lev 对齐
            da_o3 = ds["O3"].sel(lat=slice(LAT_BAND[0], LAT_BAND[1]))

            # MERRA2 原始 O3 是 kg/kg，直接强制转 mol/mol
            da_o3 = da_o3 * KGKG_TO_MOLMOL

            # 纬度权重
            weights_lat = np.cos(np.deg2rad(da_o3.lat))

            year_results = {}
            for tag, lev_w in weight_dict.items():
                # 垂直积分：sum( O3_molmol * overlap_Pa ) * DU_FACTOR
                column = (da_o3 * lev_w).sum(dim="lev") * DU_FACTOR

                # 极区平均
                ts = column.weighted(weights_lat).mean(dim=["lat", "lon"]).compute()
                year_results[tag] = ts

            return year_results

    except Exception as e:
        return f"Error {os.path.basename(file_path)}: {e}"

# ============================================================
# 5. 主程序
# ============================================================
def run_main():
    exp_out_dir = os.path.join(OUT_ROOT, EXP_NAME)
    os.makedirs(exp_out_dir, exist_ok=True)

    all_files = sorted(glob.glob(os.path.join(DATA_DIR_O3, "MERRA2.O3.*.nc")))
    if not all_files:
        print(f"No files found in {DATA_DIR_O3}")
        return

    # 读一个样例文件获取 lev 坐标
    with xr.open_dataset(all_files[0]) as ds_tmp:
        lev_coords = ds_tmp["lev"].values

    print("[INFO] Raw lev coordinates:")
    print(lev_coords)

    # 预计算各 pressure range 的 overlap 权重
    weight_dict = {
        tag: get_lev_weights(lev_coords, ptop, pbot)
        for ptop, pbot, tag in PRESSURE_RANGES
    }

    # 打印非零权重层，方便自检
    for ptop, pbot, tag in PRESSURE_RANGES:
        w = weight_dict[tag]
        nz = w.where(w > 0, drop=True)
        print(f"\n[INFO] Nonzero weights for {tag} ({ptop}-{pbot} hPa):")
        print(nz)

    print(f"\n🚀 开始并行处理 MERRA2 O3 ({N_CORES} cores)...")
    results = Parallel(n_jobs=N_CORES)(
        delayed(process_single_year_file)(f, weight_dict)
        for f in tqdm(all_files)
    )

    clean_results = [r for r in results if isinstance(r, dict)]
    error_results = [r for r in results if isinstance(r, str)]

    if error_results:
        print("\n[WARN] Some files failed:")
        for msg in error_results:
            print("  ", msg)

    if not clean_results:
        print("[ERROR] No valid results generated.")
        return

    # ========================================================
    # 汇总输出
    # ========================================================
    for _, _, tag in PRESSURE_RANGES:
        print(f"\n📦 正在汇总 {tag}...")

        combined = xr.concat([r[tag] for r in clean_results], dim="time").sortby("time")

        if AUTO_ENSURE_DAILY:
            # 若同一天有多个时次，自动 daily mean
            if len(np.unique(combined.time.dt.date)) < len(combined.time):
                combined = combined.resample(time="1D").mean()

        # 保存 nc
        out_nc = os.path.join(exp_out_dir, f"O3_pc_{EXP_NAME}_{tag}.nc")
        combined.name = "O3_column"
        combined.attrs = {
            "units": "DU",
            "lat_band": f"{LAT_BAND[0]}-{LAT_BAND[1]}N",
            "source_o3_unit": "kg/kg",
            "converted_to": "mol/mol before column integration",
            "method": "pressure-level overlap integration (fixed bug)",
            "pressure_range": tag
        }
        combined.to_netcdf(out_nc)
        print(f"  Saved: {out_nc}")

        # ====================================================
        # 极值分析：3–4 月最小值
        # ====================================================
        o3_5d = combined.rolling(time=5, center=True, min_periods=5).mean()
        spring = o3_5d.sel(time=o3_5d.time.dt.month.isin([3, 4]))
        spring_min = spring.groupby("time.year").min()

        # 过滤数据不足 58 天的年份
        counts = spring.groupby("time.year").count()
        valid_years = counts.year.where(counts >= 58, drop=True)
        spring_min = spring_min.sel(year=valid_years)

        sorted_years = spring_min.sortby(spring_min).year.values

        lowest10_txt = os.path.join(exp_out_dir, f"lowest10_years_sorted_{tag}.txt")
        np.savetxt(lowest10_txt, sorted_years[:10], fmt="%04d")
        print(f"  Saved: {lowest10_txt}")

        n_25pct = max(1, int(0.25 * len(sorted_years)))
        low25_txt = os.path.join(exp_out_dir, f"low25pct_years_{tag}.txt")
        np.savetxt(low25_txt, sorted_years[:n_25pct], fmt="%04d")
        print(f"  Saved: {low25_txt}")

        print(f"  Mean DU = {float(combined.mean().values):.3f}")
        print(f"  Min  DU = {float(combined.min().values):.3f}")
        print(f"  Max  DU = {float(combined.max().values):.3f}")

if __name__ == "__main__":
    run_main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
from netCDF4 import Dataset, num2date
from tqdm.auto import tqdm

# ============================================================
# MERRA2 T_min_60_90N calculation: netCDF4 low-level version
# ------------------------------------------------------------
# T(time, lev, lat, lon)
# -> zonal mean over lon
# -> select 60–90N
# -> minimum over lat
# -> output: T_min_60_90N(time, lev)
# ============================================================

MERRA2_T_DIR = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/T"

OUT_DIR = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data/MERRA2"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_FILE = os.path.join(OUT_DIR, "T_min_60_90N_MERRA2.nc")

FILE_PATTERN = "MERRA2.T.*.nc"

LAT_BAND = (60.0, 90.0)

# 每次读多少个 time step
TIME_CHUNK_SIZE = 10

OVERWRITE = True


# ============================================================
# Helper functions
# ============================================================

def find_coord_name(nc, candidates):
    for name in candidates:
        if name in nc.variables:
            return name
    return None


def find_temp_var_name(nc):
    candidates = ["T", "T_PL", "temperature", "Temperature", "air_temperature"]
    for name in candidates:
        if name in nc.variables:
            return name

    # fallback: 找一个四维变量
    for name, var in nc.variables.items():
        if hasattr(var, "dimensions") and len(var.dimensions) >= 3:
            if name not in ["time", "lev", "plev", "level", "lat", "lon"]:
                return name

    raise ValueError(f"Cannot detect temperature variable. Variables: {list(nc.variables.keys())}")


def lev_to_pa(lev_vals):
    lev_vals = np.asarray(lev_vals, dtype=float)
    if np.nanmax(lev_vals) <= 2000.0:
        return lev_vals * 100.0, "hPa"
    else:
        return lev_vals, "Pa"


def get_lat_indices(lat_vals, lat_band):
    lat_vals = np.asarray(lat_vals, dtype=float)
    lat0, lat1 = lat_band

    mask = (lat_vals >= lat0) & (lat_vals <= lat1)
    idx = np.where(mask)[0]

    if idx.size == 0:
        raise ValueError(f"No latitudes found in {lat_band}. lat range = {lat_vals.min()}–{lat_vals.max()}")

    return idx


def convert_times(nc, time_name):
    time_var = nc.variables[time_name]
    time_vals = time_var[:]

    units = getattr(time_var, "units", None)
    calendar = getattr(time_var, "calendar", "standard")

    if units is None:
        raise ValueError("Time variable has no units attribute; cannot decode time.")

    times = num2date(
        time_vals,
        units=units,
        calendar=calendar,
        only_use_cftime_datetimes=True,
        only_use_python_datetimes=False
    )

    return np.asarray(times)


def read_one_file_lowlevel(file_path, file_index, n_files):
    basename = os.path.basename(file_path)

    print(f"\n[OPEN {file_index}/{n_files}] {basename}", flush=True)

    with Dataset(file_path, mode="r") as nc:
        var_name = find_temp_var_name(nc)
        time_name = find_coord_name(nc, ["time", "TIME"])
        lev_name  = find_coord_name(nc, ["lev", "plev", "level", "lev_p"])
        lat_name  = find_coord_name(nc, ["lat", "latitude", "LAT"])
        lon_name  = find_coord_name(nc, ["lon", "longitude", "LON"])

        if time_name is None:
            raise ValueError(f"No time variable found in {basename}")
        if lev_name is None:
            raise ValueError(f"No lev variable found in {basename}")
        if lat_name is None:
            raise ValueError(f"No lat variable found in {basename}")
        if lon_name is None:
            raise ValueError(f"No lon variable found in {basename}")

        Tvar = nc.variables[var_name]

        print(f"[INFO] var={var_name}, dims={Tvar.dimensions}, shape={Tvar.shape}", flush=True)
        print(f"[INFO] coords: time={time_name}, lev={lev_name}, lat={lat_name}, lon={lon_name}", flush=True)

        dims = Tvar.dimensions

        # 确认维度位置
        time_axis = dims.index(time_name)
        lev_axis  = dims.index(lev_name)
        lat_axis  = dims.index(lat_name)
        lon_axis  = dims.index(lon_name)

        if time_axis != 0:
            raise ValueError(
                f"This script assumes time is first dimension. "
                f"Found dims={dims}. Need small modification."
            )

        lev_vals_raw = nc.variables[lev_name][:]
        lev_pa, lev_unit_guess = lev_to_pa(lev_vals_raw)

        lat_vals = nc.variables[lat_name][:]
        lat_idx = get_lat_indices(lat_vals, LAT_BAND)

        times = convert_times(nc, time_name)
        n_time = len(times)
        n_lev = len(lev_pa)

        print(f"[INFO] n_time={n_time}, n_lev={n_lev}, n_lat_selected={len(lat_idx)}", flush=True)
        print(f"[INFO] lev unit guess={lev_unit_guess}; lev Pa range={np.nanmin(lev_pa)}–{np.nanmax(lev_pa)}", flush=True)

        chunk_results = []

        # 只支持最常见的 MERRA2 维度顺序: time, lev, lat, lon
        # 你的截图显示 raw dims 是 ('time', 'lev', 'lat', 'lon')，所以这里正好匹配。
        if dims != (time_name, lev_name, lat_name, lon_name):
            raise ValueError(
                f"Unexpected dimension order: {dims}. "
                f"Expected {(time_name, lev_name, lat_name, lon_name)}."
            )

        iterator = range(0, n_time, TIME_CHUNK_SIZE)

        for i0 in tqdm(
            iterator,
            total=int(np.ceil(n_time / TIME_CHUNK_SIZE)),
            desc=f"[{file_index:02d}/{n_files:02d}] {basename}",
            dynamic_ncols=True,
            leave=True
        ):
            i1 = min(i0 + TIME_CHUNK_SIZE, n_time)

            # 读取一个小块: time_chunk x lev x selected_lat x lon
            arr = Tvar[i0:i1, :, lat_idx, :]

            # masked array -> nan array
            arr = np.asarray(arr.filled(np.nan) if np.ma.isMaskedArray(arr) else arr, dtype=np.float32)

            # zonal mean over lon
            arr_zm = np.nanmean(arr, axis=3)  # time x lev x lat

            # minimum over selected lat
            arr_min = np.nanmin(arr_zm, axis=2)  # time x lev

            chunk_results.append(arr_min.astype(np.float32))

        data_one = np.concatenate(chunk_results, axis=0)

        da_one = xr.DataArray(
            data_one,
            coords={
                "time": times,
                "lev": lev_pa.astype(np.float64),
            },
            dims=("time", "lev"),
            name="T_min_60_90N",
        )

        da_one.attrs["units"] = "K"
        da_one.attrs["description"] = (
            "Polar-cap minimum temperature. "
            "Computed from zonal-mean T, then minimum over 60-90N "
            "at each time and pressure level."
        )
        da_one.attrs["lat_band"] = "60-90N"
        da_one.attrs["method"] = "zonal mean over longitude, then minimum over latitude"
        da_one.attrs["source_file"] = basename
        da_one.attrs["original_vertical_unit_guess"] = lev_unit_guess
        da_one["lev"].attrs["units"] = "Pa"

        print(
            f"[OK] {basename}: output shape={da_one.shape}, "
            f"min={float(np.nanmin(data_one)):.2f} K, "
            f"max={float(np.nanmax(data_one)):.2f} K",
            flush=True
        )

        return da_one


def main():
    if os.path.exists(OUT_FILE) and not OVERWRITE:
        print(f"[SKIP] Output exists: {OUT_FILE}")
        return

    files = sorted(glob.glob(os.path.join(MERRA2_T_DIR, FILE_PATTERN)))

    if not files:
        raise FileNotFoundError(f"No files found: {os.path.join(MERRA2_T_DIR, FILE_PATTERN)}")

    print(f"[INFO] Found {len(files)} MERRA2 T files.", flush=True)
    print(f"[INFO] First file: {files[0]}", flush=True)
    print(f"[INFO] Last  file: {files[-1]}", flush=True)
    print(f"[INFO] Output: {OUT_FILE}", flush=True)
    print(f"[INFO] TIME_CHUNK_SIZE = {TIME_CHUNK_SIZE}", flush=True)

    results = []

    for idx, f in enumerate(
        tqdm(files, desc="All MERRA2 T files", dynamic_ncols=True, leave=True),
        start=1
    ):
        da_one = read_one_file_lowlevel(f, idx, len(files))
        results.append(da_one)

    print("\n[INFO] Concatenating all files...", flush=True)

    out = xr.concat(results, dim="time").sortby("time")

    out.name = "T_min_60_90N"
    out.attrs["units"] = "K"
    out.attrs["description"] = (
        "MERRA2 polar-cap minimum temperature. "
        "Computed from zonal-mean T, then minimum over 60-90N."
    )
    out.attrs["lat_band"] = "60-90N"
    out.attrs["method"] = "zonal mean over longitude, then minimum over latitude"
    out.attrs["note"] = "Processed with netCDF4 file-by-file, time-chunk-by-time-chunk."

    out["lev"].attrs["units"] = "Pa"

    print("[INFO] Final DataArray:", flush=True)
    print(out, flush=True)

    print("[INFO] Saving NetCDF...", flush=True)

    encoding = {
        "T_min_60_90N": {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32",
        }
    }

    out = out.astype("float32")
    out.to_netcdf(OUT_FILE, encoding=encoding)

    print("✅ Done.", flush=True)
    print(f"Saved: {OUT_FILE}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import os

# --- 配置路径 ---
# 请确保此路径与你之前代码中的 OUT_ROOT 和 EXP_NAME 一致
RESULT_DIR = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data/MERRA2"
FILE_NAME  = "O3_pc_MERRA2_30_70hPa.nc"  # 或者 "O3_pc_MERRA2_1_100hPa.nc"
FILE_PATH  = os.path.join(RESULT_DIR, FILE_NAME)

def diagnostic_o3():
    if not os.path.exists(FILE_PATH):
        print(f"错误：找不到文件 {FILE_PATH}")
        return

    # 1. 加载数据
    ds = xr.open_dataset(FILE_PATH)
    da = ds["O3_column"]
    
    print("="*60)
    print(f"文件: {FILE_NAME}")
    print(f"数据维度: {da.dims}")
    print(f"时间范围: {da.time.values[0]} 至 {da.time.values[-1]}")
    
    # 2. 统计摘要
    print("-" * 20 + " 数值统计 " + "-" * 20)
    print(f"最大值: {da.max().values:.2f} DU")
    print(f"最小值: {da.min().values:.2f} DU")
    print(f"平均值: {da.mean().values:.2f} DU")
    
    # 3. 检查是否存在异常值 (例如负值或天文数字)
    if da.min() < 0:
        print("警告：检测到负值！请检查单位转换逻辑。")
    if da.max() > 1000:
        print("警告：数值过大 (>1000 DU)！请检查积分公式中的常数项。")

    # 4. 打印前 10 天的数据看一眼
    print("-" * 20 + " 前10天样本 " + "-" * 20)
    print(da.head(10).to_series())

    # 5. 快速可视化
    plt.figure(figsize=(12, 5))
    
    # 绘制全时间序列
    plt.subplot(1, 2, 1)
    da.plot()
    plt.title(f"Full Timeseries: {FILE_NAME}")
    plt.ylabel("O3 Column (DU)")

    # 绘制季节性循环 (Climatology)
    plt.subplot(1, 2, 2)
    da.groupby("time.month").mean().plot(marker='o')
    plt.title("Seasonal Cycle (Monthly Mean)")
    plt.ylabel("DU")
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    # 6. 检查特定的极端年份 (可选)
    # 比如 2011 或 2020 年通常是北极臭氧低值年
    try:
        sample_yr = "2020"
        spring_min = da.sel(time=sample_yr).min().values
        print(f"--- {sample_yr} 年最低值: {spring_min:.2f} DU ---")
    except:
        pass

if __name__ == "__main__":
    diagnostic_o3()

In [ ]:
# ============================================================
# Block C (MERRA2 Scatter Pro): 可定制时间窗与统计维度的 EP-O3 分析
# 目标：
# 1) 读取 MERRA2 EP flux daily series 与 MERRA2 O3 partial-column series
# 2) 支持 DJF / JF / MA / NDJ 等季节聚合
# 3) 支持 mean / min
# 4) 画 standardized scatter，标出 lowest-O3 years
#
# 注：
# - 不需要 bridge year 处理（MERRA2 没有 001/002 拼接问题）
# - 自动兼容 Fz 文件里 plev 是 100 hPa 还是 10000 Pa
# ============================================================

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- Paths ----------------
FILE_MERRA_EP = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"
DATA_BASE     = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE     = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

EXP_NAME = "MERRA2"

# ---------------- Config ----------------
MONTH_MAP = {
    'J': 1, 'F': 2, 'M': 3, 'A': 4, 'm': 5, 'j': 6,
    'x': 7, 'a': 8, 'S': 9, 'O': 10, 'N': 11, 'D': 12
}

SHORT_MAP = {
    "DJF": [12, 1, 2],
    "JF":  [1, 2],
    "FM":  [2, 3],
    "MA":  [3, 4],
    "NDJ": [11, 12, 1],
    "JFM": [1, 2, 3],
    "OND": [10, 11, 12]
}

# 若 Fz 文件里还有 lat 维，就做 40-80N 平均
LAT_BAND = (40, 80)

# 是否在 O3 上先做 5-day running mean 再提取季节统计
APPLY_O3_5D = True


# ============================================================
# Helpers
# ============================================================
def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})

def ensure_daily_if_needed(da):
    """
    如果同一天有多个样本，就 resample 到 daily mean。
    正常情况下你的 MERRA2 已经是日平均，因此这里只做保护。
    """
    try:
        t = pd.to_datetime(da.time.values)
        n_total = len(t)
        n_unique_day = len(pd.Index(t.normalize()).unique())
        if n_unique_day < n_total:
            print("  Detected sub-daily timestamps -> resampling to daily mean.")
            return da.resample(time="1D").mean()
    except Exception as e:
        print(f"  Daily-check skipped: {e}")

    return da

def is_cross_year_season(season_str):
    months = SHORT_MAP.get(season_str, [12, 1, 2])
    return any(m in [11, 12] for m in months) and any(m in [1, 2] for m in months)

def get_seasonal_data(da, season_str, method="mean"):
    """
    返回按 year 聚合后的季节统计序列（DataArray, dim='year'）
    season_str: DJF / JF / MA / ...
    method: mean / min
    """
    months = SHORT_MAP.get(season_str, [12, 1, 2])

    ts = da.to_series().sort_index()
    all_years = np.unique(ts.index.year)

    results = {}
    cross_year = is_cross_year_season(season_str)

    for yr in all_years:
        try:
            if cross_year:
                early_months = [m for m in months if m < 6]
                late_months  = [m for m in months if m >= 6]

                parts = []
                for m in late_months:
                    parts.append(ts[f"{int(yr-1):04d}-{m:02d}"])
                for m in early_months:
                    parts.append(ts[f"{int(yr):04d}-{m:02d}"])

                combined = pd.concat(parts)
            else:
                combined = pd.concat([ts[f"{int(yr):04d}-{m:02d}"] for m in months])

            combined = combined.dropna()

            # 每个月至少 ~28 天
            if len(combined) < len(months) * 28:
                continue

            if method == "mean":
                val = combined.mean()
            elif method == "min":
                val = combined.min()
            else:
                raise ValueError(f"Unsupported method: {method}")

            results[int(yr)] = val

        except Exception:
            continue

    if len(results) == 0:
        return xr.DataArray(
            np.array([], dtype=float),
            coords={"year": np.array([], dtype=int)},
            dims=("year",),
            name=f"{season_str}_{method}"
        )

    years = np.array(sorted(results.keys()), dtype=int)
    vals  = np.array([results[y] for y in years], dtype=float)

    return xr.DataArray(
        vals,
        coords={"year": years},
        dims=("year",),
        name=f"{season_str}_{method}"
    )

def standardize_da(da):
    mu = float(da.mean().values)
    sd = float(da.std().values)
    if (not np.isfinite(sd)) or (sd == 0):
        raise ValueError("Standard deviation is zero or NaN; cannot standardize.")
    return (da - mu) / sd

def select_100hpa_level(da_plev):
    """
    自动兼容 plev 是 Pa 还是 hPa：
    - 若 plev 最大值 > 2000，视为 Pa，目标取 10000
    - 否则视为 hPa，目标取 100
    """
    pvals = np.asarray(da_plev["plev"].values, dtype=float)
    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0  # Pa
        unit_guess = "Pa"
    else:
        target = 100.0    # hPa
        unit_guess = "hPa"

    out = da_plev.sel(plev=target, method="nearest")
    print(f"  Selected 100 hPa level using target={target:g} ({unit_guess} interpretation).")
    return out

def dynamic_axis_limits(x, y, pad=0.4):
    vals = np.concatenate([np.asarray(x), np.asarray(y)])
    vmax = np.nanmax(np.abs(vals))
    vmax = max(vmax + pad, 2.0)
    return (-vmax, vmax)


# ============================================================
# Plotting
# ============================================================
def plot_custom_relationship(exp_name, ep_nc, o3_nc, txt_path,
                             fz_season="DJF", fz_method="mean",
                             o3_season="MA", o3_method="min",
                             lat_band=(40, 80), apply_o3_5d=True):

    print(f"\n[{exp_name}] Analyzing Fz({fz_season}, {fz_method}) vs O3({o3_season}, {o3_method})...")

    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        print("  Missing file(s). Skip.")
        return

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    # ---------------- Fz ----------------
    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"

    fz_raw = ds_ep[var_fz]

    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    # 若还存在 lat 维，则做 40-80N cos(lat) 加权平均
    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    fz_raw = ensure_daily_if_needed(fz_raw)

    # 与你原脚本保持一致：取 -Fz
    x_source = -1.0 * fz_raw
    fz_stat = get_seasonal_data(x_source, fz_season, method=fz_method)

    # ---------------- O3 ----------------
    da_o3 = ensure_daily_if_needed(da_o3)

    if apply_o3_5d:
        da_o3 = da_o3.rolling(time=5, center=True, min_periods=5).mean()

    o3_stat = get_seasonal_data(da_o3, o3_season, method=o3_method)

    # ---------------- Align years ----------------
    common_years = np.intersect1d(fz_stat.year.values, o3_stat.year.values)

    if len(common_years) < 3:
        print("  Too few common years after filtering. Skip.")
        ds_ep.close()
        return

    x_raw = fz_stat.sel(year=common_years)
    y_raw = o3_stat.sel(year=common_years)

    valid = np.isfinite(x_raw.values) & np.isfinite(y_raw.values)
    x_raw = x_raw.sel(year=x_raw.year.values[valid])
    y_raw = y_raw.sel(year=y_raw.year.values[valid])

    if len(x_raw.year) < 3:
        print("  Too few valid years after NaN filtering. Skip.")
        ds_ep.close()
        return

    # ---------------- Standardize ----------------
    x = standardize_da(x_raw)
    y = standardize_da(y_raw)

    # ---------------- Plot ----------------
    plt.figure(figsize=(7.2, 6.2), dpi=140)
    ax = plt.gca()

    ax.grid(True, linestyle=":", alpha=0.5)
    ax.axhline(0, color="k", lw=0.9, alpha=0.5)
    ax.axvline(0, color="k", lw=0.9, alpha=0.5)

    key_colors = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
    key_years = np.loadtxt(txt_path, dtype=int)[:3] if os.path.exists(txt_path) else np.array([], dtype=int)
    key_years = np.atleast_1d(key_years)

    # 背景点
    bg_mask = ~np.isin(x.year.values, key_years)
    ax.scatter(
        x.values[bg_mask],
        y.values[bg_mask],
        color="black",
        alpha=0.80,
        s=42,
        edgecolors="none",
        label="Other Years"
    )

    # 高亮关键年
    for i, yr in enumerate(key_years):
        if yr in x.year.values:
            idx = np.where(x.year.values == yr)[0][0]
            ax.scatter(
                x.values[idx],
                y.values[idx],
                color=key_colors[i % len(key_colors)],
                s=54,
                edgecolors="k",
                linewidths=0.8,
                label=f"Year {int(yr):04d}",
                zorder=10
            )

    # 相关系数
    r, p = pearsonr(x.values, y.values)

    ax.text(
        0.05, 0.95,
        f"r = {r:.3f}\np = {p:.2e}\nN = {len(x)}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="0.7")
    )

    # 动态坐标范围
    xlim = dynamic_axis_limits(x.values, y.values, pad=0.5)
    ylim = dynamic_axis_limits(x.values, y.values, pad=0.5)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.set_xlabel(f"{fz_season} $-F_z$ at 100 hPa ({fz_method}, $\\sigma$)", fontsize=11)
    ax.set_ylabel(f"{o3_season} O$_3$ Column ({o3_method}, $\\sigma$)", fontsize=11)
    ax.set_title(f"{exp_name} | {fz_season} $-F_z$ vs {o3_season} O$_3$", loc="left", fontweight="bold", fontsize=12)

    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)

    out_dir = os.path.join(PLOT_BASE, exp_name, "EP_O3_Custom")
    os.makedirs(out_dir, exist_ok=True)

    save_path = os.path.join(
        out_dir,
        f"Scatter_{fz_season}_{fz_method}_vs_{o3_season}_{o3_method}.png"
    )

    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.show()
    print("  Saved:", save_path)

    ds_ep.close()


# ============================================================
# Run examples
# ============================================================
configs = [
    {
        "fz_season": "DJF",
        "fz_method": "mean",
        "o3_season": "MA",
        "o3_method": "min"
    },
    {
        "fz_season": "JF",
        "fz_method": "mean",
        "o3_season": "MA",
        "o3_method": "min"
    }
]

tasks = [
    (
        "MERRA2",
        FILE_MERRA_EP,
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        os.path.join(DATA_BASE, "MERRA2", "lowest10_years_sorted_30_70hPa.txt")
    ),
    # 如果你也想直接顺手画 1-100 hPa，可把下面这组打开
    # (
    #     "MERRA2",
    #     FILE_MERRA_EP,
    #     os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_1_100hPa.nc"),
    #     os.path.join(DATA_BASE, "MERRA2", "lowest10_years_sorted_1_100hPa.txt")
    # )
]

for name, ep, o3, txt in tasks:
    for cfg in configs:
        plot_custom_relationship(
            name,
            ep,
            o3,
            txt,
            lat_band=LAT_BAND,
            apply_o3_5d=APPLY_O3_5D,
            **cfg
        )

In [ ]:
# ============================================================
# Block D (MERRA2): Sliding-window correlation curves (ALL years)
# 目标：
# 1) MERRA2 单独分析
# 2) 90-day 和 60-day 分开作图
# 3) 不需要 bridge year 修正
# 4) 自动兼容 Fz 文件里 plev 是 100 hPa 还是 10000 Pa
# ============================================================

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- Paths ----------------
FILE_MERRA_EP = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"
DATA_BASE     = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE     = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

EXP_NAME = "MERRA2"

# ---------------- Config ----------------
WINDOW_DAYS_LIST = [90, 60]
LAT_BAND = (40, 80)
APPLY_O3_5D = True
MIN_N = 5

# 默认与前面 scatter 保持一致：30-70 hPa
O3_FILE = os.path.join(DATA_BASE, EXP_NAME, "O3_pc_MERRA2_30_70hPa.nc")


# ============================================================
# Helpers
# ============================================================
def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})

def ensure_daily_if_needed(da):
    """
    若同一天有多个样本，则 resample 到 daily mean。
    正常情况下 MERRA2 已经是 daily mean，这里只是保护。
    """
    try:
        t = pd.to_datetime(da.time.values)
        n_total = len(t)
        n_unique_day = len(pd.Index(t.normalize()).unique())
        if n_unique_day < n_total:
            print("  Detected sub-daily timestamps -> resampling to daily mean.")
            return da.resample(time="1D").mean()
    except Exception as e:
        print(f"  Daily-check skipped: {e}")
    return da

def select_100hpa_level(da_plev):
    """
    自动兼容 plev 是 Pa 还是 hPa：
    - 若最大值 > 2000，视为 Pa，取 10000
    - 否则视为 hPa，取 100
    """
    pvals = np.asarray(da_plev["plev"].values, dtype=float)
    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0
        unit_guess = "Pa"
    else:
        target = 100.0
        unit_guess = "hPa"

    out = da_plev.sel(plev=target, method="nearest")
    print(f"  Selected 100 hPa level using target={target:g} ({unit_guess} interpretation).")
    return out

def build_window_starts():
    """
    窗口起点：12-01 到 02-01，逐日滑动
    """
    starts = []
    for d in range(1, 32):
        starts.append((12, d))
    for d in range(1, 32):
        starts.append((1, d))
    starts.append((2, 1))
    return starts

def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    """
    返回按 year 的 MA minimum O3（目标变量）
    """
    o3_da = ensure_daily_if_needed(o3_da)

    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    ts = o3_da.to_series().sort_index()
    all_years = np.unique(ts.index.year)

    results = {}
    for yr in all_years:
        try:
            seg = pd.concat([
                ts[f"{int(yr):04d}-03"],
                ts[f"{int(yr):04d}-04"]
            ]).dropna()

            # 3-4 月至少 ~58 个有效日
            if len(seg) < 58:
                continue

            results[int(yr)] = seg.min()
        except Exception:
            continue

    if len(results) == 0:
        return xr.DataArray(
            np.array([], dtype=float),
            coords={"year": np.array([], dtype=int)},
            dims=("year",),
            name="MA_min_O3"
        )

    years = np.array(sorted(results.keys()), dtype=int)
    vals  = np.array([results[y] for y in years], dtype=float)

    return xr.DataArray(
        vals,
        coords={"year": years},
        dims=("year",),
        name="MA_min_O3"
    )

def get_window_mean_series(da, start_month, start_day, window_days):
    """
    对每个 target year，取固定长度滑动窗口的 -Fz 均值
    - 若起点在 12 月：窗口从 (year-1)-12-dd 开始
    - 若起点在 1/2 月：窗口从 year-mm-dd 开始
    """
    da = ensure_daily_if_needed(da)
    ts = da.to_series().sort_index()
    all_years = np.unique(ts.index.year)

    results = {}
    for yr in all_years:
        try:
            start_year = int(yr) - 1 if start_month == 12 else int(yr)
            start_key = f"{start_year:04d}-{start_month:02d}-{start_day:02d}"

            seg = ts[start_key:]
            if len(seg) < window_days:
                continue

            win = seg.iloc[:window_days].dropna()
            if len(win) < window_days:
                continue

            results[int(yr)] = win.mean()
        except Exception:
            continue

    if len(results) == 0:
        return xr.DataArray(
            np.array([], dtype=float),
            coords={"year": np.array([], dtype=int)},
            dims=("year",),
            name=f"window_{window_days}d_mean"
        )

    years = np.array(sorted(results.keys()), dtype=int)
    vals  = np.array([results[y] for y in years], dtype=float)

    return xr.DataArray(
        vals,
        coords={"year": years},
        dims=("year",),
        name=f"window_{window_days}d_mean"
    )

def compute_r_curve_merra_all(window_days, lat_band=(40, 80), apply_o3_5d=True, min_n=5):
    if not os.path.exists(FILE_MERRA_EP):
        print(f"[{EXP_NAME}] Missing EP file.")
        return None, None

    if not os.path.exists(O3_FILE):
        print(f"[{EXP_NAME}] Missing O3 file: {O3_FILE}")
        return None, None

    ds_ep = xr.open_dataset(FILE_MERRA_EP)
    da_o3 = xr.open_dataarray(O3_FILE)

    # ---------------- Fz ----------------
    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"
    fz_raw = ds_ep[var_fz]

    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    fz_raw = ensure_daily_if_needed(fz_raw)

    # 与 scatter 保持一致：使用 -Fz
    x_source = -1.0 * fz_raw

    # ---------------- O3 ----------------
    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    starts = build_window_starts()
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]
    r_list = []

    for start_month, start_day in starts:
        x_raw = get_window_mean_series(x_source, start_month, start_day, window_days)

        common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
        if len(common_years) < min_n:
            r_list.append(np.nan)
            continue

        x_use = x_raw.sel(year=common_years)
        y_use = y_raw.sel(year=common_years)

        valid = np.isfinite(x_use.values) & np.isfinite(y_use.values)
        x_use = x_use.sel(year=x_use.year.values[valid])
        y_use = y_use.sel(year=y_use.year.values[valid])

        if len(x_use.year) < min_n:
            r_list.append(np.nan)
            continue

        try:
            r, _ = pearsonr(x_use.values, y_use.values)
        except Exception:
            r = np.nan

        r_list.append(r)

    ds_ep.close()
    return labels, r_list

def plot_single_curve(labels, r_list, window_days, suffix="ALL"):
    out_dir = os.path.join(PLOT_BASE, EXP_NAME, "EP_O3_SlidingR")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5), dpi=140)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels))
    plt.plot(x, r_list, lw=2.2, marker="o", ms=3)

    tick_idx = np.arange(0, len(labels), 5)
    plt.xticks(tick_idx, [labels[i] for i in tick_idx], rotation=45)

    plt.ylim(-1, 1)
    plt.xlim(0, len(labels) - 1)
    plt.ylabel("Pearson r")
    plt.xlabel("Window start date (MM-DD)")
    plt.title(
        f"{EXP_NAME} | {window_days}-day sliding-window correlation with MA minimum O$_3$ ({suffix})",
        loc="left",
        fontweight="bold"
    )

    plt.tight_layout()

    save_path = os.path.join(out_dir, f"SlidingR_{window_days}d_vs_MAmin_{suffix}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)


# ---------------- Run ----------------
for wd in WINDOW_DAYS_LIST:
    labels, r_list = compute_r_curve_merra_all(
        window_days=wd,
        lat_band=LAT_BAND,
        apply_o3_5d=APPLY_O3_5D,
        min_n=MIN_N
    )
    if labels is not None:
        plot_single_curve(labels, r_list, window_days=wd, suffix="ALL")

In [ ]:
# ============================================================
# Block E (MERRA2): Sliding-window correlation curves (LOW 25% years)
# 目标：
# 1) MERRA2 单独分析
# 2) 仅使用 low25pct_years_30_70hPa
# 3) 90-day 和 60-day 分开作图
# 4) 不需要 bridge year 修正
# ============================================================

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- Paths ----------------
FILE_MERRA_EP = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"
DATA_BASE     = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE     = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

EXP_NAME = "MERRA2"

# ---------------- Config ----------------
WINDOW_DAYS_LIST = [90, 60]
LAT_BAND = (40, 80)
APPLY_O3_5D = True
MIN_N = 5

# 默认与前面 scatter 保持一致：30-70 hPa
O3_FILE    = os.path.join(DATA_BASE, EXP_NAME, "O3_pc_MERRA2_30_70hPa.nc")
LOW25_FILE = os.path.join(DATA_BASE, EXP_NAME, "low25pct_years_30_70hPa.txt")


# ============================================================
# Helpers
# ============================================================
def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})

def ensure_daily_if_needed(da):
    try:
        t = pd.to_datetime(da.time.values)
        n_total = len(t)
        n_unique_day = len(pd.Index(t.normalize()).unique())
        if n_unique_day < n_total:
            print("  Detected sub-daily timestamps -> resampling to daily mean.")
            return da.resample(time="1D").mean()
    except Exception as e:
        print(f"  Daily-check skipped: {e}")
    return da

def select_100hpa_level(da_plev):
    pvals = np.asarray(da_plev["plev"].values, dtype=float)
    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0
        unit_guess = "Pa"
    else:
        target = 100.0
        unit_guess = "hPa"

    out = da_plev.sel(plev=target, method="nearest")
    print(f"  Selected 100 hPa level using target={target:g} ({unit_guess} interpretation).")
    return out

def build_window_starts():
    starts = []
    for d in range(1, 32):
        starts.append((12, d))
    for d in range(1, 32):
        starts.append((1, d))
    starts.append((2, 1))
    return starts

def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    o3_da = ensure_daily_if_needed(o3_da)

    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    ts = o3_da.to_series().sort_index()
    all_years = np.unique(ts.index.year)

    results = {}
    for yr in all_years:
        try:
            seg = pd.concat([
                ts[f"{int(yr):04d}-03"],
                ts[f"{int(yr):04d}-04"]
            ]).dropna()

            if len(seg) < 58:
                continue

            results[int(yr)] = seg.min()
        except Exception:
            continue

    if len(results) == 0:
        return xr.DataArray(
            np.array([], dtype=float),
            coords={"year": np.array([], dtype=int)},
            dims=("year",),
            name="MA_min_O3"
        )

    years = np.array(sorted(results.keys()), dtype=int)
    vals  = np.array([results[y] for y in years], dtype=float)

    return xr.DataArray(
        vals,
        coords={"year": years},
        dims=("year",),
        name="MA_min_O3"
    )

def get_window_mean_series(da, start_month, start_day, window_days):
    da = ensure_daily_if_needed(da)
    ts = da.to_series().sort_index()
    all_years = np.unique(ts.index.year)

    results = {}
    for yr in all_years:
        try:
            start_year = int(yr) - 1 if start_month == 12 else int(yr)
            start_key = f"{start_year:04d}-{start_month:02d}-{start_day:02d}"

            seg = ts[start_key:]
            if len(seg) < window_days:
                continue

            win = seg.iloc[:window_days].dropna()
            if len(win) < window_days:
                continue

            results[int(yr)] = win.mean()
        except Exception:
            continue

    if len(results) == 0:
        return xr.DataArray(
            np.array([], dtype=float),
            coords={"year": np.array([], dtype=int)},
            dims=("year",),
            name=f"window_{window_days}d_mean"
        )

    years = np.array(sorted(results.keys()), dtype=int)
    vals  = np.array([results[y] for y in years], dtype=float)

    return xr.DataArray(
        vals,
        coords={"year": years},
        dims=("year",),
        name=f"window_{window_days}d_mean"
    )

def compute_r_curve_merra_low25(window_days, lat_band=(40, 80), apply_o3_5d=True, min_n=5):
    if not os.path.exists(FILE_MERRA_EP):
        print(f"[{EXP_NAME}] Missing EP file.")
        return None, None

    if not os.path.exists(O3_FILE):
        print(f"[{EXP_NAME}] Missing O3 file: {O3_FILE}")
        return None, None

    if not os.path.exists(LOW25_FILE):
        print(f"[{EXP_NAME}] Missing low25 file: {LOW25_FILE}")
        return None, None

    ds_ep = xr.open_dataset(FILE_MERRA_EP)
    da_o3 = xr.open_dataarray(O3_FILE)

    low25_years = np.loadtxt(LOW25_FILE, dtype=int)
    low25_years = np.atleast_1d(low25_years)

    # ---------------- Fz ----------------
    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"
    fz_raw = ds_ep[var_fz]

    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    fz_raw = ensure_daily_if_needed(fz_raw)
    x_source = -1.0 * fz_raw

    # ---------------- O3 ----------------
    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    starts = build_window_starts()
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]
    r_list = []

    for start_month, start_day in starts:
        x_raw = get_window_mean_series(x_source, start_month, start_day, window_days)

        common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
        use_years = np.intersect1d(common_years, low25_years)

        if len(use_years) < min_n:
            r_list.append(np.nan)
            continue

        x_use = x_raw.sel(year=use_years)
        y_use = y_raw.sel(year=use_years)

        valid = np.isfinite(x_use.values) & np.isfinite(y_use.values)
        x_use = x_use.sel(year=x_use.year.values[valid])
        y_use = y_use.sel(year=y_use.year.values[valid])

        if len(x_use.year) < min_n:
            r_list.append(np.nan)
            continue

        try:
            r, _ = pearsonr(x_use.values, y_use.values)
        except Exception:
            r = np.nan

        r_list.append(r)

    ds_ep.close()
    return labels, r_list

def plot_single_curve(labels, r_list, window_days, suffix="LOW25"):
    out_dir = os.path.join(PLOT_BASE, EXP_NAME, "EP_O3_SlidingR")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5), dpi=140)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels))
    plt.plot(x, r_list, lw=2.2, marker="o", ms=3)

    tick_idx = np.arange(0, len(labels), 5)
    plt.xticks(tick_idx, [labels[i] for i in tick_idx], rotation=45)

    plt.ylim(-1, 1)
    plt.xlim(0, len(labels) - 1)
    plt.ylabel("Pearson r")
    plt.xlabel("Window start date (MM-DD)")
    plt.title(
        f"{EXP_NAME} | {window_days}-day sliding-window correlation with MA minimum O$_3$ ({suffix})",
        loc="left",
        fontweight="bold"
    )

    plt.tight_layout()

    save_path = os.path.join(out_dir, f"SlidingR_{window_days}d_vs_MAmin_{suffix}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)


# ---------------- Run ----------------
for wd in WINDOW_DAYS_LIST:
    labels, r_list = compute_r_curve_merra_low25(
        window_days=wd,
        lat_band=LAT_BAND,
        apply_o3_5d=APPLY_O3_5D,
        min_n=MIN_N
    )
    if labels is not None:
        plot_single_curve(labels, r_list, window_days=wd, suffix="LOW25")

In [ ]:
# ============================================================
# Block F (FAST DEBUG PARALLEL + BOOTSTRAP CI): Combined comparison plots with MERRA2
# 目标：
# 1) 画 4 张图：
#    - 90-day ALL
#    - 90-day LOW25
#    - 60-day ALL
#    - 60-day LOW25
# 2) 每张图 3 条线：
#    - B2000WCN coupled   (= BWCN + B2000WCN merged)
#    - B2000WCN NOCOUPL
#    - MERRA2
# 3) 与 Block G 保持一致：
#    - 时间轴：n-1年09-01 到 n年02-01
#    - bridge-year 删除规则：只要窗口起点在前一年月份(9/10/11/12)，
#      bridge case 就删除 target year = 104
#    - O3 目标变量：MA 的 5-day running mean minimum O3
# 4) 提速：
#    - 不再用 pandas/to_series/string slicing
#    - 预构建 date->index 查找表
#    - 预先缓存每个 source / window_days / start_date 的 x_raw(year)
#    - all / low25 共用同一套缓存
#    - 最多 12 核并行预计算
# 5) 独立 debug 输出目录，不与当前运行冲突
# 6) 带进度条
# 7) 新增 bootstrap:
#    - 对每个 start_date 的 year-wise paired samples 计算 Pearson r 的 95% CI
#    - 画 ribbon + 稀疏 error bar
# ============================================================

import os
from datetime import date, timedelta
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, total=None, desc=None):
        return iterable if iterable is not None else range(total)

# ---------------- 基础配置 ----------------
FILE_BWCN_EP     = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP    = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
FILE_NOCOUPL_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"
FILE_MERRA_EP    = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

LAT_BAND = (40, 80)
APPLY_O3_5D = True
MIN_N = 5

# 时间轴：n-1年09-01 到 n年02-01
START_MMDD = (9, 1)
END_MMDD   = (2, 1)
PREV_YEAR_MONTHS = {9, 10, 11, 12}

# 并行核数（最多 12）
MAX_WORKERS = 12

# 独立 debug tag：你可以手改，避免和别的 block 冲突
DEBUG_TAG = "DEBUG_FAST_v1"

# 是否输出每条曲线的 csv 调试文件
SAVE_DEBUG_CSV = True

# ---------------- Bootstrap / plotting 配置 ----------------
BOOTSTRAP_N = 2000
BOOTSTRAP_CI = 95.0
BOOTSTRAP_SEED = 42

SHOW_RIBBON = True
SHOW_ERRORBAR = True
ERRORBAR_EVERY = 8
ERRORBAR_CAPSIZE = 2.2
ERRORBAR_ELW = 0.9
RIBBON_ALPHA = 0.16           # 蓝红模式线默认 ribbon alpha
MERRA_RIBBON_ALPHA = 0.35     # 你要求的灰色阴影区 alpha=0.35


# ============================================================
# 通用工具
# ============================================================
def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})

def select_100hpa_level(da_plev):
    """
    自动兼容 plev 是 Pa 还是 hPa：
    - 若最大值 > 2000，视为 Pa，取 10000
    - 否则视为 hPa，取 100
    """
    pvals = np.asarray(da_plev["plev"].values, dtype=float)
    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0
        unit_guess = "Pa"
    else:
        target = 100.0
        unit_guess = "hPa"

    out = da_plev.sel(plev=target, method="nearest")
    print(f"  Selected 100 hPa level using target={target:g} ({unit_guess} interpretation).")
    return out

def assert_daily_unique(da, name=""):
    """
    cftime-safe daily 唯一性检查
    """
    da = da.sortby("time")

    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)

    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")

    return da

def uses_previous_year(start_month):
    return start_month in PREV_YEAR_MONTHS

def should_drop_bridge_year(is_bridge_case, start_month):
    """
    与 Block G 对齐：
    只要窗口起点在前一年月份(9/10/11/12)，bridge case 就删除 year=104
    """
    return is_bridge_case and uses_previous_year(start_month)

def build_window_starts(start_mmdd=(9, 1), end_mmdd=(2, 1)):
    """
    用普通日期只生成 month-day 序列
    """
    start = date(2000, start_mmdd[0], start_mmdd[1])
    end   = date(2001, end_mmdd[0], end_mmdd[1])

    out = []
    cur = start
    while cur <= end:
        out.append((cur.month, cur.day))
        cur += timedelta(days=1)
    return out


# ============================================================
# 预处理：source -> 轻量 numpy 结构
# ============================================================
def compute_ma_min_o3_series_fast(o3_da, apply_o3_5d=True):
    """
    返回：
    - y_years: 每年 year
    - y_vals : 每年 MA minimum O3
    """
    o3_da = assert_daily_unique(o3_da, name="O3").sortby("time")

    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    years  = o3_da.time.dt.year.values.astype(int)
    months = o3_da.time.dt.month.values.astype(int)
    vals   = o3_da.values.astype(np.float64)

    out_years = []
    out_vals  = []

    for yr in np.unique(years):
        mask = (years == yr) & np.isin(months, [3, 4])
        seg = vals[mask]

        n_valid = np.isfinite(seg).sum()
        if n_valid < 58:
            continue

        out_years.append(int(yr))
        out_vals.append(float(np.nanmin(seg)))

    return np.asarray(out_years, dtype=int), np.asarray(out_vals, dtype=float)

def load_one_source_light(ep_nc, o3_nc, low25_txt=None, lat_band=(40, 80), apply_o3_5d=True):
    """
    读取一个 source，并转成轻量、可并行传输的数据结构
    返回：
    {
      "fz_vals": 1D np.ndarray,
      "time_year": 1D int,
      "time_month": 1D int,
      "time_day": 1D int,
      "all_years": 1D int,
      "date_to_idx": dict[(year, month, day)] = idx,
      "y_years": 1D int,
      "y_vals": 1D float,
      "low25": 1D int or None
    }
    """
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        raise FileNotFoundError(f"Missing file(s): {ep_nc} or {o3_nc}")

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    # ---- Fz ----
    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"
    fz_raw = ds_ep[var_fz]

    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # 与 Block G 保持一致：使用 -Fz
    fz_raw = -1.0 * fz_raw
    fz_raw = assert_daily_unique(fz_raw, name=os.path.basename(ep_nc)).sortby("time").load()

    # ---- O3 ----
    da_o3 = assert_daily_unique(da_o3, name=os.path.basename(o3_nc)).sortby("time").load()

    # 时间轴分量
    time_year  = fz_raw.time.dt.year.values.astype(int)
    time_month = fz_raw.time.dt.month.values.astype(int)
    time_day   = fz_raw.time.dt.day.values.astype(int)

    # 轻量 time lookup
    date_to_idx = {
        (int(y), int(m), int(d)): int(i)
        for i, (y, m, d) in enumerate(zip(time_year, time_month, time_day))
    }

    # MA minimum O3
    y_years, y_vals = compute_ma_min_o3_series_fast(da_o3, apply_o3_5d=apply_o3_5d)

    # LOW25
    low25 = None
    if low25_txt is not None and os.path.exists(low25_txt):
        low25 = np.atleast_1d(np.loadtxt(low25_txt, dtype=int))

    out = {
        "fz_vals": fz_raw.values.astype(np.float64),
        "time_year": time_year,
        "time_month": time_month,
        "time_day": time_day,
        "all_years": np.unique(time_year).astype(int),
        "ntime": int(fz_raw.sizes["time"]),
        "date_to_idx": date_to_idx,
        "y_years": y_years,
        "y_vals": y_vals,
        "low25": low25,
    }

    ds_ep.close()
    return out


# ============================================================
# Bootstrap 子程序：针对每个 start_date 的 year-wise paired samples
# ============================================================
def bootstrap_pearson_r_ci(
    df_samples,
    n_boot=2000,
    ci=95.0,
    seed=42,
    exclude_bridge_year_from_bootstrap=True,
    bridge_year=104,
):
    """
    对当前 start_date 下的逐年配对样本 (x_raw, y_raw) 做 bootstrap，
    返回 Pearson r 的 percentile-based confidence interval。

    输入 df_samples 需要包含列：
      - part : 样本来自哪个部分（BWCN / B2000WCN / B2000WCN_NOCOUPL / MERRA2）
      - year : 样本年份
      - x_raw: 当前滑动窗口的 Fz mean
      - y_raw: 当前年份的 MA minimum O3

    统计逻辑：
      1) 每个 year 视为一个独立样本
      2) 抽样单位必须是配对样本 (x_i, y_i)，不能把 x / y 分开抽
      3) 每次带放回抽取与原样本数相同数量的样本
      4) 对每个 bootstrap sample 重新计算 Pearson r
      5) 取 2.5% / 97.5% 分位数作为 95% CI

    特别处理：
      - 若 exclude_bridge_year_from_bootstrap=True，则 bootstrap 抽样池中剔除：
          * B2000WCN 的 year=104
          * B2000WCN_NOCOUPL 的 year=104
        以避免 bridge year 在 bootstrap 中反复被抽到。
    """
    need_cols = {"part", "year", "x_raw", "y_raw"}
    if not need_cols.issubset(df_samples.columns):
        raise ValueError(f"df_samples must contain columns: {sorted(list(need_cols))}")

    df_clean = df_samples.loc[:, ["part", "year", "x_raw", "y_raw"]].copy()
    df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()

    n_original = len(df_clean)

    if n_original < 4:
        return {
            "r_obs": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_original": n_original,
            "n_boot_pool": 0,
            "n_boot_valid": 0,
        }

    x_obs = df_clean["x_raw"].to_numpy(dtype=float)
    y_obs = df_clean["y_raw"].to_numpy(dtype=float)

    if np.nanstd(x_obs) == 0 or np.nanstd(y_obs) == 0:
        return {
            "r_obs": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_original": n_original,
            "n_boot_pool": 0,
            "n_boot_valid": 0,
        }

    try:
        r_obs, _ = pearsonr(x_obs, y_obs)
    except Exception:
        r_obs = np.nan

    df_pool = df_clean.copy()

    if exclude_bridge_year_from_bootstrap:
        bad_mask = (
            df_pool["year"].eq(bridge_year) &
            df_pool["part"].isin(["B2000WCN", "B2000WCN_NOCOUPL"])
        )
        df_pool = df_pool.loc[~bad_mask].copy()

    n_boot_pool = len(df_pool)

    if n_boot_pool < 4:
        return {
            "r_obs": r_obs,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_original": n_original,
            "n_boot_pool": n_boot_pool,
            "n_boot_valid": 0,
        }

    x_pool = df_pool["x_raw"].to_numpy(dtype=float)
    y_pool = df_pool["y_raw"].to_numpy(dtype=float)

    if np.nanstd(x_pool) == 0 or np.nanstd(y_pool) == 0:
        return {
            "r_obs": r_obs,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_original": n_original,
            "n_boot_pool": n_boot_pool,
            "n_boot_valid": 0,
        }

    rng = np.random.default_rng(seed)
    draw_idx = rng.integers(0, n_boot_pool, size=(n_boot, n_original))

    xb = x_pool[draw_idx]
    yb = y_pool[draw_idx]

    xb_mean = xb.mean(axis=1, keepdims=True)
    yb_mean = yb.mean(axis=1, keepdims=True)

    xb_anom = xb - xb_mean
    yb_anom = yb - yb_mean

    num = np.sum(xb_anom * yb_anom, axis=1)
    den = np.sqrt(np.sum(xb_anom**2, axis=1) * np.sum(yb_anom**2, axis=1))

    r_boot = np.full(n_boot, np.nan, dtype=float)
    ok = den > 0
    r_boot[ok] = num[ok] / den[ok]

    r_boot_valid = r_boot[np.isfinite(r_boot)]
    n_boot_valid = len(r_boot_valid)

    if n_boot_valid < max(100, int(0.2 * n_boot)):
        return {
            "r_obs": r_obs,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "n_original": n_original,
            "n_boot_pool": n_boot_pool,
            "n_boot_valid": n_boot_valid,
        }

    alpha = 100.0 - ci
    lo_q = alpha / 2.0
    hi_q = 100.0 - alpha / 2.0

    ci_low = float(np.percentile(r_boot_valid, lo_q))
    ci_high = float(np.percentile(r_boot_valid, hi_q))

    return {
        "r_obs": float(r_obs),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "n_original": n_original,
        "n_boot_pool": n_boot_pool,
        "n_boot_valid": n_boot_valid,
    }


# ============================================================
# 并行缓存：每个 part / window_days 预计算全部起点
# ============================================================
def precompute_source_window_cache_worker(source_light, window_days, starts):
    """
    对单个 source + 单个 window_days，预计算所有起点的 x_raw(year)
    返回：
      {
        (m, d): {
           "years": np.ndarray[int],
           "vals" : np.ndarray[float]
        },
        ...
      }
    """
    fz_vals     = source_light["fz_vals"]
    all_years   = source_light["all_years"]
    ntime       = source_light["ntime"]
    date_to_idx = source_light["date_to_idx"]

    cache = {}

    for start_month, start_day in starts:
        out_years = []
        out_vals  = []

        for yr in all_years:
            start_year = int(yr) - 1 if uses_previous_year(start_month) else int(yr)
            start_idx = date_to_idx.get((start_year, start_month, start_day), None)
            if start_idx is None:
                continue

            end_idx = start_idx + window_days
            if end_idx > ntime:
                continue

            win = fz_vals[start_idx:end_idx]
            if np.isfinite(win).sum() < window_days:
                continue

            out_years.append(int(yr))
            out_vals.append(float(np.nanmean(win)))

        cache[(start_month, start_day)] = {
            "years": np.asarray(out_years, dtype=int),
            "vals":  np.asarray(out_vals,  dtype=float),
        }

    return cache


# ============================================================
# source 组织
# ============================================================
def build_sources_light():
    sources = {
        "coupled": {
            "label": "B2000WCN",
            "parts": [
                {
                    "name": "BWCN",
                    "is_bridge_case": False,
                    "data": load_one_source_light(
                        FILE_BWCN_EP,
                        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "BWCN", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D
                    )
                },
                {
                    "name": "B2000WCN",
                    "is_bridge_case": True,
                    "data": load_one_source_light(
                        FILE_B2000_EP,
                        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "B2000WCN", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D
                    )
                },
            ]
        },
        "nocoupl": {
            "label": "B2000WCN NOCOUPL",
            "parts": [
                {
                    "name": "B2000WCN_NOCOUPL",
                    "is_bridge_case": True,
                    "data": load_one_source_light(
                        FILE_NOCOUPL_EP,
                        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D
                    )
                }
            ]
        },
        "merra2": {
            "label": "MERRA2",
            "parts": [
                {
                    "name": "MERRA2",
                    "is_bridge_case": False,
                    "data": load_one_source_light(
                        FILE_MERRA_EP,
                        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "MERRA2", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D
                    )
                }
            ]
        }
    }
    return sources

def attach_parallel_xcache(sources, starts, window_days_list, max_workers=12):
    """
    给每个 part["data"] 挂上：
      part["data"]["x_cache"][window_days][(m,d)] = {"years":..., "vals":...}
    """
    tasks = []
    for group in sources.values():
        for part in group["parts"]:
            part["data"]["x_cache"] = {}
            for wd in window_days_list:
                tasks.append((part, wd))

    n_workers = min(max_workers, len(tasks))
    print(f"\n[Parallel cache] launching {len(tasks)} tasks with up to {n_workers} workers ...")

    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        fut_to_meta = {}
        for part, wd in tasks:
            fut = ex.submit(
                precompute_source_window_cache_worker,
                part["data"],
                wd,
                starts
            )
            fut_to_meta[fut] = (part, wd)

        for fut in tqdm(as_completed(fut_to_meta), total=len(fut_to_meta), desc="Precomputing x_cache"):
            part, wd = fut_to_meta[fut]
            part["data"]["x_cache"][wd] = fut.result()


# ============================================================
# 相关计算
# ============================================================
def make_df_for_window_from_cache(source_dict, part_name, start_month, start_day, window_days,
                                  subset="all", is_bridge_case=False):
    """
    使用已缓存的 x_cache 构造样本表
    """
    x_pack = source_dict["x_cache"][window_days][(start_month, start_day)]
    x_years = x_pack["years"]
    x_vals  = x_pack["vals"]

    y_years = source_dict["y_years"]
    y_vals  = source_dict["y_vals"]

    # bridge case：与 Block G 一致
    if should_drop_bridge_year(is_bridge_case, start_month):
        keep = x_years != BRIDGE_YEAR
        x_years = x_years[keep]
        x_vals  = x_vals[keep]

    if x_years.size == 0 or y_years.size == 0:
        return pd.DataFrame(columns=["part", "year", "x_raw", "y_raw"])

    # year 对齐
    y_map = {int(yr): float(val) for yr, val in zip(y_years, y_vals)}

    rows_year = []
    rows_x = []
    rows_y = []

    if subset.lower() == "low25":
        low25 = source_dict["low25"]
        if low25 is None:
            return pd.DataFrame(columns=["part", "year", "x_raw", "y_raw"])
        low25_set = set(np.atleast_1d(low25).astype(int).tolist())
    else:
        low25_set = None

    for yr, xv in zip(x_years, x_vals):
        yr = int(yr)

        if low25_set is not None and yr not in low25_set:
            continue
        if yr not in y_map:
            continue
        yv = y_map[yr]

        if not (np.isfinite(xv) and np.isfinite(yv)):
            continue

        rows_year.append(yr)
        rows_x.append(float(xv))
        rows_y.append(float(yv))

    if len(rows_year) == 0:
        return pd.DataFrame(columns=["part", "year", "x_raw", "y_raw"])

    return pd.DataFrame({
        "part": [part_name] * len(rows_year),
        "year": rows_year,
        "x_raw": rows_x,
        "y_raw": rows_y,
    })

def compute_curve_for_group(group_info, starts, window_days, subset="all", min_n=5,
                            n_boot=2000, ci=95.0, base_seed=42, seed_offset=0):
    """
    对一条线（可能由多个部分组成，比如 BWCN+B2000WCN）计算滑动相关曲线，
    并为每个 start_date 计算 bootstrap CI。

    返回：
      labels, out_dict

    out_dict 包含：
      - r
      - ci_low
      - ci_high
      - n
      - n_boot_pool
      - n_boot_valid
    """
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]

    r_list = []
    lo_list = []
    hi_list = []
    n_list = []
    n_boot_pool_list = []
    n_boot_valid_list = []

    subset_flag = 0 if subset.lower() == "all" else 1

    for i_start, (start_month, start_day) in enumerate(starts):
        df_list = []

        for part in group_info["parts"]:
            df_part = make_df_for_window_from_cache(
                source_dict=part["data"],
                part_name=part["name"],
                start_month=start_month,
                start_day=start_day,
                window_days=window_days,
                subset=subset,
                is_bridge_case=part["is_bridge_case"]
            )
            df_list.append(df_part)

        df_all = pd.concat(df_list, ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()

        n_now = len(df_all)
        n_list.append(n_now)

        if n_now < min_n:
            r_list.append(np.nan)
            lo_list.append(np.nan)
            hi_list.append(np.nan)
            n_boot_pool_list.append(0)
            n_boot_valid_list.append(0)
            continue

        x = df_all["x_raw"].to_numpy(dtype=float)
        y = df_all["y_raw"].to_numpy(dtype=float)

        if np.nanstd(x) == 0 or np.nanstd(y) == 0:
            r_list.append(np.nan)
            lo_list.append(np.nan)
            hi_list.append(np.nan)
            n_boot_pool_list.append(0)
            n_boot_valid_list.append(0)
            continue

        # 每个 start_date 给一个确定性 seed，保证可复现
        this_seed = int(base_seed + seed_offset * 100000 + window_days * 1000 + i_start * 10 + subset_flag)

        boot = bootstrap_pearson_r_ci(
            df_all,
            n_boot=n_boot,
            ci=ci,
            seed=this_seed,
            exclude_bridge_year_from_bootstrap=True,
            bridge_year=BRIDGE_YEAR,
        )

        r_list.append(boot["r_obs"])
        lo_list.append(boot["ci_low"])
        hi_list.append(boot["ci_high"])
        n_boot_pool_list.append(boot["n_boot_pool"])
        n_boot_valid_list.append(boot["n_boot_valid"])

    out = {
        "r": r_list,
        "ci_low": lo_list,
        "ci_high": hi_list,
        "n": n_list,
        "n_boot_pool": n_boot_pool_list,
        "n_boot_valid": n_boot_valid_list,
    }

    return labels, out


# ============================================================
# 输出
# ============================================================
OUT_DIR = os.path.join(PLOT_BASE, "COMBINED_COMPARE_WITH_MERRA2", "EP_O3_SlidingR")
os.makedirs(OUT_DIR, exist_ok=True)

def save_debug_curve_csv(labels, curves_dict, counts_dict, window_days, subset):
    if not SAVE_DEBUG_CSV:
        return

    subset_upper = subset.upper()
    df = pd.DataFrame({
        "label": labels,

        "coupled_r": curves_dict["coupled"]["r"],
        "coupled_ci_low": curves_dict["coupled"]["ci_low"],
        "coupled_ci_high": curves_dict["coupled"]["ci_high"],
        "coupled_n": curves_dict["coupled"]["n"],
        "coupled_n_boot_pool": curves_dict["coupled"]["n_boot_pool"],
        "coupled_n_boot_valid": curves_dict["coupled"]["n_boot_valid"],

        "nocoupl_r": curves_dict["nocoupl"]["r"],
        "nocoupl_ci_low": curves_dict["nocoupl"]["ci_low"],
        "nocoupl_ci_high": curves_dict["nocoupl"]["ci_high"],
        "nocoupl_n": curves_dict["nocoupl"]["n"],
        "nocoupl_n_boot_pool": curves_dict["nocoupl"]["n_boot_pool"],
        "nocoupl_n_boot_valid": curves_dict["nocoupl"]["n_boot_valid"],

        "merra2_r": curves_dict["merra2"]["r"],
        "merra2_ci_low": curves_dict["merra2"]["ci_low"],
        "merra2_ci_high": curves_dict["merra2"]["ci_high"],
        "merra2_n": curves_dict["merra2"]["n"],
        "merra2_n_boot_pool": curves_dict["merra2"]["n_boot_pool"],
        "merra2_n_boot_valid": curves_dict["merra2"]["n_boot_valid"],
    })

    save_csv = os.path.join(
        OUT_DIR,
        f"SlidingR_COMPARE_{window_days}d_{subset_upper}_B2000_NOCOUPL_MERRA2_DEBUG.csv"
    )
    df.to_csv(save_csv, index=False)
    print("Saved debug csv:", save_csv)

def plot_three_line_comparison(labels, curves_dict, counts_dict, window_days, subset="all"):
    subset_upper = subset.upper()

    plt.figure(figsize=(12.8, 6.0), dpi=150)
    ax = plt.gca()

    ax.axhline(0, color="k", lw=0.8, alpha=0.5)
    ax.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels), dtype=float)

    color_map = {
        "coupled": "#1f77b4",   # blue
        "nocoupl": "#d62728",   # red
        "merra2":  "black",     # black main line
    }

    order = [
        ("coupled", "B2000WCN"),
        ("nocoupl", "B2000WCN NOCOUPL"),
        ("merra2",  "MERRA2"),
    ]

    for key, label in order:
        y  = np.asarray(curves_dict[key]["r"], dtype=float)
        lo = np.asarray(curves_dict[key]["ci_low"], dtype=float)
        hi = np.asarray(curves_dict[key]["ci_high"], dtype=float)

        line_color = color_map[key]

        # ---------------- 主线 ----------------
        ax.plot(
            x, y,
            lw=2.4,
            linestyle="-",
            color=line_color,
            label=label,
            zorder=4
        )

        # ---------------- 包络区 ----------------
        if SHOW_RIBBON:
            mask_fill = np.isfinite(y) & np.isfinite(lo) & np.isfinite(hi)
            if np.any(mask_fill):
                if key == "merra2":
                    poly = ax.fill_between(
                        x[mask_fill],
                        lo[mask_fill],
                        hi[mask_fill],
                        facecolor="lightgray",
                        edgecolor="gray",
                        alpha=MERRA_RIBBON_ALPHA,   # 你要求的 alpha=0.35
                        linewidth=0.6,
                        zorder=1
                    )
                    poly.set_hatch("///")
                else:
                    ax.fill_between(
                        x[mask_fill],
                        lo[mask_fill],
                        hi[mask_fill],
                        color=line_color,
                        alpha=RIBBON_ALPHA,
                        linewidth=0,
                        zorder=1
                    )

        # ---------------- 稀疏 error bars ----------------
        if SHOW_ERRORBAR:
            idx_all = np.arange(len(x))
            mask_err = (
                np.isfinite(y) & np.isfinite(lo) & np.isfinite(hi) &
                ((idx_all % ERRORBAR_EVERY) == 0)
            )
            if np.any(mask_err):
                yerr = np.vstack([
                    y[mask_err] - lo[mask_err],
                    hi[mask_err] - y[mask_err]
                ])

                ecolor = "black" if key == "merra2" else line_color

                ax.errorbar(
                    x[mask_err],
                    y[mask_err],
                    yerr=yerr,
                    fmt="none",
                    ecolor=ecolor,
                    elinewidth=ERRORBAR_ELW,
                    capsize=ERRORBAR_CAPSIZE,
                    alpha=0.9,
                    zorder=5
                )

    tick_idx = [i for i, lab in enumerate(labels) if lab.endswith("-01")]
    ax.set_xticks(tick_idx)
    ax.set_xticklabels([labels[i] for i in tick_idx], rotation=45)

    ax.set_ylim(-1, 1)
    ax.set_xlim(0, len(labels) - 1)
    ax.set_ylabel("Pearson r")
    ax.set_xlabel("Window start date relative to target year n (n-1 Sep -> n Feb)")
    ax.set_title(
        f"{window_days}-day sliding-window correlation with MA minimum O$_3$ ({subset_upper})",
        loc="left",
        fontweight="bold"
    )

    txt = (
        f"Bootstrap CI = {BOOTSTRAP_CI:.0f}%\n"
        f"Nboot = {BOOTSTRAP_N}"
    )
    ax.text(
        0.985, 0.97, txt,
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor="0.7")
    )

    ax.legend(frameon=False, ncol=3)
    plt.tight_layout()

    save_path = os.path.join(
        OUT_DIR,
        f"SlidingR_COMPARE_{window_days}d_{subset_upper}_B2000_NOCOUPL_MERRA2_BOOTCI.png"
    )
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

    save_debug_curve_csv(labels, curves_dict, counts_dict, window_days, subset)


# ============================================================
# Run
# ============================================================
print("=" * 80)
print("Building lightweight sources ...")
sources = build_sources_light()

starts = build_window_starts(START_MMDD, END_MMDD)
print(f"Total start dates = {len(starts)}")
print(f"Output dir = {OUT_DIR}")

# 并行预缓存
attach_parallel_xcache(
    sources=sources,
    starts=starts,
    window_days_list=WINDOW_DAYS_LIST,
    max_workers=MAX_WORKERS
)

# 曲线计算 + 出图（这一步已经很快了）
jobs = [(wd, subset) for wd in WINDOW_DAYS_LIST for subset in ["all", "low25"]]

for wd, subset in tqdm(jobs, total=len(jobs), desc="Computing curves / plotting"):
    labels, out_coupled = compute_curve_for_group(
        sources["coupled"],
        starts=starts,
        window_days=wd,
        subset=subset,
        min_n=MIN_N,
        n_boot=BOOTSTRAP_N,
        ci=BOOTSTRAP_CI,
        base_seed=BOOTSTRAP_SEED,
        seed_offset=1
    )
    _, out_nocoupl = compute_curve_for_group(
        sources["nocoupl"],
        starts=starts,
        window_days=wd,
        subset=subset,
        min_n=MIN_N,
        n_boot=BOOTSTRAP_N,
        ci=BOOTSTRAP_CI,
        base_seed=BOOTSTRAP_SEED,
        seed_offset=2
    )
    _, out_merra2 = compute_curve_for_group(
        sources["merra2"],
        starts=starts,
        window_days=wd,
        subset=subset,
        min_n=MIN_N,
        n_boot=BOOTSTRAP_N,
        ci=BOOTSTRAP_CI,
        base_seed=BOOTSTRAP_SEED,
        seed_offset=3
    )

    curves = {
        "coupled": out_coupled,
        "nocoupl": out_nocoupl,
        "merra2": out_merra2,
    }

    counts = {
        "coupled": out_coupled["n"],
        "nocoupl": out_nocoupl["n"],
        "merra2": out_merra2["n"],
    }

    plot_three_line_comparison(
        labels=labels,
        curves_dict=curves,
        counts_dict=counts,
        window_days=wd,
        subset=subset
    )

print("\nAll done.")
print("Files written to:", OUT_DIR)

##这里的bootstrap是有放回的全部抽样。

In [ ]:
# ============================================================
# Block G: Animated scatter GIFs + MP4 (MERRA2 / Coupled / NOCOUPL)
# ============================================================

import os
import glob
from datetime import timedelta, date

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import imageio.v2 as imageio


# ============================================================
# 基础配置
# ============================================================
FILE_BWCN_EP     = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP    = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
FILE_NOCOUPL_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"
FILE_MERRA_EP    = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

LAT_BAND = (40, 80)
APPLY_O3_5D = True

# 时间轴：n-1年09-01 到 n年02-01
START_MMDD = (9, 1)
END_MMDD   = (2, 1)

# 前一年起点月份
PREV_YEAR_MONTHS = {9, 10, 11, 12}

# GIF / MP4 设置
FRAME_DURATION = 1.0
MP4_FPS = 10

# 是否固定坐标轴范围（避免动画抖动）
USE_FIXED_LIM = True
FIXED_LIM = (-4.5, 4.5)

# 背景点 / low25 点样式
BG_COLOR = "0.35"
LOW25_COLOR = "red"
BG_ALPHA = 0.75
LOW25_ALPHA = 0.95
BG_SIZE = 34
LOW25_SIZE = 42

# 极端年标记配置
MARK_EXTREMES = True
EXTREME_TOPN = 5
EXTREME_BY = "y_raw"          # 以原始 O3 最低年定义极端
EXTREME_ASCENDING = True      # True = 取最小值 = 最低 O3
EXTREME_SIZE = 120
EXTREME_LINEWIDTH = 1.6
EXTREME_COLORS = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']



# ============================================================
# Helpers
# ============================================================
def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})


def select_100hpa_level(da_plev):
    """
    自动兼容 plev 是 Pa 还是 hPa：
    - 若最大值 > 2000，视为 Pa，取 10000
    - 否则视为 hPa，取 100
    """
    pvals = np.asarray(da_plev["plev"].values, dtype=float)
    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0
        unit_guess = "Pa"
    else:
        target = 100.0
        unit_guess = "hPa"

    out = da_plev.sel(plev=target, method="nearest")
    print(f"  Selected 100 hPa level using target={target:g} ({unit_guess} interpretation).")
    return out


def assert_daily_unique(da, name=""):
    """
    兼容 cftime / datetime64 的 daily 唯一性检查。
    不做 pandas 转换，不刷屏。
    """
    da = da.sortby("time")

    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)

    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")

    return da


def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)


def uses_previous_year(start_month):
    return start_month in PREV_YEAR_MONTHS


def should_drop_bridge_year(is_bridge_case, start_month):
    """
    对 bridge case：
    只要窗口起点在前一年（9/10/11/12月），
    就删除 target year = 104。
    """
    return is_bridge_case and uses_previous_year(start_month)


def build_window_starts(start_mmdd=(9, 1), end_mmdd=(2, 1)):
    """
    用普通日期只生成 month-day 序列。
    """
    start = date(2000, start_mmdd[0], start_mmdd[1])
    end   = date(2001, end_mmdd[0], end_mmdd[1])

    out = []
    cur = start
    while cur <= end:
        out.append((cur.month, cur.day))
        cur += timedelta(days=1)
    return out


def add_days_to_time(t, ndays):
    """
    同时兼容 cftime / python datetime / numpy datetime64
    """
    try:
        return t + timedelta(days=ndays)
    except Exception:
        return t + np.timedelta64(ndays, "D")


def get_exact_time_value(da, year, month, day):
    mask = (
        (da.time.dt.year == year) &
        (da.time.dt.month == month) &
        (da.time.dt.day == day)
    )
    tt = da.time.where(mask, drop=True)
    if tt.size == 0:
        return None
    return tt.values[0]


def standardize_array(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if (not np.isfinite(sd)) or (sd == 0):
        return np.full_like(x, np.nan, dtype=float)
    return (x - mu) / sd


def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    """
    目标变量：MA 的 5-day running mean minimum O3
    完全避免 pandas / to_series / 字符串切片
    """
    o3_da = assert_daily_unique(o3_da, name="O3").sortby("time")

    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(o3_da.time.dt.year.values.astype(int))

    out_years = []
    out_vals = []

    for yr in years:
        mask = (
            (o3_da.time.dt.year == yr) &
            (o3_da.time.dt.month.isin([3, 4]))
        )
        seg = o3_da.where(mask, drop=True)

        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid < 58:
            continue

        out_years.append(int(yr))
        out_vals.append(float(seg.min().values))

    return xr.DataArray(
        np.array(out_vals, dtype=float),
        coords={"year": np.array(out_years, dtype=int)},
        dims=("year",),
        name="MA_min_O3"
    )


def get_window_mean_series(da, start_month, start_day, window_days):
    """
    对每个 target year，取固定长度滑动窗口均值
    完全避免 pandas string slicing
    """
    da = assert_daily_unique(da, name="Fz").sortby("time")
    years = np.unique(da.time.dt.year.values.astype(int))

    out_years = []
    out_vals = []

    for yr in years:
        start_year = int(yr) - 1 if uses_previous_year(start_month) else int(yr)

        start_time = get_exact_time_value(da, start_year, start_month, start_day)
        if start_time is None:
            continue

        end_time = add_days_to_time(start_time, window_days - 1)

        win = da.sel(time=slice(start_time, end_time))
        n_valid = int(win.count().values) if win.size > 0 else 0

        if n_valid < window_days:
            continue

        out_years.append(int(yr))
        out_vals.append(float(win.mean().values))

    return xr.DataArray(
        np.array(out_vals, dtype=float),
        coords={"year": np.array(out_years, dtype=int)},
        dims=("year",),
        name=f"window_{window_days}d_mean"
    )


def load_one_source(ep_nc, o3_nc, low25_txt=None, lat_band=(40, 80),
                    apply_o3_5d=True, drop_leap_day=False):
    """
    加载一个实验源，返回：
    {
      "ds": ds_ep,
      "fz_raw": 100 hPa, lat-mean, sign corrected (-Fz),
      "y_raw": MA minimum O3,
      "low25": low25 years or None
    }
    """
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        raise FileNotFoundError(f"Missing file(s): {ep_nc} or {o3_nc}")

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    # ---- Fz ----
    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"
    fz_raw = ds_ep[var_fz]

    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # 与前面保持一致：使用 -Fz
    fz_raw = -1.0 * fz_raw

    if drop_leap_day:
        fz_raw = drop_feb29(fz_raw)
        da_o3  = drop_feb29(da_o3)

    fz_raw = assert_daily_unique(fz_raw, name=os.path.basename(ep_nc))
    da_o3  = assert_daily_unique(da_o3,  name=os.path.basename(o3_nc))

    # ---- O3 ----
    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    # ---- LOW25 ----
    low25 = None
    if low25_txt is not None and os.path.exists(low25_txt):
        low25 = np.atleast_1d(np.loadtxt(low25_txt, dtype=int))

    return {
        "ds": ds_ep,
        "fz_raw": fz_raw,
        "y_raw": y_raw,
        "low25": low25,
    }


def make_df_for_window(source_dict, part_name, start_month, start_day, window_days,
                       is_bridge_case=False):
    """
    给定某个 source，生成该窗口下的散点样本表
    columns = [part, year, x_raw, y_raw, is_low25]
    """
    x_raw = get_window_mean_series(source_dict["fz_raw"], start_month, start_day, window_days)
    y_raw = source_dict["y_raw"]

    if should_drop_bridge_year(is_bridge_case, start_month):
        if BRIDGE_YEAR in x_raw.year.values:
            x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

    common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
    if len(common_years) == 0:
        return None

    x_use = x_raw.sel(year=common_years)
    y_use = y_raw.sel(year=common_years)

    low25 = source_dict["low25"]
    if low25 is None:
        is_low25 = np.zeros(len(common_years), dtype=bool)
    else:
        is_low25 = np.isin(common_years, low25)

    valid = np.isfinite(x_use.values) & np.isfinite(y_use.values)
    if not np.any(valid):
        return None

    df = {
        "part": np.array([part_name] * valid.sum()),
        "year": x_use.year.values[valid].astype(int),
        "x_raw": x_use.values[valid].astype(float),
        "y_raw": y_use.values[valid].astype(float),
        "is_low25": is_low25[valid],
    }
    return df


def concat_dict_rows(dict_list):
    """
    把多个字典表拼起来
    """
    dict_list = [d for d in dict_list if d is not None]
    if len(dict_list) == 0:
        return None

    out = {}
    keys = dict_list[0].keys()
    for k in keys:
        out[k] = np.concatenate([d[k] for d in dict_list])
    return out


def standardize_group_df(df):
    if df is None or len(df["year"]) == 0:
        return None

    xs = standardize_array(df["x_raw"])
    ys = standardize_array(df["y_raw"])

    valid = np.isfinite(xs) & np.isfinite(ys)
    if not np.any(valid):
        return None

    return {
        "part": df["part"][valid],
        "year": df["year"][valid],
        "x": xs[valid],
        "y": ys[valid],
        "is_low25": df["is_low25"][valid],
        "x_raw": df["x_raw"][valid],
        "y_raw": df["y_raw"][valid],
    }


def compute_axis_limits(dfs, fixed=True, fixed_lim=(-4.5, 4.5), pad=0.4):
    if fixed:
        return fixed_lim, fixed_lim

    vals = []
    for df in dfs:
        if df is not None:
            vals.extend(df["x"])
            vals.extend(df["y"])

    vals = np.asarray(vals, dtype=float)
    if vals.size == 0:
        return (-3, 3), (-3, 3)

    vmax = np.nanmax(np.abs(vals))
    vmax = max(vmax + pad, 2.5)
    return (-vmax, vmax), (-vmax, vmax)


def select_panel_extremes(df, topn=5, extreme_by="y_raw", ascending=True):
    """
    从某个 panel 当前帧的数据中，挑选极端年
    返回索引数组（相对于 df 内部）
    """
    if df is None or len(df["year"]) == 0:
        return np.array([], dtype=int)

    vals = np.asarray(df[extreme_by], dtype=float)
    valid_idx = np.where(np.isfinite(vals))[0]
    if valid_idx.size == 0:
        return np.array([], dtype=int)

    order = np.argsort(vals[valid_idx])
    if not ascending:
        order = order[::-1]

    keep = valid_idx[order[:min(topn, len(order))]]
    return keep


def overlay_panel_extremes(ax, df, idx_extreme):
    """
    只画极端点（不再标注文字）
    """
    if df is None or len(idx_extreme) == 0:
        return []

    legend_entries = []

    for i, idx in enumerate(idx_extreme):
        c = EXTREME_COLORS[i % len(EXTREME_COLORS)]

        x = float(df["x"][idx])
        y = float(df["y"][idx])
        part = str(df["part"][idx])
        year = int(df["year"][idx])

        # --- 标签规则 ---
        if part == "BWCN":
            label = f"BWCN-{year:04d}"
        else:
            label = f"{year:04d}"

        legend_entries.append((i, label, c, part))

        # --- 点绘制 ---
        if part == "BWCN":
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                facecolors="none",
                edgecolors=c,
                linewidths=EXTREME_LINEWIDTH,
                zorder=10
            )
        else:
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                color=c,
                edgecolors="k",
                linewidths=0.8,
                zorder=10
            )

    return legend_entries


def build_sources():
    """
    三条线：
    1) coupled  = BWCN + B2000WCN
    2) nocoupl  = B2000WCN_NOCOUPL
    3) merra2   = MERRA2
    """
    sources = {
        "coupled": {
            "label": "B2000WCN",
            "parts": [
                {
                    "name": "BWCN",
                    "is_bridge_case": False,
                    "data": load_one_source(
                        FILE_BWCN_EP,
                        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "BWCN", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
                {
                    "name": "B2000WCN",
                    "is_bridge_case": True,
                    "data": load_one_source(
                        FILE_B2000_EP,
                        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "B2000WCN", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
            ]
        },
        "nocoupl": {
            "label": "B2000WCN NOCOUPL",
            "parts": [
                {
                    "name": "B2000WCN_NOCOUPL",
                    "is_bridge_case": True,
                    "data": load_one_source(
                        FILE_NOCOUPL_EP,
                        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        },
        "merra2": {
            "label": "MERRA2",
            "parts": [
                {
                    "name": "MERRA2",
                    "is_bridge_case": False,
                    "data": load_one_source(
                        FILE_MERRA_EP,
                        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "MERRA2", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        }
    }
    return sources


def close_sources(sources):
    for group in sources.values():
        for part in group["parts"]:
            ds = part["data"].get("ds", None)
            if ds is not None:
                ds.close()


def build_group_scatter_df(group_info, start_month, start_day, window_days):
    """
    为某一组（coupled / nocoupl / merra2）在某一帧构造散点数据
    """
    rows = []
    for part in group_info["parts"]:
        df_part = make_df_for_window(
            source_dict=part["data"],
            part_name=part["name"],
            start_month=start_month,
            start_day=start_day,
            window_days=window_days,
            is_bridge_case=part["is_bridge_case"]
        )
        rows.append(df_part)

    df_all = concat_dict_rows(rows)
    df_all = standardize_group_df(df_all)
    return df_all


def draw_one_panel(ax, df, panel_title, xlim, ylim):
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.axhline(0, color="k", lw=0.8, alpha=0.45)
    ax.axvline(0, color="k", lw=0.8, alpha=0.45)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    if df is None or len(df["year"]) == 0:
        ax.set_title(panel_title, fontweight="bold")
        return

    low_mask = df["is_low25"]
    bg_mask = ~low_mask

    # 背景点
    if np.any(bg_mask):
        ax.scatter(
            df["x"][bg_mask], df["y"][bg_mask],
            s=BG_SIZE, color=BG_COLOR, alpha=BG_ALPHA,
            edgecolors="none", zorder=2
        )

    # low25
    if np.any(low_mask):
        ax.scatter(
            df["x"][low_mask], df["y"][low_mask],
            s=LOW25_SIZE, color=LOW25_COLOR, alpha=LOW25_ALPHA,
            edgecolors="k", linewidths=0.4, zorder=5
        )

    # =============================
    # 极端年（收集 legend）
    # =============================
    legend_entries = []
    if MARK_EXTREMES:
        idx_ext = select_panel_extremes(
            df,
            topn=EXTREME_TOPN,
            extreme_by=EXTREME_BY,
            ascending=EXTREME_ASCENDING
        )
        legend_entries = overlay_panel_extremes(ax, df, idx_ext)

    # =============================
    # legend（右下角，小字体）
    # =============================
    if legend_entries:
        from matplotlib.lines import Line2D

        handles = []
        labels = []

        for i, label, c, part in legend_entries:
            if part == "BWCN":
                h = Line2D([0], [0],
                           marker='o',
                           color='none',
                           markeredgecolor=c,
                           markerfacecolor='none',
                           markersize=6,
                           linewidth=0)
            else:
                h = Line2D([0], [0],
                           marker='o',
                           color='none',
                           markeredgecolor='k',
                           markerfacecolor=c,
                           markersize=6,
                           linewidth=0)

            handles.append(h)
            labels.append(f"Top{i+1}: {label}")

        ax.legend(
            handles, labels,
            loc="lower right",
            fontsize=6,     # ✅ 字体减半
            frameon=False
        )

    # =============================
    # 统计信息（保持）
    # =============================
    n_all = len(df["x"])
    if n_all >= 3:
        try:
            r_all, p_all = pearsonr(df["x"], df["y"])
            txt_all = f"ALL\nr = {r_all:.3f}\np = {p_all:.2e}\nN = {n_all}"
        except Exception:
            txt_all = f"ALL\nN = {n_all}"
    else:
        txt_all = f"ALL\nN = {n_all}"

    ax.text(
        0.04, 0.96, txt_all,
        transform=ax.transAxes, va="top", ha="left",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35)
    )

    ax.set_title(panel_title, fontweight="bold")
    ax.set_xlabel(r"Window mean $-F_z$ ($\sigma$)")
    ax.set_ylabel(r"MA min O$_3$ ($\sigma$)")


def plot_one_frame(sources, start_month, start_day, window_days, save_path):
    """
    画单帧三实验组图
    布局：
      第一行中间：MERRA2
      第二行左：Coupled
      第二行右：NOCOUPL
    """
    df_merra   = build_group_scatter_df(sources["merra2"],  start_month, start_day, window_days)
    df_coupled = build_group_scatter_df(sources["coupled"], start_month, start_day, window_days)
    df_noc     = build_group_scatter_df(sources["nocoupl"], start_month, start_day, window_days)

    xlim, ylim = compute_axis_limits(
        [df_merra, df_coupled, df_noc],
        fixed=USE_FIXED_LIM,
        fixed_lim=FIXED_LIM
    )

    fig = plt.figure(figsize=(12.6, 9.0), dpi=160)
    gs = fig.add_gridspec(
        2, 3,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.0, 1.0, 1.0],
        hspace=0.38, wspace=0.32
    )

    # top center
    ax_top = fig.add_subplot(gs[0, 1])

    # bottom left / right
    ax_bl = fig.add_subplot(gs[1, 0])
    ax_br = fig.add_subplot(gs[1, 2])

    draw_one_panel(ax_top, df_merra,   "MERRA2",           xlim, ylim)
    draw_one_panel(ax_bl,  df_coupled, "B2000WCN",         xlim, ylim)
    draw_one_panel(ax_br,  df_noc,     "B2000WCN NOCOUPL", xlim, ylim)

    fig.suptitle(
        f"{window_days}-day window  |  Start date = {start_month:02d}-{start_day:02d}",
        fontsize=16, fontweight="bold", y=0.97
    )
    fig.text(
        0.5, 0.935,
        f"All years shown; low25 highlighted in red; Top {EXTREME_TOPN} lowest O3 years labeled in each panel",
        ha="center", va="center", fontsize=11
    )

    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def make_gif_from_frames(frame_paths, gif_path, duration=1.0):
    images = [imageio.imread(fp) for fp in frame_paths]
    imageio.mimsave(gif_path, images, duration=duration, loop=0)
    print("Saved GIF:", gif_path)


def make_mp4_from_frames(frame_paths, mp4_path, fps=10):
    """
    解决尺寸不匹配导致的卡顿和拉伸问题
    """
    print(f"Compiling MP4 at {fps} FPS -> {mp4_path}")

    with imageio.get_writer(
        mp4_path,
        fps=fps,
        codec="libx264",
        quality=9,
        macro_block_size=None
    ) as writer:
        for fp in frame_paths:
            img = imageio.imread(fp)

            # 强制裁剪为偶数长宽，避免 ffmpeg 抖动
            h, w = img.shape[:2]
            new_h = h if h % 2 == 0 else h - 1
            new_w = w if w % 2 == 0 else w - 1
            img = img[:new_h, :new_w, ...]

            writer.append_data(img)

    print("Saved MP4:", mp4_path)


def make_animation_for_window(sources, window_days, make_gif=True, make_mp4=True):
    starts = build_window_starts(START_MMDD, END_MMDD)

    out_dir = os.path.join(PLOT_BASE, "ANIMATED_SCATTER_GIF", f"{window_days}d")
    frame_dir = os.path.join(out_dir, "frames")
    os.makedirs(frame_dir, exist_ok=True)

    frame_paths = []

    for i, (m, d) in enumerate(starts):
        frame_path = os.path.join(frame_dir, f"frame_{i:03d}_{m:02d}-{d:02d}.png")
        print(f"[{window_days}d] Rendering frame {i+1}/{len(starts)} : {m:02d}-{d:02d}")
        plot_one_frame(
            sources=sources,
            start_month=m,
            start_day=d,
            window_days=window_days,
            save_path=frame_path
        )
        frame_paths.append(frame_path)

    if make_gif:
        gif_path = os.path.join(out_dir, f"AnimatedScatter_{window_days}d_MERRA2_B2000_NOCOUPL.gif")
        make_gif_from_frames(frame_paths, gif_path, duration=FRAME_DURATION)

    if make_mp4:
        mp4_path = os.path.join(out_dir, f"AnimatedScatter_{window_days}d_MERRA2_B2000_NOCOUPL.mp4")
        make_mp4_from_frames(frame_paths, mp4_path, fps=MP4_FPS)


# ============================================================
# Run
# ============================================================
sources = build_sources()

try:
    for wd in WINDOW_DAYS_LIST:
        make_animation_for_window(
            sources=sources,
            window_days=wd,
            make_gif=True,
            make_mp4=True
        )
finally:
    close_sources(sources)

In [ ]:
# ============================================================
# Block G (RAW VERSION + extreme-year labels)
# 使用 RAW 数据绘图，不做标准化
# X 轴显示为 10^-3 hPa m s^-2
# 手动固定坐标轴：
#   xlim = MANUAL_XLIM
#   ylim = MANUAL_YLIM
#
# 新增：
# 1) 每个 panel 标记 TopN 个极端年（默认按 y_raw 最低 = O3 最低）
# 2) coupled 面板中标签显示为 part-year，避免 BWCN / B2000WCN 混淆
# 3) GIF 和 MP4 都整合在主流程中
# ============================================================

import os
import glob
from datetime import timedelta, date

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import imageio.v2 as imageio


# ============================================================
# 基础配置
# ============================================================
FILE_BWCN_EP     = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP    = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
FILE_NOCOUPL_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"
FILE_MERRA_EP    = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

# 新版输出目录，避免覆盖旧版
PLOT_BASE_RAW = os.path.join(PLOT_BASE, "ANIMATED_SCATTER_RAW_X1E3_MANUALLIM_EXTREME")

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

LAT_BAND = (40, 80)
APPLY_O3_5D = True

# 时间轴：n-1年09-01 到 n年02-01
START_MMDD = (9, 1)
END_MMDD   = (2, 1)

# 前一年起点月份
PREV_YEAR_MONTHS = {9, 10, 11, 12}

# GIF / MP4 设置
FRAME_DURATION = 1.0
MP4_FPS = 10
FIG_DPI = 220

# 点样式
BG_COLOR = "0.35"
LOW25_COLOR = "red"
BG_ALPHA = 0.75
LOW25_ALPHA = 0.95
BG_SIZE = 34
LOW25_SIZE = 42

# ---------------- 手动固定坐标轴 ----------------
USE_MANUAL_AXIS_LIMITS = True
MANUAL_XLIM = (0, 5)
MANUAL_YLIM = (60, 170)

# X 轴显示缩放：原始 -Fz 乘以 1000 后绘图
X_PLOT_SCALE = 1.0e3

# ---------------- 极端年标记配置 ----------------
MARK_EXTREMES = True
EXTREME_TOPN = 5
EXTREME_BY = "y_raw"          # 按原始 O3 最低年定义极端
EXTREME_ASCENDING = True      # True = 取最小值 = 最低 O3
EXTREME_SIZE = 120
EXTREME_LINEWIDTH = 1.6
EXTREME_COLORS = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']


# ============================================================
# Helpers
# ============================================================
def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})


def select_100hpa_level(da_plev):
    """
    自动兼容 plev 是 Pa 还是 hPa：
    - 若最大值 > 2000，视为 Pa，取 10000
    - 否则视为 hPa，取 100
    """
    pvals = np.asarray(da_plev["plev"].values, dtype=float)
    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0
        unit_guess = "Pa"
    else:
        target = 100.0
        unit_guess = "hPa"

    out = da_plev.sel(plev=target, method="nearest")
    print(f"  Selected 100 hPa level using target={target:g} ({unit_guess} interpretation).")
    return out


def assert_daily_unique(da, name=""):
    """
    兼容 cftime / datetime64 的 daily 唯一性检查
    """
    da = da.sortby("time")

    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)

    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")

    return da


def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)


def uses_previous_year(start_month):
    return start_month in PREV_YEAR_MONTHS


def should_drop_bridge_year(is_bridge_case, start_month):
    """
    对 bridge case：
    只要窗口起点在前一年（9/10/11/12月），
    就删除 target year = 104。
    """
    return is_bridge_case and uses_previous_year(start_month)


def build_window_starts(start_mmdd=(9, 1), end_mmdd=(2, 1)):
    """
    用普通日期只生成 month-day 序列
    """
    start = date(2000, start_mmdd[0], start_mmdd[1])
    end   = date(2001, end_mmdd[0], end_mmdd[1])

    out = []
    cur = start
    while cur <= end:
        out.append((cur.month, cur.day))
        cur += timedelta(days=1)
    return out


def add_days_to_time(t, ndays):
    """
    同时兼容 cftime / python datetime / numpy datetime64
    """
    try:
        return t + timedelta(days=ndays)
    except Exception:
        return t + np.timedelta64(ndays, "D")


def get_exact_time_value(da, year, month, day):
    mask = (
        (da.time.dt.year == year) &
        (da.time.dt.month == month) &
        (da.time.dt.day == day)
    )
    tt = da.time.where(mask, drop=True)
    if tt.size == 0:
        return None
    return tt.values[0]


def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    """
    目标变量：MA 的 5-day running mean minimum O3
    """
    o3_da = assert_daily_unique(o3_da, name="O3").sortby("time")

    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(o3_da.time.dt.year.values.astype(int))

    out_years = []
    out_vals = []

    for yr in years:
        mask = (
            (o3_da.time.dt.year == yr) &
            (o3_da.time.dt.month.isin([3, 4]))
        )
        seg = o3_da.where(mask, drop=True)

        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid < 58:
            continue

        out_years.append(int(yr))
        out_vals.append(float(seg.min().values))

    return xr.DataArray(
        np.array(out_vals, dtype=float),
        coords={"year": np.array(out_years, dtype=int)},
        dims=("year",),
        name="MA_min_O3"
    )


def get_window_mean_series(da, start_month, start_day, window_days):
    """
    对每个 target year，取固定长度滑动窗口均值
    """
    da = assert_daily_unique(da, name="Fz").sortby("time")
    years = np.unique(da.time.dt.year.values.astype(int))

    out_years = []
    out_vals = []

    for yr in years:
        start_year = int(yr) - 1 if uses_previous_year(start_month) else int(yr)

        start_time = get_exact_time_value(da, start_year, start_month, start_day)
        if start_time is None:
            continue

        end_time = add_days_to_time(start_time, window_days - 1)

        win = da.sel(time=slice(start_time, end_time))
        n_valid = int(win.count().values) if win.size > 0 else 0

        if n_valid < window_days:
            continue

        out_years.append(int(yr))
        out_vals.append(float(win.mean().values))

    return xr.DataArray(
        np.array(out_vals, dtype=float),
        coords={"year": np.array(out_years, dtype=int)},
        dims=("year",),
        name=f"window_{window_days}d_mean"
    )


def load_one_source(ep_nc, o3_nc, low25_txt=None, lat_band=(40, 80),
                    apply_o3_5d=True, drop_leap_day=False):
    """
    加载一个实验源，返回：
    {
      "ds": ds_ep,
      "fz_raw": 100 hPa, lat-mean, sign corrected (-Fz),
      "y_raw": MA minimum O3,
      "low25": low25 years or None
    }
    """
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        raise FileNotFoundError(f"Missing file(s): {ep_nc} or {o3_nc}")

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    # ---- Fz ----
    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"
    fz_raw = ds_ep[var_fz]

    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # 与前面保持一致：使用 -Fz
    fz_raw = -1.0 * fz_raw

    if drop_leap_day:
        fz_raw = drop_feb29(fz_raw)
        da_o3  = drop_feb29(da_o3)

    fz_raw = assert_daily_unique(fz_raw, name=os.path.basename(ep_nc))
    da_o3  = assert_daily_unique(da_o3,  name=os.path.basename(o3_nc))

    # ---- O3 ----
    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    # ---- LOW25 ----
    low25 = None
    if low25_txt is not None and os.path.exists(low25_txt):
        low25 = np.atleast_1d(np.loadtxt(low25_txt, dtype=int))

    return {
        "ds": ds_ep,
        "fz_raw": fz_raw,
        "y_raw": y_raw,
        "low25": low25,
    }


def make_df_for_window(source_dict, part_name, start_month, start_day, window_days,
                       is_bridge_case=False):
    """
    给定某个 source，生成该窗口下的散点样本表
    columns = [part, year, x_raw, y_raw, is_low25]
    """
    x_raw = get_window_mean_series(source_dict["fz_raw"], start_month, start_day, window_days)
    y_raw = source_dict["y_raw"]

    if should_drop_bridge_year(is_bridge_case, start_month):
        if BRIDGE_YEAR in x_raw.year.values:
            x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

    common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
    if len(common_years) == 0:
        return None

    x_use = x_raw.sel(year=common_years)
    y_use = y_raw.sel(year=common_years)

    low25 = source_dict["low25"]
    if low25 is None:
        is_low25 = np.zeros(len(common_years), dtype=bool)
    else:
        is_low25 = np.isin(common_years, low25)

    valid = np.isfinite(x_use.values) & np.isfinite(y_use.values)
    if not np.any(valid):
        return None

    df = {
        "part": np.array([part_name] * valid.sum()),
        "year": x_use.year.values[valid].astype(int),
        "x_raw": x_use.values[valid].astype(float),
        "y_raw": y_use.values[valid].astype(float),
        "is_low25": is_low25[valid],
    }
    return df


def concat_dict_rows(dict_list):
    """
    把多个字典表拼起来
    """
    dict_list = [d for d in dict_list if d is not None]
    if len(dict_list) == 0:
        return None

    out = {}
    keys = dict_list[0].keys()
    for k in keys:
        out[k] = np.concatenate([d[k] for d in dict_list])
    return out


def finalize_group_df_raw(df):
    """
    不做标准化，直接保留 raw 值用于绘图
    但 X 轴显示时乘以 1000，改成 10^-3 单位
    """
    if df is None or len(df["year"]) == 0:
        return None

    valid = np.isfinite(df["x_raw"]) & np.isfinite(df["y_raw"])
    if not np.any(valid):
        return None

    x_plot = df["x_raw"][valid].astype(float) * X_PLOT_SCALE
    y_plot = df["y_raw"][valid].astype(float)

    return {
        "part": df["part"][valid],
        "year": df["year"][valid],
        "x": x_plot,
        "y": y_plot,
        "is_low25": df["is_low25"][valid],
        "x_raw": df["x_raw"][valid].astype(float),
        "y_raw": df["y_raw"][valid].astype(float),
    }


def build_group_scatter_df(group_info, start_month, start_day, window_days):
    """
    为某一组（coupled / nocoupl / merra2）在某一帧构造 RAW 散点数据
    """
    rows = []
    for part in group_info["parts"]:
        df_part = make_df_for_window(
            source_dict=part["data"],
            part_name=part["name"],
            start_month=start_month,
            start_day=start_day,
            window_days=window_days,
            is_bridge_case=part["is_bridge_case"]
        )
        rows.append(df_part)

    df_all = concat_dict_rows(rows)
    df_all = finalize_group_df_raw(df_all)
    return df_all


def save_axis_limits_txt(out_dir, window_days, xlim, ylim):
    os.makedirs(out_dir, exist_ok=True)
    txt_path = os.path.join(out_dir, f"RAW_axis_limits_{window_days}d.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"window_days = {window_days}\n")
        f.write(f"xlim = ({xlim[0]:.10g}, {xlim[1]:.10g})\n")
        f.write(f"ylim = ({ylim[0]:.10g}, {ylim[1]:.10g})\n")
        f.write(f"x_plot_scale = {X_PLOT_SCALE}\n")
        f.write("x axis label unit = 10^-3 hPa m s^-2\n")
        f.write("manual axis limits are enabled\n")
        f.write(f"mark_extremes = {MARK_EXTREMES}\n")
        f.write(f"extreme_topn = {EXTREME_TOPN}\n")
        f.write(f"extreme_by = {EXTREME_BY}\n")
        f.write(f"extreme_ascending = {EXTREME_ASCENDING}\n")
    print("Saved axis-limit record:", txt_path)


def build_sources():
    """
    三条线：
    1) coupled  = BWCN + B2000WCN
    2) nocoupl  = B2000WCN_NOCOUPL
    3) merra2   = MERRA2
    """
    sources = {
        "coupled": {
            "label": "B2000WCN",
            "parts": [
                {
                    "name": "BWCN",
                    "is_bridge_case": False,
                    "data": load_one_source(
                        FILE_BWCN_EP,
                        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "BWCN", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
                {
                    "name": "B2000WCN",
                    "is_bridge_case": True,
                    "data": load_one_source(
                        FILE_B2000_EP,
                        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "B2000WCN", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
            ]
        },
        "nocoupl": {
            "label": "B2000WCN NOCOUPL",
            "parts": [
                {
                    "name": "B2000WCN_NOCOUPL",
                    "is_bridge_case": True,
                    "data": load_one_source(
                        FILE_NOCOUPL_EP,
                        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        },
        "merra2": {
            "label": "MERRA2",
            "parts": [
                {
                    "name": "MERRA2",
                    "is_bridge_case": False,
                    "data": load_one_source(
                        FILE_MERRA_EP,
                        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
                        low25_txt=os.path.join(DATA_BASE, "MERRA2", "low25pct_years_30_70hPa.txt"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        }
    }
    return sources


def close_sources(sources):
    for group in sources.values():
        for part in group["parts"]:
            ds = part["data"].get("ds", None)
            if ds is not None:
                ds.close()


def select_panel_extremes(df, topn=5, extreme_by="y_raw", ascending=True):
    """
    从某个 panel 当前帧的数据中，挑选极端年
    返回索引数组（相对于 df 内部）
    """
    if df is None or len(df["year"]) == 0:
        return np.array([], dtype=int)

    vals = np.asarray(df[extreme_by], dtype=float)
    valid_idx = np.where(np.isfinite(vals))[0]
    if valid_idx.size == 0:
        return np.array([], dtype=int)

    order = np.argsort(vals[valid_idx])
    if not ascending:
        order = order[::-1]

    keep = valid_idx[order[:min(topn, len(order))]]
    return keep


def overlay_panel_extremes(ax, df, idx_extreme):
    """
    只画极端点，不在点旁边直接写文字。
    返回 legend_entries 供 draw_one_panel() 统一生成图例。
    """
    if df is None or len(idx_extreme) == 0:
        return []

    legend_entries = []

    for i, idx in enumerate(idx_extreme):
        c = EXTREME_COLORS[i % len(EXTREME_COLORS)]

        x = float(df["x"][idx])
        y = float(df["y"][idx])
        part = str(df["part"][idx])
        year = int(df["year"][idx])

        # 标签规则：
        # - coupled 里默认年份表示 B2000WCN
        # - 只有 BWCN 额外写 BWCN-
        # - nocoupl / merra2 只写年份
        if part == "BWCN":
            label = f"BWCN-{year:04d}"
        else:
            label = f"{year:04d}"

        legend_entries.append((i, label, c, part))

        # 点样式：
        # - BWCN 仍然保持空心
        # - 其余保持实心
        if part == "BWCN":
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                facecolors="none",
                edgecolors=c,
                linewidths=EXTREME_LINEWIDTH,
                zorder=10
            )
        else:
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                color=c,
                edgecolors="k",
                linewidths=0.8,
                zorder=10
            )

    return legend_entries


def draw_one_panel(ax, df, panel_title, xlim, ylim):
    from matplotlib.lines import Line2D

    ax.grid(True, linestyle=":", alpha=0.45)
    ax.axhline(0, color="k", lw=0.8, alpha=0.45)
    ax.axvline(0, color="k", lw=0.8, alpha=0.45)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    if df is None or len(df["year"]) == 0:
        ax.set_title(panel_title, fontweight="bold")
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center",
                transform=ax.transAxes, fontsize=11)
        ax.set_xlabel(r"Window mean $-F_z$ ($10^{-3}$ hPa m s$^{-2}$)")
        ax.set_ylabel(r"MA min O$_3$ (raw)")
        return

    low_mask = df["is_low25"]
    bg_mask = ~low_mask

    # 普通点
    if np.any(bg_mask):
        ax.scatter(
            df["x"][bg_mask], df["y"][bg_mask],
            s=BG_SIZE, color=BG_COLOR, alpha=BG_ALPHA,
            edgecolors="none", zorder=2
        )

    # low25
    if np.any(low_mask):
        ax.scatter(
            df["x"][low_mask], df["y"][low_mask],
            s=LOW25_SIZE, color=LOW25_COLOR, alpha=LOW25_ALPHA,
            edgecolors="k", linewidths=0.4, zorder=5
        )

    # 极端年：只画点，并收集 legend 信息
    legend_entries = []
    if MARK_EXTREMES:
        idx_ext = select_panel_extremes(
            df,
            topn=EXTREME_TOPN,
            extreme_by=EXTREME_BY,
            ascending=EXTREME_ASCENDING
        )
        legend_entries = overlay_panel_extremes(ax, df, idx_ext)

    # ALL 统计
    n_all = len(df["x"])
    if n_all >= 3:
        try:
            r_all, p_all = pearsonr(df["x"], df["y"])
            txt_all = f"ALL\nr = {r_all:.3f}\np = {p_all:.2e}\nN = {n_all}"
        except Exception:
            txt_all = f"ALL\nN = {n_all}"
    else:
        txt_all = f"ALL\nN = {n_all}"

    ax.text(
        0.04, 0.96, txt_all,
        transform=ax.transAxes, va="top", ha="left", fontsize=9.5,
        color="k",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor="0.7")
    )

    # LOW25 统计
    n_low = int(np.sum(low_mask))
    if n_low >= 3:
        try:
            r_low, p_low = pearsonr(df["x"][low_mask], df["y"][low_mask])
            txt_low = f"LOW25\nr = {r_low:.3f}\np = {p_low:.2e}\nN = {n_low}"
        except Exception:
            txt_low = f"LOW25\nN = {n_low}"
    else:
        txt_low = f"LOW25\nN = {n_low}"

    ax.text(
        0.96, 0.96, txt_low,
        transform=ax.transAxes, va="top", ha="right", fontsize=9.5,
        color=LOW25_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor=LOW25_COLOR)
    )

    # 右下角极端年 legend
    if legend_entries:
        handles = []
        labels = []

        for i, label, c, part in legend_entries:
            if part == "BWCN":
                h = Line2D(
                    [0], [0],
                    marker="o",
                    color="none",
                    markeredgecolor=c,
                    markerfacecolor="none",
                    markeredgewidth=1.2,
                    markersize=5.0,
                    linewidth=0
                )
            else:
                h = Line2D(
                    [0], [0],
                    marker="o",
                    color="none",
                    markeredgecolor="k",
                    markerfacecolor=c,
                    markeredgewidth=0.6,
                    markersize=5.0,
                    linewidth=0
                )

            handles.append(h)
            labels.append(f"Top{i+1}: {label}")

        ax.legend(
            handles, labels,
            loc="lower right",
            fontsize=4.2,   # 原来 8.2 左右的一半
            frameon=False,
            handletextpad=0.35,
            borderpad=0.2,
            labelspacing=0.25
        )

    ax.set_title(panel_title, fontweight="bold")
    ax.set_xlabel(r"Window mean $-F_z$ ($10^{-3}$ hPa m s$^{-2}$)")
    ax.set_ylabel(r"MA min O$_3$ (raw)")


def plot_one_frame(sources, start_month, start_day, window_days, save_path, xlim, ylim):
    """
    画单帧三实验组图
    布局：
      第一行中间：MERRA2
      第二行左：Coupled
      第二行右：NOCOUPL
    """
    df_merra   = build_group_scatter_df(sources["merra2"],  start_month, start_day, window_days)
    df_coupled = build_group_scatter_df(sources["coupled"], start_month, start_day, window_days)
    df_noc     = build_group_scatter_df(sources["nocoupl"], start_month, start_day, window_days)

    fig = plt.figure(figsize=(12.6, 9.0), dpi=160)
    gs = fig.add_gridspec(
        2, 3,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.0, 1.0, 1.0],
        hspace=0.38, wspace=0.32
    )

    ax_top = fig.add_subplot(gs[0, 1])
    ax_bl  = fig.add_subplot(gs[1, 0])
    ax_br  = fig.add_subplot(gs[1, 2])

    draw_one_panel(ax_top, df_merra,   "MERRA2",           xlim, ylim)
    draw_one_panel(ax_bl,  df_coupled, "B2000WCN",         xlim, ylim)
    draw_one_panel(ax_br,  df_noc,     "B2000WCN NOCOUPL", xlim, ylim)

    fig.suptitle(
        f"RAW scatter | {window_days}-day window | Start date = {start_month:02d}-{start_day:02d}",
        fontsize=16, fontweight="bold", y=0.97
    )
    fig.text(
        0.5, 0.935,
        f"All years shown; low25 years highlighted in red; Top {EXTREME_TOPN} lowest O3 years highlighted in each panel",
        ha="center", va="center", fontsize=11
    )

    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def make_gif_from_frames(frame_paths, gif_path, duration=1.0):
    images = [imageio.imread(fp) for fp in frame_paths]
    imageio.mimsave(gif_path, images, duration=duration, loop=0)
    print("Saved RAW GIF:", gif_path)


def make_mp4_from_frames(frame_paths, mp4_path, fps=10):
    """
    从 RAW frames 生成 MP4
    """
    print(f"Compiling RAW MP4 at {fps} FPS -> {mp4_path}")

    with imageio.get_writer(mp4_path, fps=fps, codec="libx264", quality=9, macro_block_size=None) as writer:
        for fp in frame_paths:
            img = imageio.imread(fp)

            h, w = img.shape[:2]
            new_h = h if h % 2 == 0 else h - 1
            new_w = w if w % 2 == 0 else w - 1
            img = img[:new_h, :new_w, ...]

            writer.append_data(img)

    print("Saved RAW MP4:", mp4_path)


def make_animation_for_window(sources, window_days):
    starts = build_window_starts(START_MMDD, END_MMDD)

    out_dir = os.path.join(PLOT_BASE_RAW, f"{window_days}d")
    frame_dir = os.path.join(out_dir, "frames_raw")
    os.makedirs(frame_dir, exist_ok=True)

    if USE_MANUAL_AXIS_LIMITS:
        xlim, ylim = MANUAL_XLIM, MANUAL_YLIM
        print(f"[RAW {window_days}d] Using manual xlim = {xlim}")
        print(f"[RAW {window_days}d] Using manual ylim = {ylim}")
    else:
        raise RuntimeError("This script is configured for manual axis limits only.")

    save_axis_limits_txt(out_dir, window_days, xlim, ylim)

    frame_paths = []

    for i, (m, d) in enumerate(starts):
        frame_path = os.path.join(frame_dir, f"frame_raw_{i:03d}_{m:02d}-{d:02d}.png")
        print(f"[RAW {window_days}d] Rendering frame {i+1}/{len(starts)} : {m:02d}-{d:02d}")

        plot_one_frame(
            sources=sources,
            start_month=m,
            start_day=d,
            window_days=window_days,
            save_path=frame_path,
            xlim=xlim,
            ylim=ylim
        )
        frame_paths.append(frame_path)

    gif_path = os.path.join(out_dir, f"AnimatedScatter_RAW_X1E3_MANUALLIM_EXTREME_{window_days}d_MERRA2_B2000_NOCOUPL.gif")
    mp4_path = os.path.join(out_dir, f"AnimatedScatter_RAW_X1E3_MANUALLIM_EXTREME_{window_days}d_MERRA2_B2000_NOCOUPL.mp4")

    make_gif_from_frames(frame_paths, gif_path, duration=FRAME_DURATION)
    make_mp4_from_frames(frame_paths, mp4_path, fps=MP4_FPS)

    return gif_path, mp4_path


# ============================================================
# Run
# ============================================================
if __name__ == "__main__":
    os.makedirs(PLOT_BASE_RAW, exist_ok=True)

    sources = build_sources()

    try:
        for wd in WINDOW_DAYS_LIST:
            make_animation_for_window(sources, wd)
    finally:
        close_sources(sources)

    print("\n[RAW animated scatter with x scaled to 1e-3, manual limits, and extreme-year labels] All done.")

In [ ]:
# ============================================================
# Block G (NEW RAW + MAM-MONTH COLOR VERSION, manual axis + x*1000)
# Animated scatter GIFs / MP4
# ------------------------------------------------------------
# 特点：
# 1) 使用 RAW 数据绘图，不做标准化
# 2) 不再使用 low25% 红点逻辑
# 3) Y 轴改为 MAM (Mar-Apr-May) minimum O3
# 4) 所有点按 minimum O3 出现月份着色：
#       March -> blue
#       April -> gold
#       May   -> red
# 5) 统计信息：
#       ALL / MAR / APR / MAY
#    只显示 R 和 N
# 6) 手动固定坐标轴：
#       xlim = (-10, 10)
#       ylim = (0, 150)
# 7) X 轴显示单位改为 10^-3 hPa m s^-2（即 x_raw * 1000）
# 8) 输出路径、文件名均与旧版区分
# ============================================================

import os
import glob
from datetime import timedelta, date

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr
import imageio.v2 as imageio

# ---------------- 基础配置 ----------------
FILE_BWCN_EP     = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP    = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
FILE_NOCOUPL_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"
FILE_MERRA_EP    = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

# 新版输出目录（不覆盖旧版）
PLOT_BASE_NEW = os.path.join(PLOT_BASE, "ANIMATED_SCATTER_RAW_MAM_MONTHCOLOR_X1E3_MANUALLIM")

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

LAT_BAND = (40, 80)
APPLY_O3_5D = True

# 时间轴：n-1年09-01 到 n年02-01
START_MMDD = (9, 1)
END_MMDD   = (2, 1)

# 前一年起点月份
PREV_YEAR_MONTHS = {9, 10, 11, 12}

# GIF / MP4 设置
FRAME_DURATION = 1.0
MP4_FPS = 10
FIG_DPI = 220

# MAM 有效天数阈值（允许极少量缺测）
MAM_VALID_MIN_DAYS = 88

# 点样式：按 minimum ozone month 着色
MONTH_COLOR = {
    3: "royalblue",   # March
    4: "gold",        # April
    5: "red",         # May
}
MONTH_LABEL = {
    3: "March minimum",
    4: "April minimum",
    5: "May minimum",
}
MONTH_SHORT = {
    3: "MAR",
    4: "APR",
    5: "MAY",
}

POINT_SIZE = 36
POINT_ALPHA = 0.88
POINT_EDGE_LW = 0.25

# ---------------- 手动固定坐标轴 ----------------
USE_MANUAL_AXIS_LIMITS = True
MANUAL_XLIM = (0, 5)
MANUAL_YLIM = (60, 170)

# X 轴显示为 10^-3 单位
X_PLOT_SCALE = 1.0e3


# ============================================================
# Helpers
# ============================================================
def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})

def select_100hpa_level(da_plev):
    """
    自动兼容 plev 是 Pa 还是 hPa：
    - 若最大值 > 2000，视为 Pa，取 10000
    - 否则视为 hPa，取 100
    """
    pvals = np.asarray(da_plev["plev"].values, dtype=float)
    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0
        unit_guess = "Pa"
    else:
        target = 100.0
        unit_guess = "hPa"

    out = da_plev.sel(plev=target, method="nearest")
    print(f"  Selected 100 hPa level using target={target:g} ({unit_guess} interpretation).")
    return out

def assert_daily_unique(da, name=""):
    """
    兼容 cftime / datetime64 的 daily 唯一性检查
    """
    da = da.sortby("time")

    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)

    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")

    return da

def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)

def uses_previous_year(start_month):
    return start_month in PREV_YEAR_MONTHS

def should_drop_bridge_year(is_bridge_case, start_month):
    """
    对 bridge case：
    只要窗口起点在前一年（9/10/11/12月），
    就删除 target year = 104。
    """
    return is_bridge_case and uses_previous_year(start_month)

def build_window_starts(start_mmdd=(9, 1), end_mmdd=(2, 1)):
    """
    用普通日期只生成 month-day 序列
    """
    start = date(2000, start_mmdd[0], start_mmdd[1])
    end   = date(2001, end_mmdd[0], end_mmdd[1])

    out = []
    cur = start
    while cur <= end:
        out.append((cur.month, cur.day))
        cur += timedelta(days=1)
    return out

def add_days_to_time(t, ndays):
    """
    同时兼容 cftime / python datetime / numpy datetime64
    """
    try:
        return t + timedelta(days=ndays)
    except Exception:
        return t + np.timedelta64(ndays, "D")

def get_exact_time_value(da, year, month, day):
    mask = (
        (da.time.dt.year == year) &
        (da.time.dt.month == month) &
        (da.time.dt.day == day)
    )
    tt = da.time.where(mask, drop=True)
    if tt.size == 0:
        return None
    return tt.values[0]

def safe_corr(x, y):
    """
    至少 3 个样本、且 x/y 不能是常数，才计算 Pearson r
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    n = len(x)

    if n < 3:
        return np.nan, n

    if np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan, n

    try:
        r, _ = pearsonr(x, y)
        return float(r), n
    except Exception:
        return np.nan, n

def format_rn_text(label, x, y):
    r, n = safe_corr(x, y)
    rtxt = "nan" if not np.isfinite(r) else f"{r:.3f}"
    return f"{label}\nR = {rtxt}\nN = {n}"

def get_mam_min_o3_dataset(o3_da, apply_o3_5d=True, min_valid_days=88):
    """
    目标变量：MAM 的 5-day running mean minimum O3
    同时记录该 minimum 出现在 3/4/5 月中的哪一个月
    返回：
      xr.Dataset with vars:
        - y_raw(year): MAM minimum O3
        - min_month(year): 3 / 4 / 5
    """
    o3_da = assert_daily_unique(o3_da, name="O3").sortby("time")

    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(o3_da.time.dt.year.values.astype(int))

    out_years = []
    out_vals = []
    out_months = []

    for yr in years:
        mask = (
            (o3_da.time.dt.year == yr) &
            (o3_da.time.dt.month.isin([3, 4, 5]))
        )
        seg = o3_da.where(mask, drop=True)

        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid < min_valid_days:
            continue

        vals = np.asarray(seg.values, dtype=float)
        valid = np.isfinite(vals)
        if not np.any(valid):
            continue

        valid_idx = np.where(valid)[0]
        idx_local = valid_idx[np.argmin(vals[valid])]

        min_val = float(vals[idx_local])
        min_month = int(seg.time.dt.month.values[idx_local])

        if min_month not in [3, 4, 5]:
            continue

        out_years.append(int(yr))
        out_vals.append(min_val)
        out_months.append(min_month)

    return xr.Dataset(
        data_vars={
            "y_raw": ("year", np.array(out_vals, dtype=float)),
            "min_month": ("year", np.array(out_months, dtype=int)),
        },
        coords={"year": np.array(out_years, dtype=int)}
    )

def get_window_mean_series(da, start_month, start_day, window_days):
    """
    对每个 target year，取固定长度滑动窗口均值
    """
    da = assert_daily_unique(da, name="Fz").sortby("time")
    years = np.unique(da.time.dt.year.values.astype(int))

    out_years = []
    out_vals = []

    for yr in years:
        start_year = int(yr) - 1 if uses_previous_year(start_month) else int(yr)

        start_time = get_exact_time_value(da, start_year, start_month, start_day)
        if start_time is None:
            continue

        end_time = add_days_to_time(start_time, window_days - 1)

        win = da.sel(time=slice(start_time, end_time))
        n_valid = int(win.count().values) if win.size > 0 else 0

        if n_valid < window_days:
            continue

        out_years.append(int(yr))
        out_vals.append(float(win.mean().values))

    return xr.DataArray(
        np.array(out_vals, dtype=float),
        coords={"year": np.array(out_years, dtype=int)},
        dims=("year",),
        name=f"window_{window_days}d_mean"
    )

def load_one_source(ep_nc, o3_nc, lat_band=(40, 80),
                    apply_o3_5d=True, drop_leap_day=False):
    """
    加载一个实验源，返回：
    {
      "ds": ds_ep,
      "fz_raw": 100 hPa, lat-mean, sign corrected (-Fz),
      "y_ds": xr.Dataset(year -> y_raw, min_month)
    }
    """
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        raise FileNotFoundError(f"Missing file(s): {ep_nc} or {o3_nc}")

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    # ---- Fz ----
    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"
    fz_raw = ds_ep[var_fz]

    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # 与前面保持一致：使用 -Fz
    fz_raw = -1.0 * fz_raw

    if drop_leap_day:
        fz_raw = drop_feb29(fz_raw)
        da_o3  = drop_feb29(da_o3)

    fz_raw = assert_daily_unique(fz_raw, name=os.path.basename(ep_nc))
    da_o3  = assert_daily_unique(da_o3,  name=os.path.basename(o3_nc))

    # ---- O3: MAM minimum + month ----
    y_ds = get_mam_min_o3_dataset(
        da_o3,
        apply_o3_5d=apply_o3_5d,
        min_valid_days=MAM_VALID_MIN_DAYS
    )

    return {
        "ds": ds_ep,
        "fz_raw": fz_raw,
        "y_ds": y_ds,
    }

def make_df_for_window(source_dict, part_name, start_month, start_day, window_days,
                       is_bridge_case=False):
    """
    给定某个 source，生成该窗口下的散点样本表
    columns = [part, year, x_raw, y_raw, min_month]
    """
    x_raw = get_window_mean_series(source_dict["fz_raw"], start_month, start_day, window_days)
    y_ds = source_dict["y_ds"]

    if should_drop_bridge_year(is_bridge_case, start_month):
        if BRIDGE_YEAR in x_raw.year.values:
            x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

    common_years = np.intersect1d(x_raw.year.values, y_ds.year.values)
    if len(common_years) == 0:
        return None

    x_use = x_raw.sel(year=common_years)
    y_use = y_ds["y_raw"].sel(year=common_years)
    m_use = y_ds["min_month"].sel(year=common_years)

    valid = (
        np.isfinite(x_use.values) &
        np.isfinite(y_use.values) &
        np.isin(m_use.values, [3, 4, 5])
    )
    if not np.any(valid):
        return None

    df = {
        "part": np.array([part_name] * valid.sum()),
        "year": x_use.year.values[valid].astype(int),
        "x_raw": x_use.values[valid].astype(float),
        "y_raw": y_use.values[valid].astype(float),
        "min_month": m_use.values[valid].astype(int),
    }
    return df

def concat_dict_rows(dict_list):
    """
    把多个字典表拼起来
    """
    dict_list = [d for d in dict_list if d is not None]
    if len(dict_list) == 0:
        return None

    out = {}
    keys = dict_list[0].keys()
    for k in keys:
        out[k] = np.concatenate([d[k] for d in dict_list])
    return out

def finalize_group_df_raw(df):
    """
    不做标准化，直接保留 raw 值用于绘图
    但 X 轴显示时乘以 1000，改成 10^-3 单位
    """
    if df is None or len(df["year"]) == 0:
        return None

    valid = (
        np.isfinite(df["x_raw"]) &
        np.isfinite(df["y_raw"]) &
        np.isin(df["min_month"], [3, 4, 5])
    )
    if not np.any(valid):
        return None

    x_plot = df["x_raw"][valid].astype(float) * X_PLOT_SCALE
    y_plot = df["y_raw"][valid].astype(float)

    return {
        "part": df["part"][valid],
        "year": df["year"][valid],
        "x": x_plot,
        "y": y_plot,
        "min_month": df["min_month"][valid].astype(int),
        "x_raw": df["x_raw"][valid].astype(float),
        "y_raw": df["y_raw"][valid].astype(float),
    }

def build_group_scatter_df(group_info, start_month, start_day, window_days):
    """
    为某一组（coupled / nocoupl / merra2）在某一帧构造 RAW 散点数据
    """
    rows = []
    for part in group_info["parts"]:
        df_part = make_df_for_window(
            source_dict=part["data"],
            part_name=part["name"],
            start_month=start_month,
            start_day=start_day,
            window_days=window_days,
            is_bridge_case=part["is_bridge_case"]
        )
        rows.append(df_part)

    df_all = concat_dict_rows(rows)
    df_all = finalize_group_df_raw(df_all)
    return df_all

def save_axis_limits_txt(out_dir, window_days, xlim, ylim):
    os.makedirs(out_dir, exist_ok=True)
    txt_path = os.path.join(out_dir, f"RAW_MAMmonth_axis_limits_{window_days}d.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"window_days = {window_days}\n")
        f.write(f"xlim = ({xlim[0]:.10g}, {xlim[1]:.10g})\n")
        f.write(f"ylim = ({ylim[0]:.10g}, {ylim[1]:.10g})\n")
        f.write(f"x_plot_scale = {X_PLOT_SCALE}\n")
        f.write("x axis label unit = 10^-3 hPa m s^-2\n")
        f.write("color map: March=royalblue, April=gold, May=red\n")
        f.write("manual axis limits are enabled\n")
    print("Saved axis-limit record:", txt_path)

def build_sources():
    """
    三条线：
    1) coupled  = BWCN + B2000WCN
    2) nocoupl  = B2000WCN_NOCOUPL
    3) merra2   = MERRA2
    """
    sources = {
        "coupled": {
            "label": "B2000WCN",
            "parts": [
                {
                    "name": "BWCN",
                    "is_bridge_case": False,
                    "data": load_one_source(
                        FILE_BWCN_EP,
                        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
                {
                    "name": "B2000WCN",
                    "is_bridge_case": True,
                    "data": load_one_source(
                        FILE_B2000_EP,
                        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
            ]
        },
        "nocoupl": {
            "label": "B2000WCN NOCOUPL",
            "parts": [
                {
                    "name": "B2000WCN_NOCOUPL",
                    "is_bridge_case": True,
                    "data": load_one_source(
                        FILE_NOCOUPL_EP,
                        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        },
        "merra2": {
            "label": "MERRA2",
            "parts": [
                {
                    "name": "MERRA2",
                    "is_bridge_case": False,
                    "data": load_one_source(
                        FILE_MERRA_EP,
                        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
                        lat_band=LAT_BAND,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        }
    }
    return sources

def close_sources(sources):
    for group in sources.values():
        for part in group["parts"]:
            ds = part["data"].get("ds", None)
            if ds is not None:
                ds.close()

def draw_one_panel(ax, df, panel_title, xlim, ylim):
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.axhline(0, color="k", lw=0.8, alpha=0.45)
    ax.axvline(0, color="k", lw=0.8, alpha=0.45)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    if df is None or len(df["year"]) == 0:
        ax.set_title(panel_title, fontweight="bold")
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center",
                transform=ax.transAxes, fontsize=11)
        ax.set_xlabel(r"Window mean $-F_z$ ($10^{-3}$ hPa m s$^{-2}$)")
        ax.set_ylabel(r"MAM min O$_3$ (5-day mean, raw)")
        return

    # 按 minimum month 上色
    for month in [3, 4, 5]:
        mask = df["min_month"] == month
        if np.any(mask):
            ax.scatter(
                df["x"][mask], df["y"][mask],
                s=POINT_SIZE,
                color=MONTH_COLOR[month],
                alpha=POINT_ALPHA,
                edgecolors="k",
                linewidths=POINT_EDGE_LW,
                zorder=3
            )

    # 统计：ALL / MAR / APR / MAY
    txt_all = format_rn_text("ALL", df["x"], df["y"])

    mask_mar = df["min_month"] == 3
    mask_apr = df["min_month"] == 4
    mask_may = df["min_month"] == 5

    txt_mar = format_rn_text("MAR", df["x"][mask_mar], df["y"][mask_mar])
    txt_apr = format_rn_text("APR", df["x"][mask_apr], df["y"][mask_apr])
    txt_may = format_rn_text("MAY", df["x"][mask_may], df["y"][mask_may])

    ax.text(
        0.04, 0.96, txt_all,
        transform=ax.transAxes, va="top", ha="left", fontsize=9.2,
        color="k",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.87, edgecolor="0.7")
    )

    ax.text(
        0.96, 0.96, txt_mar,
        transform=ax.transAxes, va="top", ha="right", fontsize=9.0,
        color=MONTH_COLOR[3],
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.87, edgecolor=MONTH_COLOR[3])
    )

    ax.text(
        0.04, 0.04, txt_apr,
        transform=ax.transAxes, va="bottom", ha="left", fontsize=9.0,
        color="darkgoldenrod",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.87, edgecolor="goldenrod")
    )

    ax.text(
        0.96, 0.04, txt_may,
        transform=ax.transAxes, va="bottom", ha="right", fontsize=9.0,
        color=MONTH_COLOR[5],
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.87, edgecolor=MONTH_COLOR[5])
    )

    ax.set_title(panel_title, fontweight="bold")
    ax.set_xlabel(r"Window mean $-F_z$ ($10^{-3}$ hPa m s$^{-2}$)")
    ax.set_ylabel(r"MAM min O$_3$ (5-day mean, raw)")

def plot_one_frame(sources, start_month, start_day, window_days, save_path, xlim, ylim):
    """
    画单帧三实验组图
    布局：
      第一行中间：MERRA2
      第二行左：Coupled
      第二行右：NOCOUPL
    """
    df_merra   = build_group_scatter_df(sources["merra2"],  start_month, start_day, window_days)
    df_coupled = build_group_scatter_df(sources["coupled"], start_month, start_day, window_days)
    df_noc     = build_group_scatter_df(sources["nocoupl"], start_month, start_day, window_days)

    fig = plt.figure(figsize=(12.8, 9.2), dpi=160)
    gs = fig.add_gridspec(
        2, 3,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.0, 1.0, 1.0],
        hspace=0.38, wspace=0.32
    )

    ax_top = fig.add_subplot(gs[0, 1])
    ax_bl  = fig.add_subplot(gs[1, 0])
    ax_br  = fig.add_subplot(gs[1, 2])

    draw_one_panel(ax_top, df_merra,   "MERRA2",           xlim, ylim)
    draw_one_panel(ax_bl,  df_coupled, "B2000WCN",         xlim, ylim)
    draw_one_panel(ax_br,  df_noc,     "B2000WCN NOCOUPL", xlim, ylim)

    fig.suptitle(
        f"RAW scatter | MAM minimum-month colored | {window_days}-day window | Start date = {start_month:02d}-{start_day:02d}",
        fontsize=15.5, fontweight="bold", y=0.972
    )

    legend_handles = [
        Line2D([0], [0], marker='o', color='w', label='March minimum',
               markerfacecolor=MONTH_COLOR[3], markeredgecolor='k', markersize=8),
        Line2D([0], [0], marker='o', color='w', label='April minimum',
               markerfacecolor=MONTH_COLOR[4], markeredgecolor='k', markersize=8),
        Line2D([0], [0], marker='o', color='w', label='May minimum',
               markerfacecolor=MONTH_COLOR[5], markeredgecolor='k', markersize=8),
    ]
    fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.93),
               ncol=3, frameon=False, fontsize=10.5)

    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)

def make_gif_for_window(sources, window_days):
    starts = build_window_starts(START_MMDD, END_MMDD)

    out_dir = os.path.join(PLOT_BASE_NEW, f"{window_days}d")
    frame_dir = os.path.join(out_dir, "frames_mam_monthcolor")
    os.makedirs(frame_dir, exist_ok=True)

    if USE_MANUAL_AXIS_LIMITS:
        xlim, ylim = MANUAL_XLIM, MANUAL_YLIM
        print(f"[MAM-month RAW {window_days}d] Using manual xlim = {xlim}")
        print(f"[MAM-month RAW {window_days}d] Using manual ylim = {ylim}")
    else:
        raise RuntimeError("This script is configured for manual axis limits only.")

    save_axis_limits_txt(out_dir, window_days, xlim, ylim)

    frame_paths = []

    for i, (m, d) in enumerate(starts):
        frame_path = os.path.join(frame_dir, f"frame_mamMonth_raw_{i:03d}_{m:02d}-{d:02d}.png")
        print(f"[MAM-month RAW {window_days}d] Rendering frame {i+1}/{len(starts)} : {m:02d}-{d:02d}")

        plot_one_frame(
            sources=sources,
            start_month=m,
            start_day=d,
            window_days=window_days,
            save_path=frame_path,
            xlim=xlim,
            ylim=ylim
        )
        frame_paths.append(frame_path)

    gif_path = os.path.join(out_dir, f"AnimatedScatter_RAW_MAMmonthColor_X1E3_MANUALLIM_{window_days}d_MERRA2_B2000_NOCOUPL.gif")

    images = [imageio.imread(fp) for fp in frame_paths]
    imageio.mimsave(gif_path, images, duration=FRAME_DURATION, loop=0)

    print("Saved GIF:", gif_path)
    return gif_path

def make_mp4_from_frames(window_days, fps=10):
    """
    从新版本 frames 生成 MP4
    """
    frame_dir = os.path.join(PLOT_BASE_NEW, f"{window_days}d", "frames_mam_monthcolor")
    out_dir = os.path.join(PLOT_BASE_NEW, f"{window_days}d")
    frame_paths = sorted(glob.glob(os.path.join(frame_dir, "*.png")))

    if not frame_paths:
        print(f"[MAM-month RAW {window_days}d] No frame PNG found, skip MP4.")
        return None

    mp4_path = os.path.join(out_dir, f"AnimatedScatter_RAW_MAMmonthColor_X1E3_MANUALLIM_{window_days}d_MERRA2_B2000_NOCOUPL.mp4")
    print(f"[MAM-month RAW {window_days}d] Compiling MP4 at {fps} FPS...")

    with imageio.get_writer(mp4_path, fps=fps, codec="libx264", quality=9, macro_block_size=None) as writer:
        for fp in frame_paths:
            img = imageio.imread(fp)

            h, w = img.shape[:2]
            new_h = h if h % 2 == 0 else h - 1
            new_w = w if w % 2 == 0 else w - 1
            img = img[:new_h, :new_w, ...]

            writer.append_data(img)

    print(f"Saved MP4: {mp4_path}")
    return mp4_path


# ============================================================
# Run
# ============================================================
if __name__ == "__main__":
    os.makedirs(PLOT_BASE_NEW, exist_ok=True)

    sources = build_sources()

    try:
        for wd in WINDOW_DAYS_LIST:
            make_gif_for_window(sources, wd)
            make_mp4_from_frames(wd, fps=MP4_FPS)
    finally:
        close_sources(sources)

    print("\n[RAW + MAM minimum-month colored animated scatter with x scaled to 1e-3 and manual limits] All done.")

In [ ]:
# ============================================================
# Block H (REVISED + BOOTSTRAP CI): Event-anchored comparison with MERRA2
# ------------------------------------------------------------
# 目标：
# - 每年以 MA 中 5-day running mean O3 最低值那一天作为 anchor
# - 从 anchor 往前 lead=n 天，取起始于该日的 90d / 60d Fz window
# - 计算 Fz window mean 与 MA minimum O3 的跨年相关
#
# 输出 4 张图：
#   1) 90d ALL
#   2) 90d LOW25
#   3) 60d ALL
#   4) 60d LOW25
#
# 每张图 3 条线：
#   1) B2000WCN coupled   (= BWCN + B2000WCN merged)
#   2) B2000WCN NOCOUPL
#   3) MERRA2
#
# 关键订正：
# - 动态计算 lead_days，保证不同窗口大小的结束时间严格对齐（90天前至0天前）
# - 对 B2000WCN / B2000WCN_NOCOUPL，当 target year = 104 时：
#   只要 event-anchored window 的 start date 不在 104 年内，就删除该样本
# - 新增 bootstrap 子程序：
#   对 year-wise paired samples (x_raw, y_raw) 做 bootstrap，
#   计算 Pearson r 的 95% confidence interval
# - bootstrap 特别要求：不允许抽到 B2000WCN/NOCOUPL 的 year=104
# - MERRA2 自动兼容 plev 为 Pa / hPa
# ============================================================

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_BWCN_EP     = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP    = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
FILE_NOCOUPL_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"
FILE_MERRA_EP    = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

# 统一设定：无论窗口多长，都让窗口的结束时间落在 Anchor day 前的 90天 到 0天
TARGET_END_LEAD_MAX = 90
TARGET_END_LEAD_MIN = 0

LAT_BAND = (40, 80)
APPLY_O3_5D = True
MIN_N = 5

DEBUG_BRIDGE = False   # True 时会打印 year=104 的保留/删除情况

# ---------------- Bootstrap / plotting 配置 ----------------
BOOTSTRAP_N = 2000
BOOTSTRAP_CI = 95.0
BOOTSTRAP_SEED = 42

SHOW_RIBBON = True
SHOW_ERRORBAR = True
ERRORBAR_EVERY = 6          # 每隔几个 lead 画一个 error bar
ERRORBAR_CAPSIZE = 2.2
ERRORBAR_ELW = 0.9
RIBBON_ALPHA = 0.16

SAVE_DEBUG_CSV = True


# ============================================================
# Helpers
# ============================================================
def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})


def select_100hpa_level(da_plev):
    pvals = np.asarray(da_plev["plev"].values, dtype=float)

    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0
        unit_guess = "Pa"
    else:
        target = 100.0
        unit_guess = "hPa"

    out = da_plev.sel(plev=target, method="nearest")
    print(f"  Selected 100 hPa level using target={target:g} ({unit_guess} interpretation).")
    return out


def get_ma_min_o3_and_anchor(o3_da, apply_o3_5d=True):
    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    years = o3_da.time.dt.year.values
    months = o3_da.time.dt.month.values

    year_info = {}
    unique_years = np.unique(years)

    for yr in unique_years:
        mask = (years == yr) & np.isin(months, [3, 4])
        idxs = np.where(mask)[0]

        if len(idxs) < 58:
            continue

        vals = o3_da.isel(time=idxs).values
        finite = np.isfinite(vals)
        if finite.sum() < 58:
            continue

        valid_local = np.where(finite)[0]
        valid_vals = vals[valid_local]

        best_local_in_valid = int(np.argmin(valid_vals))
        best_local = valid_local[best_local_in_valid]

        anchor_idx = int(idxs[best_local])
        y_min = float(vals[best_local])

        year_info[int(yr)] = {
            "anchor_idx": anchor_idx,
            "y_min": y_min,
        }

    return o3_da, year_info


def prepare_experiment(exp_name, ep_nc, o3_nc, low25_txt=None, lat_band=(40, 80), apply_o3_5d=True):
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        raise FileNotFoundError(f"[{exp_name}] Missing EP/O3 file.")

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"
    fz_raw = ds_ep[var_fz]

    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    fz_raw = -1.0 * fz_raw

    fz_raw = fz_raw.sortby("time")
    da_o3  = da_o3.sortby("time")
    fz_raw, o3_aligned = xr.align(fz_raw, da_o3, join="inner")

    fz_raw = fz_raw.load()
    o3_aligned = o3_aligned.load()
    fz_time = fz_raw.time.load()

    _, year_info = get_ma_min_o3_and_anchor(o3_aligned, apply_o3_5d=apply_o3_5d)

    if low25_txt is not None and os.path.exists(low25_txt):
        low25_years = set(np.atleast_1d(np.loadtxt(low25_txt, dtype=int)).tolist())
    else:
        low25_years = set()

    ds_ep.close()

    return {
        "exp": exp_name,
        "fz_raw": fz_raw,
        "fz_time": fz_time,
        "year_info": year_info,
        "low25_years": low25_years,
    }


# ============================================================
# Bootstrap 子程序
# ============================================================
def bootstrap_pearson_r_ci(
    df_samples, n_boot=2000, ci=95.0, seed=42, 
    exclude_bridge_year_from_bootstrap=True, bridge_year=104
):
    need_cols = {"exp", "year", "x_raw", "y_raw"}
    if not need_cols.issubset(df_samples.columns):
        raise ValueError(f"df_samples must contain columns: {sorted(list(need_cols))}")

    df_clean = df_samples.loc[:, ["exp", "year", "x_raw", "y_raw"]].copy()
    df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()
    n_original = len(df_clean)

    if n_original < 4:
        return {"r_obs": np.nan, "ci_low": np.nan, "ci_high": np.nan, 
                "n_original": n_original, "n_boot_pool": 0, "n_boot_valid": 0}

    x_obs = df_clean["x_raw"].to_numpy(dtype=float)
    y_obs = df_clean["y_raw"].to_numpy(dtype=float)

    if np.nanstd(x_obs) == 0 or np.nanstd(y_obs) == 0:
        return {"r_obs": np.nan, "ci_low": np.nan, "ci_high": np.nan, 
                "n_original": n_original, "n_boot_pool": 0, "n_boot_valid": 0}

    try:
        r_obs, _ = pearsonr(x_obs, y_obs)
    except Exception:
        r_obs = np.nan

    df_pool = df_clean.copy()
    if exclude_bridge_year_from_bootstrap:
        bad_mask = (df_pool["year"].eq(bridge_year) & 
                    df_pool["exp"].isin(["B2000WCN", "B2000WCN_NOCOUPL"]))
        df_pool = df_pool.loc[~bad_mask].copy()

    n_boot_pool = len(df_pool)
    if n_boot_pool < 4:
        return {"r_obs": r_obs, "ci_low": np.nan, "ci_high": np.nan, 
                "n_original": n_original, "n_boot_pool": n_boot_pool, "n_boot_valid": 0}

    x_pool = df_pool["x_raw"].to_numpy(dtype=float)
    y_pool = df_pool["y_raw"].to_numpy(dtype=float)

    rng = np.random.default_rng(seed)
    draw_idx = rng.integers(0, n_boot_pool, size=(n_boot, n_original))

    xb = x_pool[draw_idx]
    yb = y_pool[draw_idx]

    xb_mean = xb.mean(axis=1, keepdims=True)
    yb_mean = yb.mean(axis=1, keepdims=True)
    xb_anom = xb - xb_mean
    yb_anom = yb - yb_mean

    num = np.sum(xb_anom * yb_anom, axis=1)
    den = np.sqrt(np.sum(xb_anom**2, axis=1) * np.sum(yb_anom**2, axis=1))

    r_boot = np.full(n_boot, np.nan, dtype=float)
    ok = den > 0
    r_boot[ok] = num[ok] / den[ok]

    r_boot_valid = r_boot[np.isfinite(r_boot)]
    n_boot_valid = len(r_boot_valid)

    if n_boot_valid < max(100, int(0.2 * n_boot)):
        return {"r_obs": r_obs, "ci_low": np.nan, "ci_high": np.nan, 
                "n_original": n_original, "n_boot_pool": n_boot_pool, "n_boot_valid": n_boot_valid}

    alpha = 100.0 - ci
    ci_low = float(np.percentile(r_boot_valid, alpha / 2.0))
    ci_high = float(np.percentile(r_boot_valid, 100.0 - alpha / 2.0))

    return {"r_obs": float(r_obs), "ci_low": ci_low, "ci_high": ci_high, 
            "n_original": n_original, "n_boot_pool": n_boot_pool, "n_boot_valid": n_boot_valid}


# ============================================================
# Sample builder
# ============================================================
def build_samples_event_anchored(prep, lead_days, window_days, subset="all", debug_bridge=False):
    fz = prep["fz_raw"].values
    fz_time = prep["fz_time"]
    ntime = len(fz)

    rows = []
    for yr, info in prep["year_info"].items():
        if subset == "low25" and (yr not in prep["low25_years"]):
            continue

        anchor_idx = info["anchor_idx"]
        y_min = info["y_min"]

        start_idx = anchor_idx - lead_days
        end_idx = start_idx + window_days

        if start_idx < 0 or end_idx > ntime:
            continue

        if prep["exp"] in {"B2000WCN", "B2000WCN_NOCOUPL"} and (yr == BRIDGE_YEAR):
            start_year = int(fz_time.isel(time=start_idx).dt.year.item())
            if debug_bridge:
                start_time = pd.to_datetime(str(fz_time.isel(time=start_idx).values))
                anchor_time = pd.to_datetime(str(fz_time.isel(time=anchor_idx).values))
                print(f"[DEBUG bridge] {prep['exp']} | target={yr} | lead={lead_days} | "
                      f"window={window_days}d | start={start_time.date()} | "
                      f"anchor={anchor_time.date()} | start_year={start_year}")
            if start_year != BRIDGE_YEAR:
                continue

        x_vals = fz[start_idx:end_idx]
        if np.isfinite(x_vals).sum() < window_days:
            continue

        rows.append({
            "exp": prep["exp"],
            "year": int(yr),
            "x_raw": float(np.nanmean(x_vals)),
            "y_raw": float(y_min),
        })

    return pd.DataFrame(rows)


# ============================================================
# Curve calculator
# ============================================================
def compute_group_curve_with_bootstrap(
    prep_list, window_days, leads_list, subset="all", min_n=5, 
    debug_bridge=False, n_boot=2000, ci=95.0, base_seed=42, seed_offset=0
):
    r_list, lo_list, hi_list = [], [], []
    n_list, n_boot_pool_list, n_boot_valid_list = [], [], []

    subset_flag = 0 if subset.lower() == "all" else 1

    # 根据传入的 leads_list 进行计算
    leads_this = [ld for ld in leads_list if ld >= window_days]

    for lead in leads_this:
        dfs = []
        for prep in prep_list:
            df = build_samples_event_anchored(
                prep, lead_days=lead, window_days=window_days,
                subset=subset, debug_bridge=debug_bridge
            )
            dfs.append(df)

        df_all = pd.concat(dfs, ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
        n_now = len(df_all)
        n_list.append(n_now)

        if n_now < min_n:
            r_list.append(np.nan); lo_list.append(np.nan); hi_list.append(np.nan)
            n_boot_pool_list.append(0); n_boot_valid_list.append(0)
            continue

        x = df_all["x_raw"].to_numpy(dtype=float)
        y = df_all["y_raw"].to_numpy(dtype=float)

        if np.nanstd(x) == 0 or np.nanstd(y) == 0:
            r_list.append(np.nan); lo_list.append(np.nan); hi_list.append(np.nan)
            n_boot_pool_list.append(0); n_boot_valid_list.append(0)
            continue

        this_seed = int(base_seed + seed_offset * 100000 + window_days * 1000 + lead * 10 + subset_flag)
        boot = bootstrap_pearson_r_ci(
            df_all, n_boot=n_boot, ci=ci, seed=this_seed,
            exclude_bridge_year_from_bootstrap=True, bridge_year=BRIDGE_YEAR,
        )

        r_list.append(boot["r_obs"])
        lo_list.append(boot["ci_low"])
        hi_list.append(boot["ci_high"])
        n_boot_pool_list.append(boot["n_boot_pool"])
        n_boot_valid_list.append(boot["n_boot_valid"])

    out = {
        "r": r_list, "ci_low": lo_list, "ci_high": hi_list,
        "n": n_list, "n_boot_pool": n_boot_pool_list, "n_boot_valid": n_boot_valid_list,
    }

    return leads_this, out


# ============================================================
# Debug CSV saver
# ============================================================
def save_debug_curve_csv(leads, curves_dict, window_days, subset, out_dir):
    if not SAVE_DEBUG_CSV: return

    df = pd.DataFrame({"lead_days": leads})
    for key in ["coupled", "nocoupl", "merra2"]:
        for metric in ["r", "ci_low", "ci_high", "n", "n_boot_pool", "n_boot_valid"]:
            df[f"{key}_{metric}"] = curves_dict[key][metric]

    save_csv = os.path.join(out_dir, f"EventAnchored_COMPARE_{window_days}d_{subset.upper()}_B2000_NOCOUPL_MERRA2_BOOTSTRAP_DEBUG.csv")
    df.to_csv(save_csv, index=False)
    print("Saved debug csv:", save_csv)


# ============================================================
# Plotter
# ============================================================
def plot_event_anchored_three_line(leads, curves_dict, window_days, subset="all"):
    subset_upper = subset.upper()
    out_dir = os.path.join(PLOT_BASE, "COMBINED_COMPARE_WITH_MERRA2", "EP_O3_EventAnchored")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12.2, 5.8), dpi=150)
    ax = plt.gca()

    ax.axhline(0, color="k", lw=0.8, alpha=0.5)
    ax.grid(True, linestyle=":", alpha=0.35)

    x = np.asarray(leads, dtype=float)

    color_map = {"coupled": "#1f77b4", "nocoupl": "#d62728", "merra2": "black"}
    order = [("coupled", "B2000WCN"), ("nocoupl", "B2000WCN NOCOUPL"), ("merra2", "MERRA2")]

    for key, label in order:
        y  = np.asarray(curves_dict[key]["r"], dtype=float)
        lo = np.asarray(curves_dict[key]["ci_low"], dtype=float)
        hi = np.asarray(curves_dict[key]["ci_high"], dtype=float)

        line_color = color_map[key]

        # 主线
        ax.plot(x, y, lw=2.4, linestyle="-", color=line_color, label=label, zorder=4)

        # 包络区
        mask_fill = np.isfinite(y) & np.isfinite(lo) & np.isfinite(hi)
        if np.any(mask_fill):
            if key == "merra2":
                poly = ax.fill_between(x[mask_fill], lo[mask_fill], hi[mask_fill],
                                       facecolor="lightgray", edgecolor="gray",
                                       alpha=0.35, linewidth=0.6, zorder=1)
                poly.set_hatch("///")
            else:
                ax.fill_between(x[mask_fill], lo[mask_fill], hi[mask_fill],
                                color=line_color, alpha=RIBBON_ALPHA, linewidth=0, zorder=1)

        # 稀疏 error bars
        idx_all = np.arange(len(x))
        mask_err = np.isfinite(y) & np.isfinite(lo) & np.isfinite(hi) & ((idx_all % ERRORBAR_EVERY) == 0)
        if np.any(mask_err):
            yerr = np.vstack([y[mask_err] - lo[mask_err], hi[mask_err] - y[mask_err]])
            ecolor = "black" if key == "merra2" else line_color
            ax.errorbar(x[mask_err], y[mask_err], yerr=yerr, fmt="none",
                        ecolor=ecolor, elinewidth=ERRORBAR_ELW, capsize=ERRORBAR_CAPSIZE,
                        alpha=0.9, zorder=5)

    ax.set_ylim(-1, 1)
    ax.set_xlim(max(leads), min(leads))   # 确保X轴自大到小排列
    ax.set_ylabel("Pearson r")
    ax.set_xlabel("Lead days before each year's MA O$_3$-min anchor day")
    ax.set_title(
        f"Event-anchored comparison | {window_days}-day Fz window vs MA minimum O$_3$ ({subset_upper})",
        loc="left", fontweight="bold"
    )

    txt = f"Bootstrap CI = {BOOTSTRAP_CI:.0f}%\nNboot = {BOOTSTRAP_N}\n"
    ax.text(0.985, 0.04, txt, transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.86, edgecolor="0.7"))

    ax.legend(frameon=False, ncol=3)
    plt.tight_layout()

    save_path = os.path.join(out_dir, f"EventAnchored_COMPARE_{window_days}d_{subset_upper}_B2000_NOCOUPL_MERRA2_BOOTCI.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)
    save_debug_curve_csv(leads, curves_dict, window_days, subset, out_dir)


# ============================================================
# 预处理实验
# ============================================================
prep_bwcn = prepare_experiment("BWCN", FILE_BWCN_EP, os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
                               low25_txt=os.path.join(DATA_BASE, "BWCN", "low25pct_years_30_70hPa.txt"), lat_band=LAT_BAND, apply_o3_5d=APPLY_O3_5D)
prep_b2000 = prepare_experiment("B2000WCN", FILE_B2000_EP, os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
                                low25_txt=os.path.join(DATA_BASE, "B2000WCN", "low25pct_years_30_70hPa.txt"), lat_band=LAT_BAND, apply_o3_5d=APPLY_O3_5D)
prep_noc = prepare_experiment("B2000WCN_NOCOUPL", FILE_NOCOUPL_EP, os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
                              low25_txt=os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"), lat_band=LAT_BAND, apply_o3_5d=APPLY_O3_5D)
prep_merra = prepare_experiment("MERRA2", FILE_MERRA_EP, os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
                                low25_txt=os.path.join(DATA_BASE, "MERRA2", "low25pct_years_30_70hPa.txt"), lat_band=LAT_BAND, apply_o3_5d=APPLY_O3_5D)

# ============================================================
# 执行：4 张图
# ============================================================
coupled_group = [prep_bwcn, prep_b2000]
nocoupl_group = [prep_noc]
merra_group   = [prep_merra]

for wd in WINDOW_DAYS_LIST:
    
    # 核心修改：针对不同 window_days 动态生成对齐好的 lead_days 序列
    # range() 的 end 是 exclusive 的，因此减 1
    current_leads = list(range(TARGET_END_LEAD_MAX + wd, TARGET_END_LEAD_MIN + wd - 1, -1))
    
    for subset in ["all", "low25"]:
        leads, out_coupled = compute_group_curve_with_bootstrap(
            coupled_group, window_days=wd, leads_list=current_leads,
            subset=subset, min_n=MIN_N, debug_bridge=DEBUG_BRIDGE,
            n_boot=BOOTSTRAP_N, ci=BOOTSTRAP_CI, base_seed=BOOTSTRAP_SEED, seed_offset=1
        )

        _, out_nocoupl = compute_group_curve_with_bootstrap(
            nocoupl_group, window_days=wd, leads_list=current_leads,
            subset=subset, min_n=MIN_N, debug_bridge=DEBUG_BRIDGE,
            n_boot=BOOTSTRAP_N, ci=BOOTSTRAP_CI, base_seed=BOOTSTRAP_SEED, seed_offset=2
        )

        _, out_merra = compute_group_curve_with_bootstrap(
            merra_group, window_days=wd, leads_list=current_leads,
            subset=subset, min_n=MIN_N, debug_bridge=False,
            n_boot=BOOTSTRAP_N, ci=BOOTSTRAP_CI, base_seed=BOOTSTRAP_SEED, seed_offset=3
        )

        curves = {"coupled": out_coupled, "nocoupl": out_nocoupl, "merra2": out_merra}

        plot_event_anchored_three_line(leads, curves_dict=curves, window_days=wd, subset=subset)

In [ ]:
# ============================================================
# Block C3 (MERRA2 Scatter Pro): Fz vs O3 return-to-threshold timing
# 新逻辑：
# 1) 只做 MERRA2
# 2) 先按每年的 O3 平均值分组：
#    - High 25% years
#    - Low 25% years
# 3) 分别定义返回时间：
#    - High 25% years: 返回 climatology + sigma 并保持稳定
#    - Low 25% years:  返回 climatology - sigma 并保持稳定
# 4) 继续与 100 hPa -Fz 做 standardized scatter
#
# 输出：
# - 一张图：High25, Fz vs return-to-(clim + sigma)
# - 一张图：Low25,  Fz vs return-to-(clim - sigma)
# ============================================================

import os
from datetime import date
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- Paths ----------------
FILE_MERRA_EP = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/EPflux_daily/MERRA2_EPFLUX_Daily_Series_1980_2026_noW.nc"
DATA_BASE     = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE     = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

EXP_NAME = "MERRA2"

# ---------------- Config ----------------
SHORT_MAP = {
    "DJF": [12, 1, 2],
    "JF":  [1, 2],
    "FM":  [2, 3],
    "MA":  [3, 4],
    "NDJ": [11, 12, 1],
    "JFM": [1, 2, 3],
    "OND": [10, 11, 12]
}

LAT_BAND = (40, 80)
APPLY_O3_5D = True

# ---------- 分组指标 ----------
# 用哪个季节/统计定义“高 25% / 低 25% 年份”
GROUP_SEASON = "MA"
GROUP_METHOD = "mean"   # 这里你说的是 “O3平均最高/最低 25% 年份”

# ---------- 返回时间指标 ----------
REF_MONTH_DAY = (4, 1)      # 相对 4月1日计时
SEARCH_START  = (3, 1)
SEARCH_END    = (6, 30)
SIGMA_MULT    = 1.0
PERSIST_DAYS  = 7
INSIDE_FRAC   = 1.0
DROP_FEB29    = True


# ============================================================
# Helpers
# ============================================================
def sel_latband(da, lat0=40.0, lat1=80.0, lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})

def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)

def is_cross_year_season(season_str):
    months = SHORT_MAP.get(season_str, [12, 1, 2])
    return any(m in [11, 12] for m in months) and any(m in [1, 2] for m in months)

def select_100hpa_level(da_plev):
    pvals = np.asarray(da_plev["plev"].values, dtype=float)
    if np.nanmax(np.abs(pvals)) > 2000:
        target = 10000.0
        unit_guess = "Pa"
    else:
        target = 100.0
        unit_guess = "hPa"

    out = da_plev.sel(plev=target, method="nearest")
    print(f"  Selected 100 hPa level using target={target:g} ({unit_guess} interpretation).")
    return out

def standardize_da(da):
    mu = float(da.mean().values)
    sd = float(da.std().values)
    if (not np.isfinite(sd)) or (sd == 0):
        raise ValueError("Standard deviation is zero or NaN; cannot standardize.")
    return (da - mu) / sd

def dynamic_axis_limits(x, y, pad=0.4):
    vals = np.concatenate([np.asarray(x), np.asarray(y)])
    vmax = np.nanmax(np.abs(vals))
    vmax = max(vmax + pad, 2.0)
    return (-vmax, vmax)

def da_to_daily_df(da_1d):
    return pd.DataFrame({
        "year":  da_1d.time.dt.year.values.astype(int),
        "month": da_1d.time.dt.month.values.astype(int),
        "day":   da_1d.time.dt.day.values.astype(int),
        "value": da_1d.values.astype(float),
    }).sort_values(["year", "month", "day"]).reset_index(drop=True)

def get_seasonal_data_from_daily_da(da, season_str, method="mean"):
    """
    用 year/month/day 直接聚合，不依赖 to_series()+字符串切片
    """
    months = SHORT_MAP.get(season_str, [12, 1, 2])
    cross_year = is_cross_year_season(season_str)

    df = da_to_daily_df(da)
    all_years = np.unique(df["year"].values)

    results = {}

    for yr in all_years:
        if cross_year:
            early_months = [m for m in months if m < 6]
            late_months  = [m for m in months if m >= 6]

            sub = df[
                ((df["year"] == int(yr) - 1) & (df["month"].isin(late_months))) |
                ((df["year"] == int(yr))     & (df["month"].isin(early_months)))
            ].copy()
        else:
            sub = df[
                (df["year"] == int(yr)) &
                (df["month"].isin(months))
            ].copy()

        sub = sub[np.isfinite(sub["value"].values)]

        if len(sub) < len(months) * 28:
            continue

        if method == "mean":
            val = float(sub["value"].mean())
        elif method == "min":
            val = float(sub["value"].min())
        elif method == "max":
            val = float(sub["value"].max())
        else:
            raise ValueError(f"Unsupported method: {method}")

        results[int(yr)] = val

    if len(results) == 0:
        return xr.DataArray(
            np.array([], dtype=float),
            coords={"year": np.array([], dtype=int)},
            dims=("year",),
            name=f"{season_str}_{method}"
        )

    years = np.array(sorted(results.keys()), dtype=int)
    vals  = np.array([results[y] for y in years], dtype=float)

    return xr.DataArray(
        vals,
        coords={"year": years},
        dims=("year",),
        name=f"{season_str}_{method}"
    )

def get_year_groups_from_o3(o3_da, group_season="MA", group_method="mean", apply_o3_5d=True):
    """
    根据 O3 的季节统计，定义 high25 / low25 年份
    你当前要求是：O3 平均最高/最低的 25% 年份
    """
    da = o3_da.copy().sortby("time")
    if DROP_FEB29:
        da = drop_feb29(da)

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    group_metric = get_seasonal_data_from_daily_da(da, group_season, method=group_method)

    vals = group_metric.values
    yrs  = group_metric.year.values.astype(int)

    idx_sort = np.argsort(vals)   # 低 -> 高
    nsel = max(1, int(0.25 * len(vals)))

    low25_years  = yrs[idx_sort[:nsel]]
    high25_years = yrs[idx_sort[-nsel:]]

    return (
        set(low25_years.tolist()),
        set(high25_years.tolist()),
        group_metric
    )

def compute_return_timing_series(
    o3_da,
    target_mode="minus_sigma",
    restrict_years=None,
    ref_month_day=(4, 1),
    search_start=(3, 1),
    search_end=(6, 30),
    sigma_mult=1.0,
    persist_days=7,
    inside_frac=1.0,
    apply_o3_5d=True,
    drop_feb29_flag=True,
):
    """
    计算每一年的“返回某个阈值并保持稳定”的日期，相对 4/1 的天数

    target_mode:
      - "plus_sigma"  : 返回 climatology + sigma
      - "minus_sigma" : 返回 climatology - sigma

    这里不再是“落入 ±sigma band”，而是：
      - plus_sigma: value >= clim + sigma，并持续稳定
      - minus_sigma: value >= clim - sigma，并持续稳定
        （表示从低臭氧状态恢复到低侧正常阈值之上）

    返回：
      DataArray(dim='year')
    """
    da = o3_da.copy().sortby("time")

    if drop_feb29_flag:
        da = drop_feb29(da)

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    df = da_to_daily_df(da)

    clim_tbl = (
        df.groupby(["month", "day"])["value"]
        .agg(clim="mean", sigma="std")
        .reset_index()
    )
    clim_tbl["sigma"] = clim_tbl["sigma"].fillna(0.0)

    df = df.merge(clim_tbl, on=["month", "day"], how="left")

    if target_mode == "plus_sigma":
        threshold = df["clim"] + sigma_mult * df["sigma"]
        df["meets_threshold"] = np.isfinite(df["value"]) & np.isfinite(threshold) & (df["value"] >= threshold)

    elif target_mode == "minus_sigma":
        threshold = df["clim"] - sigma_mult * df["sigma"]
        df["meets_threshold"] = np.isfinite(df["value"]) & np.isfinite(threshold) & (df["value"] >= threshold)

    else:
        raise ValueError("target_mode must be 'plus_sigma' or 'minus_sigma'.")

    results = {}

    for yr in sorted(df["year"].unique()):
        if (restrict_years is not None) and (int(yr) not in restrict_years):
            continue

        sub = df[df["year"] == int(yr)].copy()
        sub = sub[
            sub.apply(
                lambda r: (int(r["month"]), int(r["day"])) >= search_start
                          and
                          (int(r["month"]), int(r["day"])) <= search_end,
                axis=1
            )
        ].copy()

        if len(sub) < persist_days:
            continue

        sub = sub.sort_values(["month", "day"]).reset_index(drop=True)
        meets = sub["meets_threshold"].values.astype(bool)

        found = False
        for i in range(len(sub) - persist_days + 1):
            win = meets[i:i + persist_days]
            frac = win.mean()

            if meets[i] and (frac >= inside_frac):
                m = int(sub.loc[i, "month"])
                d = int(sub.loc[i, "day"])

                delta_days = (date(int(yr), m, d) - date(int(yr), ref_month_day[0], ref_month_day[1])).days
                results[int(yr)] = delta_days
                found = True
                break

        if not found:
            continue

    if len(results) == 0:
        return xr.DataArray(
            np.array([], dtype=float),
            coords={"year": np.array([], dtype=int)},
            dims=("year",),
            name=f"return_timing_{target_mode}"
        )

    years = np.array(sorted(results.keys()), dtype=int)
    vals  = np.array([results[y] for y in years], dtype=float)

    return xr.DataArray(
        vals,
        coords={"year": years},
        dims=("year",),
        name=f"return_timing_{target_mode}"
    )


# ============================================================
# Plotting
# ============================================================
def plot_custom_relationship_threshold_return(
    exp_name,
    ep_nc,
    o3_nc,
    group_label,
    restrict_years,
    target_mode,
    fz_season="DJF",
    fz_method="mean",
    lat_band=(40, 80),
    apply_o3_5d=True,
    ref_month_day=(4, 1),
    search_start=(3, 1),
    search_end=(6, 30),
    sigma_mult=1.0,
    persist_days=7,
    inside_frac=1.0,
    drop_feb29_flag=True
):
    """
    画：
      x = seasonal -Fz
      y = return-to-threshold timing since Apr 1
    """
    print(f"\n[{exp_name}] {group_label}: Fz({fz_season}, {fz_method}) vs {target_mode} return timing...")

    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        print("  Missing file(s). Skip.")
        return

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    # ---------------- Fz ----------------
    var_fz = "Fz" if "Fz" in ds_ep.data_vars else "EP2_cart"
    fz_raw = ds_ep[var_fz]

    if "plev" in fz_raw.coords:
        fz_raw = select_100hpa_level(fz_raw)

    if "lat" in fz_raw.coords:
        fz_raw = sel_latband(fz_raw, lat_band[0], lat_band[1], "lat")
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    x_source = -1.0 * fz_raw
    if drop_feb29_flag:
        x_source = drop_feb29(x_source)

    fz_stat = get_seasonal_data_from_daily_da(x_source, fz_season, method=fz_method)

    # ---------------- O3 timing metric ----------------
    y_metric = compute_return_timing_series(
        da_o3,
        target_mode=target_mode,
        restrict_years=restrict_years,
        ref_month_day=ref_month_day,
        search_start=search_start,
        search_end=search_end,
        sigma_mult=sigma_mult,
        persist_days=persist_days,
        inside_frac=inside_frac,
        apply_o3_5d=apply_o3_5d,
        drop_feb29_flag=drop_feb29_flag
    )

    # ---------------- Align years ----------------
    common_years = np.intersect1d(fz_stat.year.values, y_metric.year.values)
    common_years = np.array([y for y in common_years if int(y) in restrict_years], dtype=int)

    if len(common_years) < 3:
        print("  Too few common years after filtering. Skip.")
        ds_ep.close()
        return

    x_raw = fz_stat.sel(year=common_years)
    y_raw = y_metric.sel(year=common_years)

    valid = np.isfinite(x_raw.values) & np.isfinite(y_raw.values)
    x_raw = x_raw.sel(year=x_raw.year.values[valid])
    y_raw = y_raw.sel(year=y_raw.year.values[valid])

    if len(x_raw.year) < 3:
        print("  Too few valid years after NaN filtering. Skip.")
        ds_ep.close()
        return

    x = standardize_da(x_raw)
    y = y_raw.copy()  

    # ---------------- Plot ----------------
    plt.figure(figsize=(7.5, 6.3), dpi=140)
    ax = plt.gca()

    ax.grid(True, linestyle=":", alpha=0.5)
    ax.axhline(0, color="k", lw=0.9, alpha=0.5)
    ax.axvline(0, color="k", lw=0.9, alpha=0.5)

    ax.scatter(
        x.values, y.values,
        color="black",
        alpha=0.82,
        s=46,
        edgecolors="none"
    )

    r, p = pearsonr(x.values, y.values)

    ax.text(
        0.05, 0.95,
        f"r = {r:.3f}\np = {p:.2e}\nN = {len(x)}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="0.7")
    )

    ax.set_xlim(-3, 3)
    ax.set_ylim(-40, 45)

    ax.set_xlabel(f"{fz_season} $-F_z$ at 100 hPa ({fz_method}, $\\sigma$)", fontsize=11)

    if target_mode == "plus_sigma":
        ylabel = "Return-to-(climatology + σ) date since 04-01 (days)"
        title2 = "return to climatology + σ"
    else:
        ylabel = "Return-to-(climatology - σ) date since 04-01 (days)"
        title2 = "return to climatology - σ"

    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(
        f"{exp_name} | {group_label}: {fz_season} $-F_z$ vs {title2}",
        loc="left", fontweight="bold", fontsize=12
    )

    subtitle = (
        f"group = {GROUP_SEASON} {GROUP_METHOD} top/bottom 25%, "
        f"stable = {persist_days} days, "
        f"search = {search_start[0]:02d}-{search_start[1]:02d} to {search_end[0]:02d}-{search_end[1]:02d}"
    )
    ax.text(
        0.0, 1.02, subtitle,
        transform=ax.transAxes, ha="left", va="bottom", fontsize=9
    )

    out_dir = os.path.join(PLOT_BASE, exp_name, "EP_O3_ThresholdReturnTiming")
    os.makedirs(out_dir, exist_ok=True)

    save_path = os.path.join(
        out_dir,
        f"Scatter_{group_label}_{fz_season}_{fz_method}_vs_{target_mode}_since0401.png"
    )

    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.show()
    print("  Saved:", save_path)

    ds_ep.close()


# ============================================================
# Run
# ============================================================
o3_for_group = xr.open_dataarray(os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"))

low25_years, high25_years, group_metric = get_year_groups_from_o3(
    o3_for_group,
    group_season=GROUP_SEASON,
    group_method=GROUP_METHOD,
    apply_o3_5d=APPLY_O3_5D
)

o3_for_group.close()

print("Low25 years:", sorted(low25_years))
print("High25 years:", sorted(high25_years))

configs = [
    {
        "fz_season": "DJF",
        "fz_method": "mean",
    },
    {
        "fz_season": "JF",
        "fz_method": "mean",
    }
]

for cfg in configs:
    # 1) 高 O3 年：返回 climatology + sigma
    plot_custom_relationship_threshold_return(
        EXP_NAME,
        FILE_MERRA_EP,
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        group_label="High25",
        restrict_years=high25_years,
        target_mode="plus_sigma",
        lat_band=LAT_BAND,
        apply_o3_5d=APPLY_O3_5D,
        ref_month_day=REF_MONTH_DAY,
        search_start=SEARCH_START,
        search_end=SEARCH_END,
        sigma_mult=SIGMA_MULT,
        persist_days=PERSIST_DAYS,
        inside_frac=INSIDE_FRAC,
        drop_feb29_flag=DROP_FEB29,
        **cfg
    )

    # 2) 低 O3 年：返回 climatology - sigma
    plot_custom_relationship_threshold_return(
        EXP_NAME,
        FILE_MERRA_EP,
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        group_label="Low25",
        restrict_years=low25_years,
        target_mode="minus_sigma",
        lat_band=LAT_BAND,
        apply_o3_5d=APPLY_O3_5D,
        ref_month_day=REF_MONTH_DAY,
        search_start=SEARCH_START,
        search_end=SEARCH_END,
        sigma_mult=SIGMA_MULT,
        persist_days=PERSIST_DAYS,
        inside_frac=INSIDE_FRAC,
        drop_feb29_flag=DROP_FEB29,
        **cfg
    )

    #结论，并非一个好的predictor

In [ ]:
# ============================================================
# Block G (NEW): MA minimum O3 value VS date of MA minimum O3
# ------------------------------------------------------------
# 核心变化：
# 1) 不再使用 EP flux / Fz
# 2) 只使用 daily O3 数据
# 3) 对每个 year，提取 MA (Mar-Apr) 中的 minimum O3
# 4) 横轴 = 该 minimum 出现日期，用 "days since Mar 1" 编码：
#       0  = Mar-01
#       30 = Mar-31
#       31 = Apr-01
#       60 = Apr-30
# 5) 纵轴 = MA minimum O3 value (raw)
# 6) 点颜色按 minimum 出现月份着色：
#       March -> royalblue
#       April -> gold
# 7) 统计信息：
#       ALL / MAR / APR
#    只显示 R 和 N
# 8) 输出：
#       - 一张三面板散点图 PNG
#       - 一个 combined CSV（保存每个点对应的 year / date / value）
# ============================================================

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

# ---------------- 基础路径 ----------------
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

# 新版输出目录
PLOT_BASE_NEW = os.path.join(PLOT_BASE, "SCATTER_MA_O3MIN_VALUE_VS_O3MIN_DATE")

# O3 文件（沿用你现有目录）
FILE_BWCN_O3    = os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc")
FILE_B2000_O3   = os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc")
FILE_NOCOUPL_O3 = os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc")
FILE_MERRA_O3   = os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc")

# ---------------- 设置 ----------------
APPLY_O3_5D = True

# MA = March + April
TARGET_MONTHS = [3, 4]

# MA 总天数通常 61 天，允许极少量缺测
MA_VALID_MIN_DAYS = 58

# 点着色：按 minimum O3 出现月份
MONTH_COLOR = {
    3: "royalblue",   # March
    4: "gold",        # April
}
MONTH_LABEL = {
    3: "March minimum",
    4: "April minimum",
}

POINT_SIZE = 42
POINT_ALPHA = 0.88
POINT_EDGE_LW = 0.25

# 坐标轴：X 建议固定，Y 可手动或自动
USE_MANUAL_XLIM = True
MANUAL_XLIM = (-1, 61)

USE_MANUAL_YLIM = False
MANUAL_YLIM = (60, 170)

FIG_DPI = 220

# ============================================================
# Helpers
# ============================================================
def assert_daily_unique(da, name=""):
    """
    兼容 cftime / datetime64 的 daily 唯一性检查
    """
    da = da.sortby("time")

    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)

    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")

    return da

def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)

def safe_corr(x, y):
    """
    至少 3 个样本、且 x/y 不能是常数，才计算 Pearson r
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    n = len(x)

    if n < 3:
        return np.nan, n

    if np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan, n

    try:
        r, _ = pearsonr(x, y)
        return float(r), n
    except Exception:
        return np.nan, n

def format_rn_text(label, x, y):
    r, n = safe_corr(x, y)
    rtxt = "nan" if not np.isfinite(r) else f"{r:.3f}"
    return f"{label}\nR = {rtxt}\nN = {n}"

def encode_day_from_mar1(month, day):
    """
    MA 专用日期编码：
      Mar-01 -> 0
      Mar-31 -> 30
      Apr-01 -> 31
      Apr-30 -> 60
    """
    if month == 3:
        return day - 1
    elif month == 4:
        return 31 + (day - 1)
    else:
        raise ValueError(f"Month must be 3 or 4 for MA, got {month}")

def mmdd_str(month, day):
    return f"{month:02d}-{day:02d}"

def get_ma_min_o3_dataset(o3_da, apply_o3_5d=True, min_valid_days=58):
    """
    对每个 year，提取 MA (Mar-Apr) 中的 minimum O3
    返回 xr.Dataset:
      - y_raw(year): MA minimum O3
      - min_month(year): minimum 出现月份 (3/4)
      - min_day(year): minimum 出现日
      - day_from_mar1(year): 横轴编码值
    """
    o3_da = assert_daily_unique(o3_da, name="O3").sortby("time")

    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(o3_da.time.dt.year.values.astype(int))

    out_years = []
    out_vals = []
    out_months = []
    out_days = []
    out_xdate = []

    for yr in years:
        mask = (
            (o3_da.time.dt.year == yr) &
            (o3_da.time.dt.month.isin(TARGET_MONTHS))
        )
        seg = o3_da.where(mask, drop=True)

        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid < min_valid_days:
            continue

        vals = np.asarray(seg.values, dtype=float)
        valid = np.isfinite(vals)
        if not np.any(valid):
            continue

        # 若有并列最小值，取最先出现的那一天
        valid_idx = np.where(valid)[0]
        idx_local = valid_idx[np.argmin(vals[valid])]

        min_val = float(vals[idx_local])
        min_month = int(seg.time.dt.month.values[idx_local])
        min_day = int(seg.time.dt.day.values[idx_local])

        if min_month not in [3, 4]:
            continue

        x_date = encode_day_from_mar1(min_month, min_day)

        out_years.append(int(yr))
        out_vals.append(min_val)
        out_months.append(min_month)
        out_days.append(min_day)
        out_xdate.append(x_date)

    return xr.Dataset(
        data_vars={
            "y_raw": ("year", np.array(out_vals, dtype=float)),
            "min_month": ("year", np.array(out_months, dtype=int)),
            "min_day": ("year", np.array(out_days, dtype=int)),
            "day_from_mar1": ("year", np.array(out_xdate, dtype=int)),
        },
        coords={"year": np.array(out_years, dtype=int)}
    )

def load_one_o3_source(o3_nc, apply_o3_5d=True, drop_leap_day=False):
    """
    加载一个实验源，只用 O3
    返回：
    {
      "o3_da": da_o3,
      "ma_min_ds": xr.Dataset(year -> y_raw / min_month / min_day / day_from_mar1)
    }
    """
    if not os.path.exists(o3_nc):
        raise FileNotFoundError(f"Missing O3 file: {o3_nc}")

    da_o3 = xr.open_dataarray(o3_nc)

    if drop_leap_day:
        da_o3 = drop_feb29(da_o3)

    da_o3 = assert_daily_unique(da_o3, name=os.path.basename(o3_nc))

    ma_min_ds = get_ma_min_o3_dataset(
        da_o3,
        apply_o3_5d=apply_o3_5d,
        min_valid_days=MA_VALID_MIN_DAYS
    )

    return {
        "o3_da": da_o3,
        "ma_min_ds": ma_min_ds,
    }

def make_df_for_source(source_dict, part_name):
    """
    从单个 source 构造散点表
    columns:
      part, year, x_date, y_raw, min_month, min_day, min_date_str
    """
    ds = source_dict["ma_min_ds"]
    if ds is None or ds.year.size == 0:
        return None

    year = ds.year.values.astype(int)
    x_date = ds["day_from_mar1"].values.astype(float)
    y_raw = ds["y_raw"].values.astype(float)
    min_month = ds["min_month"].values.astype(int)
    min_day = ds["min_day"].values.astype(int)

    valid = (
        np.isfinite(x_date) &
        np.isfinite(y_raw) &
        np.isin(min_month, [3, 4])
    )
    if not np.any(valid):
        return None

    min_date_str = np.array(
        [mmdd_str(m, d) for m, d in zip(min_month[valid], min_day[valid])],
        dtype=object
    )

    return {
        "part": np.array([part_name] * valid.sum(), dtype=object),
        "year": year[valid],
        "x": x_date[valid],
        "y": y_raw[valid],
        "min_month": min_month[valid],
        "min_day": min_day[valid],
        "min_date_str": min_date_str,
    }

def concat_dict_rows(dict_list):
    dict_list = [d for d in dict_list if d is not None]
    if len(dict_list) == 0:
        return None

    out = {}
    keys = dict_list[0].keys()
    for k in keys:
        out[k] = np.concatenate([d[k] for d in dict_list])
    return out

def build_group_scatter_df(group_info):
    """
    为某一组（coupled / nocoupl / merra2）构造散点数据
    """
    rows = []
    for part in group_info["parts"]:
        df_part = make_df_for_source(
            source_dict=part["data"],
            part_name=part["name"]
        )
        rows.append(df_part)

    df_all = concat_dict_rows(rows)
    return df_all

def compute_shared_ylim(df_list, manual_ylim=None):
    """
    共享 Y 轴范围
    """
    if manual_ylim is not None:
        return manual_ylim

    y_all = []
    for df in df_list:
        if df is not None and len(df["year"]) > 0:
            y_all.append(np.asarray(df["y"], dtype=float))

    if len(y_all) == 0:
        return (0, 1)

    y_all = np.concatenate(y_all)
    y_all = y_all[np.isfinite(y_all)]

    if y_all.size == 0:
        return (0, 1)

    y0 = float(np.nanmin(y_all))
    y1 = float(np.nanmax(y_all))

    if y0 == y1:
        pad = 1.0
    else:
        pad = 0.06 * (y1 - y0)

    return (y0 - pad, y1 + pad)

def save_axis_limits_txt(out_dir, xlim, ylim):
    os.makedirs(out_dir, exist_ok=True)
    txt_path = os.path.join(out_dir, "MA_O3minValue_vs_O3minDate_axis_limits.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"xlim = ({xlim[0]:.10g}, {xlim[1]:.10g})\n")
        f.write(f"ylim = ({ylim[0]:.10g}, {ylim[1]:.10g})\n")
        f.write("x axis = day since Mar 1\n")
        f.write("0=Mar-01, 30=Mar-31, 31=Apr-01, 60=Apr-30\n")
        f.write("color map: March=royalblue, April=gold\n")
    print("Saved axis-limit record:", txt_path)

def save_combined_csv(df_map, out_dir):
    """
    保存 combined CSV，方便检查每个点对应的 year / date / value
    """
    rows = []
    for group_name, df in df_map.items():
        if df is None:
            continue
        n = len(df["year"])
        for i in range(n):
            rows.append({
                "group": group_name,
                "part": df["part"][i],
                "year": int(df["year"][i]),
                "day_from_mar1": int(df["x"][i]),
                "ma_min_o3": float(df["y"][i]),
                "min_month": int(df["min_month"][i]),
                "min_day": int(df["min_day"][i]),
                "min_date_str": str(df["min_date_str"][i]),
            })

    if len(rows) == 0:
        print("No valid rows, skip CSV.")
        return None

    out_csv = os.path.join(out_dir, "MA_O3minValue_vs_O3minDate_points.csv")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print("Saved CSV:", out_csv)
    return out_csv

def build_sources():
    """
    三组：
    1) coupled  = BWCN + B2000WCN
    2) nocoupl  = B2000WCN_NOCOUPL
    3) merra2   = MERRA2
    """
    sources = {
        "coupled": {
            "label": "Coupled (BWCN+B2000WCN)",
            "parts": [
                {
                    "name": "BWCN",
                    "data": load_one_o3_source(
                        FILE_BWCN_O3,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
                {
                    "name": "B2000WCN",
                    "data": load_one_o3_source(
                        FILE_B2000_O3,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
            ]
        },
        "nocoupl": {
            "label": "B2000WCN NOCOUPL",
            "parts": [
                {
                    "name": "B2000WCN_NOCOUPL",
                    "data": load_one_o3_source(
                        FILE_NOCOUPL_O3,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        },
        "merra2": {
            "label": "MERRA2",
            "parts": [
                {
                    "name": "MERRA2",
                    "data": load_one_o3_source(
                        FILE_MERRA_O3,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        }
    }
    return sources

def close_sources(sources):
    for group in sources.values():
        for part in group["parts"]:
            da = part["data"].get("o3_da", None)
            if da is not None and hasattr(da, "close"):
                try:
                    da.close()
                except Exception:
                    pass

def draw_one_panel(ax, df, panel_title, xlim, ylim):
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.axvline(0, color="k", lw=0.8, alpha=0.35)   # Mar-01
    ax.axvline(31, color="k", lw=0.8, alpha=0.35)  # Apr-01

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # 自定义日期刻度
    xticks = [0, 14, 31, 45, 60]
    xtlabs = ["Mar-01", "Mar-15", "Apr-01", "Apr-15", "Apr-30"]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtlabs)

    if df is None or len(df["year"]) == 0:
        ax.set_title(panel_title, fontweight="bold")
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center",
                transform=ax.transAxes, fontsize=11)
        ax.set_xlabel("Date of MA minimum O$_3$ (days since Mar 1)")
        ax.set_ylabel("MA minimum O$_3$ (5-day mean, raw)")
        return

    # 按 minimum month 上色
    for month in [3, 4]:
        mask = df["min_month"] == month
        if np.any(mask):
            ax.scatter(
                df["x"][mask], df["y"][mask],
                s=POINT_SIZE,
                color=MONTH_COLOR[month],
                alpha=POINT_ALPHA,
                edgecolors="k",
                linewidths=POINT_EDGE_LW,
                zorder=3
            )

    ax.set_title(panel_title, fontweight="bold")
    ax.set_xlabel("Date of MA minimum O$_3$ (days since Mar 1)")
    ax.set_ylabel("MA minimum O$_3$ (5-day mean, raw)")

def plot_static_three_panel(sources):
    """
    三面板静态散点图
    布局：
      第一行中间：MERRA2
      第二行左：Coupled
      第二行右：NOCOUPL
    """
    os.makedirs(PLOT_BASE_NEW, exist_ok=True)

    df_merra   = build_group_scatter_df(sources["merra2"])
    df_coupled = build_group_scatter_df(sources["coupled"])
    df_noc     = build_group_scatter_df(sources["nocoupl"])

    # 保存 CSV
    save_combined_csv(
        {
            "MERRA2": df_merra,
            "Coupled_BWCN_B2000WCN": df_coupled,
            "B2000WCN_NOCOUPL": df_noc,
        },
        PLOT_BASE_NEW
    )

    # 共享坐标轴
    xlim = MANUAL_XLIM if USE_MANUAL_XLIM else (-1, 61)
    ylim = MANUAL_YLIM if USE_MANUAL_YLIM else compute_shared_ylim(
        [df_merra, df_coupled, df_noc],
        manual_ylim=None
    )

    save_axis_limits_txt(PLOT_BASE_NEW, xlim, ylim)

    fig = plt.figure(figsize=(12.8, 9.2), dpi=160)
    gs = fig.add_gridspec(
        2, 3,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.0, 1.0, 1.0],
        hspace=0.38, wspace=0.32
    )

    ax_top = fig.add_subplot(gs[0, 1])
    ax_bl  = fig.add_subplot(gs[1, 0])
    ax_br  = fig.add_subplot(gs[1, 2])

    draw_one_panel(ax_top, df_merra,   "MERRA2",                  xlim, ylim)
    draw_one_panel(ax_bl,  df_coupled, "Coupled (BWCN+B2000WCN)", xlim, ylim)
    draw_one_panel(ax_br,  df_noc,     "B2000WCN NOCOUPL",        xlim, ylim)

    fig.suptitle(
        "MA minimum O$_3$ value VS date of MA minimum O$_3$",
        fontsize=15.5, fontweight="bold", y=0.972
    )

    legend_handles = [
        Line2D([0], [0], marker='o', color='w', label='March minimum',
               markerfacecolor=MONTH_COLOR[3], markeredgecolor='k', markersize=8),
        Line2D([0], [0], marker='o', color='w', label='April minimum',
               markerfacecolor=MONTH_COLOR[4], markeredgecolor='k', markersize=8),
    ]
    fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 0.93),
               ncol=2, frameon=False, fontsize=10.5)

    out_png = os.path.join(PLOT_BASE_NEW, "Scatter_MA_O3minValue_vs_O3minDate_MERRA2_B2000_NOCOUPL.png")
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.show(fig)

    print("Saved PNG:", out_png)
    return out_png

# ============================================================
# Run
# ============================================================
if __name__ == "__main__":
    os.makedirs(PLOT_BASE_NEW, exist_ok=True)

    sources = build_sources()

    try:
        plot_static_three_panel(sources)
    finally:
        close_sources(sources)

    print("\n[MA minimum O3 value VS date of MA minimum O3] All done.")

In [ ]:
# ============================================================
# Block G (UPDATED): MA minimum O3 value VS date of MA minimum O3
# ------------------------------------------------------------
# 新增功能：
# 1) 添加极端年份标记（默认标记最低 O3 的前 5 年）
# 2) 添加 MA climatology 曲线：
#       x = day since Mar 1
#       y = climatological 5-day mean O3 during Mar-Apr
# 3) 保留 ALL / MAR / APR 统计信息（只显示 R 和 N）
#
# 重要修复：
#   coupled 的 climatology curve 不能把 BWCN + B2000WCN 的原始 daily
#   时间序列直接 concat(time)，否则会出现 duplicated timestamps。
#   正确做法：先分别算每个成员的 climatology curve，再在 curve 层面求平均。
# ============================================================

import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

# ---------------- 基础路径 ----------------
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

PLOT_BASE_NEW = os.path.join(PLOT_BASE, "SCATTER_MA_O3MIN_VALUE_VS_O3MIN_DATE")
os.makedirs(PLOT_BASE_NEW, exist_ok=True)

FILE_BWCN_O3    = os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc")
FILE_B2000_O3   = os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc")
FILE_NOCOUPL_O3 = os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc")
FILE_MERRA_O3   = os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc")

# ---------------- 设置 ----------------
APPLY_O3_5D = True
TARGET_MONTHS = [3, 4]
MA_VALID_MIN_DAYS = 58

MONTH_COLOR = {
    3: "royalblue",
    4: "gold",
}

POINT_SIZE = 42
POINT_ALPHA = 0.88
POINT_EDGE_LW = 0.25

# 极端年设置
MARK_EXTREMES = True
EXTREME_TOPN = 5
EXTREME_SIZE = 120
EXTREME_LINEWIDTH = 1.5
EXTREME_TEXT_FS = 8.5
EXTREME_COLORS = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

# 气候态曲线设置
CLIM_LINE_COLOR = "black"
CLIM_LINESTYLE = "--"
CLIM_LINEWIDTH = 1.8

USE_MANUAL_XLIM = True
MANUAL_XLIM = (-1, 61)

USE_MANUAL_YLIM = False
MANUAL_YLIM = (60, 170)

FIG_DPI = 220

DEBUG_PRINT = True

# ============================================================
# Helpers
# ============================================================
def debug(msg):
    if DEBUG_PRINT:
        print(msg)

def assert_daily_unique(da, name=""):
    da = da.sortby("time")
    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)
    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")
    return da

def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)

def safe_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    n = len(x)

    if n < 3:
        return np.nan, n
    if np.nanstd(x) == 0 or np.nanstd(y) == 0:
        return np.nan, n

    try:
        r, _ = pearsonr(x, y)
        return float(r), n
    except Exception:
        return np.nan, n

def format_rn_text(label, x, y):
    r, n = safe_corr(x, y)
    rtxt = "nan" if not np.isfinite(r) else f"{r:.3f}"
    return f"{label}\nR = {rtxt}\nN = {n}"

def encode_day_from_mar1(month, day):
    if month == 3:
        return day - 1
    elif month == 4:
        return 31 + (day - 1)
    else:
        raise ValueError(f"Month must be 3 or 4 for MA, got {month}")

def mmdd_str(month, day):
    return f"{month:02d}-{day:02d}"

def get_ma_min_o3_dataset(o3_da, apply_o3_5d=True, min_valid_days=58):
    """
    对每个 year，提取 MA (Mar-Apr) 中的 minimum O3
    返回 xr.Dataset:
      - y_raw(year)
      - min_month(year)
      - min_day(year)
      - day_from_mar1(year)
    """
    o3_da = assert_daily_unique(o3_da, name="O3").sortby("time")

    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(o3_da.time.dt.year.values.astype(int))

    out_years = []
    out_vals = []
    out_months = []
    out_days = []
    out_xdate = []

    for yr in years:
        mask = (
            (o3_da.time.dt.year == yr) &
            (o3_da.time.dt.month.isin(TARGET_MONTHS))
        )
        seg = o3_da.where(mask, drop=True)

        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid < min_valid_days:
            continue

        vals = np.asarray(seg.values, dtype=float)
        valid = np.isfinite(vals)
        if not np.any(valid):
            continue

        valid_idx = np.where(valid)[0]
        idx_local = valid_idx[np.argmin(vals[valid])]

        min_val = float(vals[idx_local])
        min_month = int(seg.time.dt.month.values[idx_local])
        min_day = int(seg.time.dt.day.values[idx_local])

        if min_month not in [3, 4]:
            continue

        x_date = encode_day_from_mar1(min_month, min_day)

        out_years.append(int(yr))
        out_vals.append(min_val)
        out_months.append(min_month)
        out_days.append(min_day)
        out_xdate.append(x_date)

    ds = xr.Dataset(
        data_vars={
            "y_raw": ("year", np.array(out_vals, dtype=float)),
            "min_month": ("year", np.array(out_months, dtype=int)),
            "min_day": ("year", np.array(out_days, dtype=int)),
            "day_from_mar1": ("year", np.array(out_xdate, dtype=int)),
        },
        coords={"year": np.array(out_years, dtype=int)}
    )

    return ds

def get_ma_climatology_curve(o3_da, apply_o3_5d=True, name=""):
    """
    构造 MA climatology 曲线：
      x = day since Mar 1
      y = climatological O3 during Mar-Apr

    额外返回：
      n_years = 实际参与 climatology 的有效年份数
    """
    o3_da = assert_daily_unique(o3_da, name=f"O3_clim_{name}").sortby("time")
    o3_da = drop_feb29(o3_da)

    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    # 只取 3-4 月
    mask = o3_da.time.dt.month.isin(TARGET_MONTHS)
    seg = o3_da.where(mask, drop=True)

    # 统计有效年份数：至少有 MA_VALID_MIN_DAYS 个有效日
    years = np.unique(seg.time.dt.year.values.astype(int))
    valid_years = []
    for yr in years:
        seg_y = seg.where(seg.time.dt.year == yr, drop=True)
        n_valid = int(seg_y.count().values) if seg_y.size > 0 else 0
        if n_valid >= MA_VALID_MIN_DAYS:
            valid_years.append(int(yr))

    if len(valid_years) == 0:
        debug(f"[DEBUG] climatology curve {name}: no valid years")
        return {"x": np.array([], dtype=float), "y": np.array([], dtype=float), "n_years": 0}

    # 只保留有效年份
    seg = seg.where(seg.time.dt.year.isin(valid_years), drop=True)

    month = seg.time.dt.month.values.astype(int)
    day = seg.time.dt.day.values.astype(int)
    x_date = np.array([encode_day_from_mar1(m, d) for m, d in zip(month, day)], dtype=int)

    seg = seg.assign_coords(day_from_mar1=("time", x_date))
    clim = seg.groupby("day_from_mar1").mean("time")

    x = clim["day_from_mar1"].values.astype(float)
    y = clim.values.astype(float)

    debug(f"[DEBUG] climatology curve {name}: {len(x)} points, n_years={len(valid_years)}")

    return {
        "x": x,
        "y": y,
        "n_years": len(valid_years),
    }

def merge_climatology_curves(curve_list, group_name=""):
    """
    对多个 climatology curve 按有效年份数加权平均：
        y_merge = sum(n_i * y_i) / sum(n_i)

    注意：
    这里合并的是“curve”，不是原始 daily time series。
    """
    curve_list = [c for c in curve_list if c is not None and len(c["x"]) > 0 and c.get("n_years", 0) > 0]
    if len(curve_list) == 0:
        return None

    x_union = np.unique(np.concatenate([c["x"] for c in curve_list])).astype(float)

    num = np.zeros(len(x_union), dtype=float)
    den = np.zeros(len(x_union), dtype=float)

    for c in curve_list:
        s = pd.Series(c["y"], index=c["x"]).reindex(x_union)
        w = float(c["n_years"])

        valid = np.isfinite(s.values)
        num[valid] += w * s.values[valid]
        den[valid] += w

    y_merge = np.full(len(x_union), np.nan, dtype=float)
    valid = den > 0
    y_merge[valid] = num[valid] / den[valid]

    debug(
        f"[DEBUG] merged climatology curve {group_name}: "
        f"{len(x_union)} points from {len(curve_list)} members, "
        f"weights={[c['n_years'] for c in curve_list]}"
    )

    return {
        "x": x_union,
        "y": y_merge,
        "n_years": int(np.sum([c["n_years"] for c in curve_list])),
    }
def load_one_o3_source(o3_nc, apply_o3_5d=True, drop_leap_day=False):
    if not os.path.exists(o3_nc):
        raise FileNotFoundError(f"Missing O3 file: {o3_nc}")

    da_o3 = xr.open_dataarray(o3_nc)

    if drop_leap_day:
        da_o3 = drop_feb29(da_o3)

    da_o3 = assert_daily_unique(da_o3, name=os.path.basename(o3_nc))

    ma_min_ds = get_ma_min_o3_dataset(
        da_o3,
        apply_o3_5d=apply_o3_5d,
        min_valid_days=MA_VALID_MIN_DAYS
    )

    clim_curve = get_ma_climatology_curve(
        da_o3,
        apply_o3_5d=apply_o3_5d,
        name=os.path.basename(o3_nc)
    )

    return {
        "o3_da": da_o3,
        "ma_min_ds": ma_min_ds,
        "ma_clim_curve": clim_curve,
    }

def make_df_for_source(source_dict, part_name):
    ds = source_dict["ma_min_ds"]
    if ds is None or ds.year.size == 0:
        return None

    year = ds.year.values.astype(int)
    x_date = ds["day_from_mar1"].values.astype(float)
    y_raw = ds["y_raw"].values.astype(float)
    min_month = ds["min_month"].values.astype(int)
    min_day = ds["min_day"].values.astype(int)

    valid = (
        np.isfinite(x_date) &
        np.isfinite(y_raw) &
        np.isin(min_month, [3, 4])
    )
    if not np.any(valid):
        return None

    min_date_str = np.array(
        [mmdd_str(m, d) for m, d in zip(min_month[valid], min_day[valid])],
        dtype=object
    )

    return {
        "part": np.array([part_name] * valid.sum(), dtype=object),
        "year": year[valid],
        "x": x_date[valid],
        "y": y_raw[valid],
        "min_month": min_month[valid],
        "min_day": min_day[valid],
        "min_date_str": min_date_str,
    }

def concat_dict_rows(dict_list):
    dict_list = [d for d in dict_list if d is not None]
    if len(dict_list) == 0:
        return None

    out = {}
    keys = dict_list[0].keys()
    for k in keys:
        out[k] = np.concatenate([d[k] for d in dict_list])
    return out

def build_group_scatter_df(group_info):
    rows = []
    for part in group_info["parts"]:
        df_part = make_df_for_source(
            source_dict=part["data"],
            part_name=part["name"]
        )
        rows.append(df_part)

    df_all = concat_dict_rows(rows)

    if df_all is not None:
        debug(f"[DEBUG] scatter df {group_info['label']}: N={len(df_all['year'])}")
    else:
        debug(f"[DEBUG] scatter df {group_info['label']}: None")

    return df_all

def build_group_clim_curve(group_info):
    """
    修复版：
    - 单成员组：直接返回该成员 climatology curve
    - 多成员组（如 coupled）：先分别取每个成员的 climatology curve，再求平均
    """
    curves = []
    for part in group_info["parts"]:
        curve = part["data"].get("ma_clim_curve", None)
        curves.append(curve)

    if len(curves) == 0:
        return None

    if len(curves) == 1:
        return curves[0]

    return merge_climatology_curves(curves, group_name=group_info["label"])

def compute_shared_ylim(df_list, clim_list, manual_ylim=None):
    if manual_ylim is not None:
        return manual_ylim

    y_all = []
    for df in df_list:
        if df is not None and len(df["year"]) > 0:
            y_all.append(np.asarray(df["y"], dtype=float))

    for clim in clim_list:
        if clim is not None:
            y_all.append(np.asarray(clim["y"], dtype=float))

    if len(y_all) == 0:
        return (0, 1)

    y_all = np.concatenate(y_all)
    y_all = y_all[np.isfinite(y_all)]

    if y_all.size == 0:
        return (0, 1)

    y0 = float(np.nanmin(y_all))
    y1 = float(np.nanmax(y_all))
    pad = 1.0 if y0 == y1 else 0.06 * (y1 - y0)
    return (y0 - pad, y1 + pad)

def save_axis_limits_txt(out_dir, xlim, ylim):
    os.makedirs(out_dir, exist_ok=True)
    txt_path = os.path.join(out_dir, "MA_O3minValue_vs_O3minDate_axis_limits.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(f"xlim = ({xlim[0]:.10g}, {xlim[1]:.10g})\n")
        f.write(f"ylim = ({ylim[0]:.10g}, {ylim[1]:.10g})\n")
        f.write("x axis = day since Mar 1\n")
        f.write("0=Mar-01, 30=Mar-31, 31=Apr-01, 60=Apr-30\n")
        f.write("color map: March=royalblue, April=gold\n")
    print("Saved axis-limit record:", txt_path)

def save_combined_csv(df_map, out_dir):
    rows = []
    for group_name, df in df_map.items():
        if df is None:
            continue
        n = len(df["year"])
        for i in range(n):
            rows.append({
                "group": group_name,
                "part": df["part"][i],
                "year": int(df["year"][i]),
                "day_from_mar1": int(df["x"][i]),
                "ma_min_o3": float(df["y"][i]),
                "min_month": int(df["min_month"][i]),
                "min_day": int(df["min_day"][i]),
                "min_date_str": str(df["min_date_str"][i]),
            })

    if len(rows) == 0:
        print("No valid rows, skip CSV.")
        return None

    out_csv = os.path.join(out_dir, "MA_O3minValue_vs_O3minDate_points.csv")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print("Saved CSV:", out_csv)
    return out_csv

def select_extreme_idx(df, topn=5):
    if df is None or len(df["year"]) == 0:
        return np.array([], dtype=int)

    y = np.asarray(df["y"], dtype=float)
    valid = np.isfinite(y)
    if not np.any(valid):
        return np.array([], dtype=int)

    idx_valid = np.where(valid)[0]
    idx_sorted = idx_valid[np.argsort(y[valid])]
    return idx_sorted[:min(topn, len(idx_sorted))]

def build_sources():
    sources = {
        "coupled": {
            "label": "Coupled (BWCN+B2000WCN)",
            "parts": [
                {
                    "name": "BWCN",
                    "data": load_one_o3_source(
                        FILE_BWCN_O3,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
                {
                    "name": "B2000WCN",
                    "data": load_one_o3_source(
                        FILE_B2000_O3,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                },
            ]
        },
        "nocoupl": {
            "label": "B2000WCN NOCOUPL",
            "parts": [
                {
                    "name": "B2000WCN_NOCOUPL",
                    "data": load_one_o3_source(
                        FILE_NOCOUPL_O3,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        },
        "merra2": {
            "label": "MERRA2",
            "parts": [
                {
                    "name": "MERRA2",
                    "data": load_one_o3_source(
                        FILE_MERRA_O3,
                        apply_o3_5d=APPLY_O3_5D,
                        drop_leap_day=False
                    )
                }
            ]
        }
    }
    return sources

def close_sources(sources):
    for group in sources.values():
        for part in group["parts"]:
            da = part["data"].get("o3_da", None)
            if da is not None and hasattr(da, "close"):
                try:
                    da.close()
                except Exception:
                    pass

def draw_one_panel(ax, df, clim_curve, panel_title, xlim, ylim):
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.axvline(0, color="k", lw=0.8, alpha=0.35)
    ax.axvline(31, color="k", lw=0.8, alpha=0.35)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    xticks = [0, 14, 31, 45, 60]
    xtlabs = ["Mar-01", "Mar-15", "Apr-01", "Apr-15", "Apr-30"]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xtlabs)

    if df is None or len(df["year"]) == 0:
        ax.set_title(panel_title, fontweight="bold")
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center",
                transform=ax.transAxes, fontsize=11)
        ax.set_xlabel("Date of MA minimum O$_3$ (days since Mar 1)")
        ax.set_ylabel("MA minimum O$_3$ (5-day mean, raw)")
        return

    # climatology curve
    if clim_curve is not None:
        ax.plot(
            clim_curve["x"], clim_curve["y"],
            color=CLIM_LINE_COLOR,
            ls=CLIM_LINESTYLE,
            lw=CLIM_LINEWIDTH,
            zorder=1
        )

    # scatter
    for month in [3, 4]:
        mask = df["min_month"] == month
        if np.any(mask):
            ax.scatter(
                df["x"][mask], df["y"][mask],
                s=POINT_SIZE,
                color=MONTH_COLOR[month],
                alpha=POINT_ALPHA,
                edgecolors="k",
                linewidths=POINT_EDGE_LW,
                zorder=3
            )

    # extreme years
    idx_ext = np.array([], dtype=int)
    if MARK_EXTREMES:
        idx_ext = select_extreme_idx(df, topn=EXTREME_TOPN)
        for i, idx in enumerate(idx_ext):
            x0 = float(df["x"][idx])
            y0 = float(df["y"][idx])
            yr = int(df["year"][idx])
            part_name = str(df["part"][idx]) # 获取该点所属的子数据集名称
            
            c = EXTREME_COLORS[i % len(EXTREME_COLORS)]

            # --- 修改逻辑：如果是 BWCN 则添加前缀 ---
            if part_name == "BWCN":
                label_text = f"BWCN{yr:04d}"
            else:
                label_text = f"{yr:04d}"
            # ------------------------------------

            ax.scatter(
                x0, y0,
                s=EXTREME_SIZE,
                facecolors="none",
                edgecolors=c,
                linewidths=EXTREME_LINEWIDTH,
                zorder=5
            )
            
            ax.text(
                x0 + 0.7, y0 + 0.3,
                label_text, # 使用带前缀或不带前缀的标签
                color=c,
                fontsize=EXTREME_TEXT_FS,
                ha="left", va="bottom",
                zorder=6
            )

    # stats
    mask_mar = df["min_month"] == 3
    mask_apr = df["min_month"] == 4

    # 1. 减小行间距：将原本的双换行 "\n\n" 改为单换行 "\n"
    txt_all = format_rn_text("ALL", df["x"], df["y"])
    txt_mar = format_rn_text("MAR", df["x"][mask_mar], df["y"][mask_mar])
    txt_apr = format_rn_text("APR", df["x"][mask_apr], df["y"][mask_apr])

    # 合并文本，使用单换行使格式更紧凑
    stat_text = txt_all + "\n" + txt_mar + "\n" + txt_apr
    
    # 2. 调整位置到左下角 (0.03, 0.03)
    # 3. 减小 fontsize 到 7.5
    # 4. 增加 alpha (透明度) 到 0.6 或更低，使背景更透明
    ax.text(
        0.03, 0.03, 
        stat_text,
        transform=ax.transAxes,
        ha="left", va="bottom",  # 锚点改为左下
        fontsize=7.5,             # 字体改小
        linespacing=1.1,         # 进一步微调行间距
        zorder=10,               # 确保在最上层
        bbox=dict(
            boxstyle="round,pad=0.3", 
            facecolor="white", 
            edgecolor="0.8", 
            alpha=0.6            # 增加透明度 (值越小越透明)
        )
    )

    ax.set_title(panel_title, fontweight="bold")
    ax.set_xlabel("Date of MA minimum O$_3$ (days since Mar 1)")
    ax.set_ylabel("MA minimum O$_3$ (5-day mean, raw)")

def plot_static_three_panel(sources):
    os.makedirs(PLOT_BASE_NEW, exist_ok=True)

    df_merra   = build_group_scatter_df(sources["merra2"])
    df_coupled = build_group_scatter_df(sources["coupled"])
    df_noc     = build_group_scatter_df(sources["nocoupl"])

    clim_merra   = build_group_clim_curve(sources["merra2"])
    clim_coupled = build_group_clim_curve(sources["coupled"])
    clim_noc     = build_group_clim_curve(sources["nocoupl"])

    save_combined_csv(
        {
            "MERRA2": df_merra,
            "Coupled_BWCN_B2000WCN": df_coupled,
            "B2000WCN_NOCOUPL": df_noc,
        },
        PLOT_BASE_NEW
    )

    xlim = MANUAL_XLIM if USE_MANUAL_XLIM else (-1, 61)
    ylim = MANUAL_YLIM if USE_MANUAL_YLIM else compute_shared_ylim(
        [df_merra, df_coupled, df_noc],
        [clim_merra, clim_coupled, clim_noc],
        manual_ylim=None
    )

    debug(f"[DEBUG] xlim = {xlim}")
    debug(f"[DEBUG] ylim = {ylim}")

    save_axis_limits_txt(PLOT_BASE_NEW, xlim, ylim)

    fig = plt.figure(figsize=(12.8, 9.2), dpi=160)
    gs = fig.add_gridspec(
        2, 3,
        height_ratios=[1.0, 1.0],
        width_ratios=[1.0, 1.0, 1.0],
        hspace=0.38, wspace=0.32
    )

    ax_top = fig.add_subplot(gs[0, 1])
    ax_bl  = fig.add_subplot(gs[1, 0])
    ax_br  = fig.add_subplot(gs[1, 2])

    draw_one_panel(ax_top, df_merra,   clim_merra,   "MERRA2",                  xlim, ylim)
    draw_one_panel(ax_bl,  df_coupled, clim_coupled, "Coupled (BWCN+B2000WCN)", xlim, ylim)
    draw_one_panel(ax_br,  df_noc,     clim_noc,     "B2000WCN NOCOUPL",        xlim, ylim)

    fig.suptitle(
        "MA minimum O$_3$ value VS date of MA minimum O$_3$",
        fontsize=15.5, fontweight="bold", y=0.972
    )

    legend_handles = [
        Line2D([0], [0], marker='o', color='w', label='March minimum',
               markerfacecolor=MONTH_COLOR[3], markeredgecolor='k', markersize=8),
        Line2D([0], [0], marker='o', color='w', label='April minimum',
               markerfacecolor=MONTH_COLOR[4], markeredgecolor='k', markersize=8),
        Line2D([0], [0], color=CLIM_LINE_COLOR, lw=CLIM_LINEWIDTH,
               ls=CLIM_LINESTYLE, label='MA climatology'),
        Line2D([0], [0], marker='o', color='w', label=f'Lowest {EXTREME_TOPN} years',
               markerfacecolor='none', markeredgecolor='k', markeredgewidth=1.2, markersize=9),
    ]
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.93),
        ncol=4,
        frameon=False,
        fontsize=10.3
    )

    out_png = os.path.join(
        PLOT_BASE_NEW,
        "Scatter_MA_O3minValue_vs_O3minDate_MERRA2_B2000_NOCOUPL_extreme_clim.png"
    )
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    print("Saved PNG:", out_png)
    return out_png

# ============================================================
# Run
# ============================================================
if __name__ == "__main__":
    os.makedirs(PLOT_BASE_NEW, exist_ok=True)

    sources = build_sources()

    try:
        plot_static_three_panel(sources)
    finally:
        close_sources(sources)

    print("\n[MA minimum O3 value VS date of MA minimum O3] All done.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import pearsonr

# ============================================================
# Paths & Config
# ============================================================
FW_TXT_BWCN    = "/mnt/soclim0/public_data/weiji/BWCN/BWCN_final_warming_date.txt"
FW_TXT_B2000   = "/mnt/soclim0/public_data/weiji/B2000WCN001002/B2000WCN_final_warming_date.txt"
FW_TXT_NOCOUPL = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/B2000WCN_NOCOUPL_final_warming_date.txt"
FW_TXT_MERRA2  = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/MERRA2_final_warming_date.txt"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"
OUT_DIR   = os.path.join(PLOT_BASE, "STATIC_SCATTER_FWD_VS_O3")
os.makedirs(OUT_DIR, exist_ok=True)

BRIDGE_YEAR = 104
APPLY_O3_5D = True

FIG_DPI = 220

BG_COLOR = "0.35"
LOW25_COLOR = "red"
BG_ALPHA = 0.75
LOW25_ALPHA = 0.95
BG_SIZE = 34
LOW25_SIZE = 42

MARK_EXTREMES = True
EXTREME_TOPN = 5
EXTREME_BY = "y_raw"
EXTREME_ASCENDING = True
EXTREME_SIZE = 120
EXTREME_LINEWIDTH = 1.6
EXTREME_COLORS = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

USE_MANUAL_AXIS_LIMITS = False
MANUAL_XLIM = (60, 150)
MANUAL_YLIM = (60, 170)

# climatology / FWD band styles
CLIM_COLOR = "black"
CLIM_LINESTYLE = "-"
CLIM_LINEWIDTH = 1.6
CLIM_ALPHA = 0.95

FWD_MEAN_COLOR = "#1f77b4"
FWD_BAND_COLOR = "#1f77b4"
FWD_MEAN_LINESTYLE = "--"
FWD_MEAN_LINEWIDTH = 1.2
FWD_BAND_ALPHA = 0.10


# ============================================================
# Helpers
# ============================================================
def doy_to_mmdd(x, pos):
    if np.isnan(x) or x < 1 or x > 365:
        return ""
    base_date = datetime(2001, 1, 1)
    target_date = base_date + timedelta(days=int(round(x)) - 1)
    return target_date.strftime("%m-%d")


def load_fwd_data(txt_path):
    fwd_dict = {}
    if not os.path.exists(txt_path):
        print(f"Warning: FWD file not found -> {txt_path}")
        return fwd_dict

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("#") or not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                year = int(parts[0])
                doy_str = parts[1]
                if doy_str.lower() != "nan":
                    fwd_dict[year] = float(doy_str)
    return fwd_dict


def assert_daily_unique(da, name=""):
    da = da.sortby("time")
    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)
    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")
    return da


def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)


def build_mmdd_order():
    month_days = [
        (1, 31), (2, 28), (3, 31), (4, 30), (5, 31), (6, 30),
        (7, 31), (8, 31), (9, 30), (10, 31), (11, 30), (12, 31)
    ]
    out = []
    for m, nd in month_days:
        for d in range(1, nd + 1):
            out.append(f"{m:02d}-{d:02d}")
    return out


MMDD_ORDER = build_mmdd_order()
MMDD_TO_DOY = {mmdd: i + 1 for i, mmdd in enumerate(MMDD_ORDER)}


def get_ma_min_o3_series(o3_nc, apply_o3_5d=True):
    if not os.path.exists(o3_nc):
        print(f"Warning: O3 file not found -> {o3_nc}")
        return {}

    da = xr.open_dataarray(o3_nc)
    da = assert_daily_unique(da, name="O3").sortby("time")

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(da.time.dt.year.values.astype(int))
    o3_dict = {}

    for yr in years:
        mask = (da.time.dt.year == yr) & (da.time.dt.month.isin([3, 4]))
        seg = da.where(mask, drop=True)
        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid >= 58:
            o3_dict[int(yr)] = float(seg.min().values)

    da.close()
    return o3_dict


def get_low25_years(txt_path):
    if txt_path and os.path.exists(txt_path):
        return set(np.loadtxt(txt_path, dtype=int).reshape(-1))
    return set()


def load_o3_daily_climatology_curve(o3_nc, apply_o3_5d=True, drop_years=None):
    """
    返回 climatology curve:
      x_clim: 1..365 (DOY, noleap)
      y_clim: daily climatological partial O3 column
      n_years: 实际参与 climatology 的有效年份数
    """
    if not os.path.exists(o3_nc):
        print(f"Warning: O3 file not found -> {o3_nc}")
        return None

    da = xr.open_dataarray(o3_nc)
    da = assert_daily_unique(da, name="O3_daily").sortby("time")
    da = drop_feb29(da)

    if drop_years:
        da = da.sel(time=~da.time.dt.year.isin(list(drop_years)))

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    # 统计有效年份数
    years = np.unique(da.time.dt.year.values.astype(int))
    valid_years = []
    for yr in years:
        seg = da.sel(time=da.time.dt.year == yr)
        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid >= 300:   # 全年 climatology，给一个宽松但合理的门槛
            valid_years.append(int(yr))

    mmdd = np.array([f"{m:02d}-{d:02d}" for m, d in zip(
        da.time.dt.month.values.astype(int),
        da.time.dt.day.values.astype(int)
    )])
    da = da.assign_coords(mmdd=("time", mmdd))

    clim = da.groupby("mmdd").mean("time")
    clim = clim.sel(mmdd=MMDD_ORDER)

    x_clim = np.arange(1, 366, dtype=float)
    y_clim = np.asarray(clim.values, dtype=float)

    da.close()
    return {
        "x": x_clim,
        "y": y_clim,
        "n_years": len(valid_years),
    }


def average_climatology_curves(curves):
    curves = [c for c in curves if c is not None]
    if len(curves) == 0:
        return None
    if len(curves) == 1:
        return curves[0]

    x = curves[0]["x"]
    weights = np.array([c.get("n_years", 1) for c in curves], dtype=float)
    y_stack = np.vstack([c["y"] for c in curves])

    y_avg = np.average(y_stack, axis=0, weights=weights)

    return {
        "x": x,
        "y": y_avg,
        "n_years": int(np.sum(weights)),
    }


def build_scatter_df(fwd_txt, o3_nc, low25_txt, part_name, is_bridge_case=False):
    fwd_data = load_fwd_data(fwd_txt)
    o3_data = get_ma_min_o3_series(o3_nc, apply_o3_5d=APPLY_O3_5D)
    low25_set = get_low25_years(low25_txt)

    common_years = sorted(list(set(fwd_data.keys()).intersection(set(o3_data.keys()))))

    if is_bridge_case and BRIDGE_YEAR in common_years:
        common_years.remove(BRIDGE_YEAR)

    x_list, y_list, low25_mask, year_list, part_list = [], [], [], [], []
    for y in common_years:
        x_list.append(fwd_data[y])
        y_list.append(o3_data[y])
        low25_mask.append(y in low25_set)
        year_list.append(y)
        part_list.append(part_name)

    return {
        "x": np.array(x_list, dtype=float),
        "y": np.array(y_list, dtype=float),
        "y_raw": np.array(y_list, dtype=float),
        "year": np.array(year_list, dtype=int),
        "part": np.array(part_list, dtype=object),
        "is_low25": np.array(low25_mask, dtype=bool)
    }


def combine_dfs(df1, df2):
    if df1 is None or df1["x"].size == 0:
        return df2
    if df2 is None or df2["x"].size == 0:
        return df1
    return {
        "x": np.concatenate([df1["x"], df2["x"]]),
        "y": np.concatenate([df1["y"], df2["y"]]),
        "y_raw": np.concatenate([df1["y_raw"], df2["y_raw"]]),
        "year": np.concatenate([df1["year"], df2["year"]]),
        "part": np.concatenate([df1["part"], df2["part"]]),
        "is_low25": np.concatenate([df1["is_low25"], df2["is_low25"]])
    }


def select_panel_extremes(df, topn=5, extreme_by="y_raw", ascending=True):
    if df is None or len(df["year"]) == 0:
        return np.array([], dtype=int)

    vals = np.asarray(df[extreme_by], dtype=float)
    valid_idx = np.where(np.isfinite(vals))[0]
    if valid_idx.size == 0:
        return np.array([], dtype=int)

    order = np.argsort(vals[valid_idx])
    if not ascending:
        order = order[::-1]

    keep = valid_idx[order[:min(topn, len(order))]]
    return keep


def overlay_panel_extremes(ax, df, idx_extreme):
    if df is None or len(idx_extreme) == 0:
        return []

    legend_entries = []

    for i, idx in enumerate(idx_extreme):
        c = EXTREME_COLORS[i % len(EXTREME_COLORS)]
        x = float(df["x"][idx])
        y = float(df["y"][idx])
        part = str(df["part"][idx])
        year = int(df["year"][idx])

        if part == "BWCN":
            label = f"BWCN-{year:04d}"
        else:
            label = f"{year:04d}"

        legend_entries.append((i, label, c, part))

        if part == "BWCN":
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                facecolors="none",
                edgecolors=c,
                linewidths=EXTREME_LINEWIDTH,
                zorder=10
            )
        else:
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                color=c,
                edgecolors="k",
                linewidths=0.8,
                zorder=10
            )

    return legend_entries


def compute_global_axis_limits(dfs, clim_curves=None, use_manual=False,
                               manual_xlim=None, manual_ylim=None,
                               xpad=5.0, ypad=3.0):
    if use_manual:
        return manual_xlim, manual_ylim

    all_x = []
    all_y = []

    for df in dfs:
        if df is not None and df["x"].size > 0:
            all_x.extend(df["x"])
            all_y.extend(df["y"])

    if clim_curves is not None:
        for c in clim_curves:
            if c is not None:
                all_x.extend(c["x"])
                all_y.extend(c["y"])

    all_x = np.asarray(all_x, dtype=float)
    all_y = np.asarray(all_y, dtype=float)

    if all_x.size == 0 or all_y.size == 0:
        return (60, 150), (60, 170)

    xmin, xmax = np.nanmin(all_x), np.nanmax(all_x)
    ymin, ymax = np.nanmin(all_y), np.nanmax(all_y)

    return (xmin - xpad, xmax + xpad), (ymin - ypad, ymax + ypad)


# ============================================================
# Plotting
# ============================================================
def draw_one_panel(ax, df, clim_curve, panel_title, xlim, ylim):
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # 右侧 Y 轴用于 climatology curve
    ax2 = ax.twinx()
    ax2.set_ylim(ylim)
    ax2.set_ylabel(r"O$_3$ climatology (DU)", color=CLIM_COLOR)
    ax2.tick_params(axis='y', colors=CLIM_COLOR)

    if df is None or len(df["x"]) == 0:
        ax.set_title(panel_title, fontweight="bold")
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel("Final Warming Date")
        ax.set_ylabel(r"MA min O$_3$ (DU)")
        ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))
        return

    low_mask = df["is_low25"]
    bg_mask = ~low_mask

    # scatter points
    if np.any(bg_mask):
        ax.scatter(
            df["x"][bg_mask], df["y"][bg_mask],
            s=BG_SIZE, color=BG_COLOR, alpha=BG_ALPHA,
            edgecolors="none", zorder=2
        )

    if np.any(low_mask):
        ax.scatter(
            df["x"][low_mask], df["y"][low_mask],
            s=LOW25_SIZE, color=LOW25_COLOR, alpha=LOW25_ALPHA,
            edgecolors="k", linewidths=0.4, zorder=5
        )

    # extreme points
    legend_entries = []
    if MARK_EXTREMES:
        idx_ext = select_panel_extremes(
            df,
            topn=EXTREME_TOPN,
            extreme_by=EXTREME_BY,
            ascending=EXTREME_ASCENDING
        )
        legend_entries = overlay_panel_extremes(ax, df, idx_ext)

    # FWD mean ± sigma band + mean line
    mean_fwd = np.nanmean(df["x"])
    std_fwd  = np.nanstd(df["x"], ddof=0)
    mean_fwd_low25 = np.nanmean(df["x"][low_mask]) if np.any(low_mask) else np.nan
    std_fwd_low25 = np.nanstd(df["x"][low_mask], ddof=0) if np.any(low_mask) else np.nan

    # mean ± 1σ band
    ax.axvspan(mean_fwd - std_fwd, mean_fwd + std_fwd,
            color=FWD_BAND_COLOR, alpha=0.18, zorder=0)

    # mean line
    ax.axvline(mean_fwd, color=FWD_MEAN_COLOR, linestyle="-",
            linewidth=1.4, alpha=0.95, zorder=1)

    # ±1σ boundary lines
    ax.axvline(mean_fwd - std_fwd, color=FWD_MEAN_COLOR, linestyle="--",
            linewidth=1.0, alpha=0.8, zorder=1)
    ax.axvline(mean_fwd + std_fwd, color=FWD_MEAN_COLOR, linestyle="--",
            linewidth=1.0, alpha=0.8, zorder=1)
    # LOW25 mean FWD line only
    if np.isfinite(mean_fwd_low25):
        ax.axvline(mean_fwd_low25, color=LOW25_COLOR, linestyle="-",
                linewidth=1.0, alpha=0.8, zorder=1)
        
    if np.isfinite(mean_fwd_low25) and np.isfinite(std_fwd_low25):
        ax.axvspan(mean_fwd_low25 - std_fwd_low25,
                mean_fwd_low25 + std_fwd_low25,
                color=LOW25_COLOR, alpha=0.12, zorder=0)
    # climatology curve on right y-axis
    if clim_curve is not None:
        ax2.plot(
            clim_curve["x"], clim_curve["y"],
            color=CLIM_COLOR, linestyle=CLIM_LINESTYLE,
            linewidth=CLIM_LINEWIDTH, alpha=CLIM_ALPHA, zorder=1
        )
    

    # statistics
    n_all = len(df["x"])
    if n_all >= 3:
        try:
            r_all, p_all = pearsonr(df["x"], df["y"])
            txt_all = f"ALL\nr = {r_all:.3f}\np = {p_all:.2e}\nN = {n_all}"
        except Exception:
            txt_all = f"ALL\nN = {n_all}"
    else:
        txt_all = f"ALL\nN = {n_all}"

    ax.text(
        0.04, 0.96, txt_all,
        transform=ax.transAxes, va="top", ha="left", fontsize=9.2, color="k",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor="0.7")
    )

    n_low = int(np.sum(low_mask))
    if n_low >= 3:
        try:
            r_low, p_low = pearsonr(df["x"][low_mask], df["y"][low_mask])
            txt_low = f"LOW25\nr = {r_low:.3f}\np = {p_low:.2e}\nN = {n_low}"
        except Exception:
            txt_low = f"LOW25\nN = {n_low}"
    else:
        txt_low = f"LOW25\nN = {n_low}"

    ax.text(
        0.96, 0.96, txt_low,
        transform=ax.transAxes, va="top", ha="right", fontsize=9.2, color=LOW25_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor=LOW25_COLOR)
    )

    # legend
    handles = []
    labels = []

    for i, label, c, part in legend_entries:
        if part == "BWCN":
            h = Line2D(
                [0], [0],
                marker="o",
                color="none",
                markeredgecolor=c,
                markerfacecolor="none",
                markeredgewidth=1.2,
                markersize=5.0,
                linewidth=0
            )
        else:
            h = Line2D(
                [0], [0],
                marker="o",
                color="none",
                markeredgecolor="k",
                markerfacecolor=c,
                markeredgewidth=0.6,
                markersize=5.0,
                linewidth=0
            )
        handles.append(h)
        labels.append(f"Top{i+1}: {label}")

    handles.extend([
        Line2D([0], [0], color=CLIM_COLOR, lw=CLIM_LINEWIDTH, ls=CLIM_LINESTYLE),
        Line2D([0], [0], color=FWD_MEAN_COLOR, lw=1.4, ls="-"),
        Patch(facecolor=FWD_BAND_COLOR, edgecolor="none", alpha=0.18),
    ])
    labels.extend([
        "O$_3$ climatology",
        f"ALL mean FWD: {doy_to_mmdd(mean_fwd, None)}",
        r"ALL FWD $\pm 1\sigma$"
    ])

    if np.isfinite(mean_fwd_low25):
        handles.append(Line2D([0], [0], color=LOW25_COLOR, lw=1.4, ls="-"))
        labels.append(f"LOW25 mean FWD: {doy_to_mmdd(mean_fwd_low25, None)}")

    ax.legend(
        handles, labels,
        loc="lower left",
        fontsize=4.8,
        frameon=False,
        handletextpad=0.35,
        borderpad=0.2,
        labelspacing=0.25
    )

    ax.set_title(panel_title, fontweight="bold")
    ax.set_xlabel("Final Warming Date")
    ax.set_ylabel(r"MA min O$_3$ (DU)")
    ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))


# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    print("Loading data...")

    # scatter data
    df_merra = build_scatter_df(
        FW_TXT_MERRA2,
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        os.path.join(DATA_BASE, "MERRA2", "low25pct_years_30_70hPa.txt"),
        part_name="MERRA2",
        is_bridge_case=False
    )

    df_bwcn = build_scatter_df(
        FW_TXT_BWCN,
        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "BWCN", "low25pct_years_30_70hPa.txt"),
        part_name="BWCN",
        is_bridge_case=False
    )

    df_b2000 = build_scatter_df(
        FW_TXT_B2000,
        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN", "low25pct_years_30_70hPa.txt"),
        part_name="B2000WCN",
        is_bridge_case=True
    )

    df_coupled = combine_dfs(df_bwcn, df_b2000)

    df_nocoupl = build_scatter_df(
        FW_TXT_NOCOUPL,
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"),
        part_name="B2000WCN_NOCOUPL",
        is_bridge_case=True
    )

    # climatology curves
    clim_merra = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years=None
    )

    clim_bwcn = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years=None
    )

    clim_b2000 = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years={BRIDGE_YEAR}
    )

    clim_coupled = average_climatology_curves([clim_bwcn, clim_b2000])

    clim_nocoupl = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years={BRIDGE_YEAR}
    )

    # global axes
    xlim, ylim = compute_global_axis_limits(
        [df_merra, df_coupled, df_nocoupl],
        clim_curves=None,
        use_manual=USE_MANUAL_AXIS_LIMITS,
        manual_xlim=MANUAL_XLIM,
        manual_ylim=MANUAL_YLIM
    )

    print("Global xlim =", xlim)
    print("Global ylim =", ylim)

    print("Plotting...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), dpi=FIG_DPI)
    plt.subplots_adjust(wspace=0.30)

    draw_one_panel(axes[0], df_merra,   clim_merra,   "MERRA2",             xlim, ylim)
    draw_one_panel(axes[1], df_coupled, clim_coupled, "B2000WCN (Coupled)", xlim, ylim)
    draw_one_panel(axes[2], df_nocoupl, clim_nocoupl, "B2000WCN NOCOUPL",   xlim, ylim)

    fig.suptitle("Scatter: MA min O$_3$ vs Final Warming Date (FWD)", fontsize=16, fontweight="bold", y=1.03)

    for ax in axes:
        for tick in ax.get_xticklabels():
            tick.set_rotation(45)

    save_path = os.path.join(OUT_DIR, "Scatter_FWD_vs_O3_MERRA2_B2000_NOCOUPL_extreme_climcurve_fwdband.png")
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    print(f"Plot saved successfully at:\n{save_path}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import pearsonr

# ============================================================
# Paths & Config
# ============================================================
FW_TXT_BWCN    = "/mnt/soclim0/public_data/weiji/BWCN/BWCN_final_warming_date.txt"
FW_TXT_B2000   = "/mnt/soclim0/public_data/weiji/B2000WCN001002/B2000WCN_final_warming_date.txt"
FW_TXT_NOCOUPL = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/B2000WCN_NOCOUPL_final_warming_date.txt"
FW_TXT_MERRA2  = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/MERRA2_final_warming_date.txt"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"
OUT_DIR   = os.path.join(PLOT_BASE, "STATIC_SCATTER_FWD_VS_O3")
os.makedirs(OUT_DIR, exist_ok=True)

BRIDGE_YEAR = 104
APPLY_O3_5D = True

FIG_DPI = 220

BG_COLOR = "0.35"
LOW25_COLOR = "red"
BG_ALPHA = 0.75
LOW25_ALPHA = 0.95
BG_SIZE = 34
LOW25_SIZE = 42

MARK_EXTREMES = True
EXTREME_TOPN = 5
EXTREME_BY = "y_raw"
EXTREME_ASCENDING = True
EXTREME_SIZE = 120
EXTREME_LINEWIDTH = 1.6
EXTREME_COLORS = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

USE_MANUAL_AXIS_LIMITS = False
MANUAL_XLIM = (60, 150)
MANUAL_YLIM = (60, 170)

# climatology / FWD band styles
CLIM_COLOR = "black"
CLIM_LINESTYLE = "-"
CLIM_LINEWIDTH = 1.6
CLIM_ALPHA = 0.95

FWD_MEAN_COLOR = "#1f77b4"
FWD_BAND_COLOR = "#1f77b4"
FWD_MEAN_LINESTYLE = "--"
FWD_MEAN_LINEWIDTH = 1.2
FWD_BAND_ALPHA = 0.10


# ============================================================
# Helpers
# ============================================================
def doy_to_mmdd(x, pos):
    if np.isnan(x) or x < 1 or x > 365:
        return ""
    base_date = datetime(2001, 1, 1)
    target_date = base_date + timedelta(days=int(round(x)) - 1)
    return target_date.strftime("%m-%d")


def load_fwd_data(txt_path):
    fwd_dict = {}
    if not os.path.exists(txt_path):
        print(f"Warning: FWD file not found -> {txt_path}")
        return fwd_dict

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("#") or not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                year = int(parts[0])
                doy_str = parts[1]
                if doy_str.lower() != "nan":
                    fwd_dict[year] = float(doy_str)
    return fwd_dict


def assert_daily_unique(da, name=""):
    da = da.sortby("time")
    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)
    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")
    return da


def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)


def build_mmdd_order():
    month_days = [
        (1, 31), (2, 28), (3, 31), (4, 30), (5, 31), (6, 30),
        (7, 31), (8, 31), (9, 30), (10, 31), (11, 30), (12, 31)
    ]
    out = []
    for m, nd in month_days:
        for d in range(1, nd + 1):
            out.append(f"{m:02d}-{d:02d}")
    return out


MMDD_ORDER = build_mmdd_order()
MMDD_TO_DOY = {mmdd: i + 1 for i, mmdd in enumerate(MMDD_ORDER)}


def get_ma_min_o3_series(o3_nc, apply_o3_5d=True):
    if not os.path.exists(o3_nc):
        print(f"Warning: O3 file not found -> {o3_nc}")
        return {}

    da = xr.open_dataarray(o3_nc)
    da = assert_daily_unique(da, name="O3").sortby("time")

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(da.time.dt.year.values.astype(int))
    o3_dict = {}

    for yr in years:
        mask = (da.time.dt.year == yr) & (da.time.dt.month.isin([3, 4]))
        seg = da.where(mask, drop=True)
        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid >= 58:
            o3_dict[int(yr)] = float(seg.min().values)

    da.close()
    return o3_dict


def get_low25_years(txt_path):
    if txt_path and os.path.exists(txt_path):
        return set(np.loadtxt(txt_path, dtype=int).reshape(-1))
    return set()


def load_o3_daily_climatology_curve(o3_nc, apply_o3_5d=True, drop_years=None):
    """
    返回 climatology curve:
      x_clim: 1..365 (DOY, noleap)
      y_clim: daily climatological partial O3 column
      n_years: 实际参与 climatology 的有效年份数
    """
    if not os.path.exists(o3_nc):
        print(f"Warning: O3 file not found -> {o3_nc}")
        return None

    da = xr.open_dataarray(o3_nc)
    da = assert_daily_unique(da, name="O3_daily").sortby("time")
    da = drop_feb29(da)

    if drop_years:
        da = da.sel(time=~da.time.dt.year.isin(list(drop_years)))

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    # 统计有效年份数
    years = np.unique(da.time.dt.year.values.astype(int))
    valid_years = []
    for yr in years:
        seg = da.sel(time=da.time.dt.year == yr)
        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid >= 300:   # 全年 climatology，给一个宽松但合理的门槛
            valid_years.append(int(yr))

    mmdd = np.array([f"{m:02d}-{d:02d}" for m, d in zip(
        da.time.dt.month.values.astype(int),
        da.time.dt.day.values.astype(int)
    )])
    da = da.assign_coords(mmdd=("time", mmdd))

    clim = da.groupby("mmdd").mean("time")
    clim = clim.sel(mmdd=MMDD_ORDER)

    x_clim = np.arange(1, 366, dtype=float)
    y_clim = np.asarray(clim.values, dtype=float)

    da.close()
    return {
        "x": x_clim,
        "y": y_clim,
        "n_years": len(valid_years),
    }


def average_climatology_curves(curves):
    curves = [c for c in curves if c is not None]
    if len(curves) == 0:
        return None
    if len(curves) == 1:
        return curves[0]

    x = curves[0]["x"]
    weights = np.array([c.get("n_years", 1) for c in curves], dtype=float)
    y_stack = np.vstack([c["y"] for c in curves])

    y_avg = np.average(y_stack, axis=0, weights=weights)

    return {
        "x": x,
        "y": y_avg,
        "n_years": int(np.sum(weights)),
    }


def build_scatter_df(fwd_txt, o3_nc, low25_txt, part_name, is_bridge_case=False):
    fwd_data = load_fwd_data(fwd_txt)
    o3_data = get_ma_min_o3_series(o3_nc, apply_o3_5d=APPLY_O3_5D)
    low25_set = get_low25_years(low25_txt)

    common_years = sorted(list(set(fwd_data.keys()).intersection(set(o3_data.keys()))))

    if is_bridge_case and BRIDGE_YEAR in common_years:
        common_years.remove(BRIDGE_YEAR)

    x_list, y_list, low25_mask, year_list, part_list = [], [], [], [], []
    for y in common_years:
        x_list.append(fwd_data[y])
        y_list.append(o3_data[y])
        low25_mask.append(y in low25_set)
        year_list.append(y)
        part_list.append(part_name)

    return {
        "x": np.array(x_list, dtype=float),
        "y": np.array(y_list, dtype=float),
        "y_raw": np.array(y_list, dtype=float),
        "year": np.array(year_list, dtype=int),
        "part": np.array(part_list, dtype=object),
        "is_low25": np.array(low25_mask, dtype=bool)
    }


def combine_dfs(df1, df2):
    if df1 is None or df1["x"].size == 0:
        return df2
    if df2 is None or df2["x"].size == 0:
        return df1
    return {
        "x": np.concatenate([df1["x"], df2["x"]]),
        "y": np.concatenate([df1["y"], df2["y"]]),
        "y_raw": np.concatenate([df1["y_raw"], df2["y_raw"]]),
        "year": np.concatenate([df1["year"], df2["year"]]),
        "part": np.concatenate([df1["part"], df2["part"]]),
        "is_low25": np.concatenate([df1["is_low25"], df2["is_low25"]])
    }


def select_panel_extremes(df, topn=5, extreme_by="y_raw", ascending=True):
    if df is None or len(df["year"]) == 0:
        return np.array([], dtype=int)

    vals = np.asarray(df[extreme_by], dtype=float)
    valid_idx = np.where(np.isfinite(vals))[0]
    if valid_idx.size == 0:
        return np.array([], dtype=int)

    order = np.argsort(vals[valid_idx])
    if not ascending:
        order = order[::-1]

    keep = valid_idx[order[:min(topn, len(order))]]
    return keep


def overlay_panel_extremes(ax, df, idx_extreme):
    if df is None or len(idx_extreme) == 0:
        return []

    legend_entries = []

    for i, idx in enumerate(idx_extreme):
        c = EXTREME_COLORS[i % len(EXTREME_COLORS)]
        x = float(df["x"][idx])
        y = float(df["y"][idx])
        part = str(df["part"][idx])
        year = int(df["year"][idx])

        if part == "BWCN":
            label = f"BWCN-{year:04d}"
        else:
            label = f"{year:04d}"

        legend_entries.append((i, label, c, part))

        if part == "BWCN":
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                facecolors="none",
                edgecolors=c,
                linewidths=EXTREME_LINEWIDTH,
                zorder=10
            )
        else:
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                color=c,
                edgecolors="k",
                linewidths=0.8,
                zorder=10
            )

    return legend_entries


def compute_global_axis_limits(dfs, clim_curves=None, use_manual=False,
                               manual_xlim=None, manual_ylim=None,
                               xpad=5.0, ypad=3.0):
    if use_manual:
        return manual_xlim, manual_ylim

    all_x = []
    all_y = []

    for df in dfs:
        if df is not None and df["x"].size > 0:
            all_x.extend(df["x"])
            all_y.extend(df["y"])

    if clim_curves is not None:
        for c in clim_curves:
            if c is not None:
                all_x.extend(c["x"])
                all_y.extend(c["y"])

    all_x = np.asarray(all_x, dtype=float)
    all_y = np.asarray(all_y, dtype=float)

    if all_x.size == 0 or all_y.size == 0:
        return (60, 150), (60, 170)

    xmin, xmax = np.nanmin(all_x), np.nanmax(all_x)
    ymin, ymax = np.nanmin(all_y), np.nanmax(all_y)

    return (xmin - xpad, xmax + xpad), (ymin - ypad, ymax + ypad)


# ============================================================
# Plotting
# ============================================================
def draw_one_panel(ax, df, clim_curve, panel_title, xlim, ylim):
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # 右侧 Y 轴用于 climatology curve
    ax2 = ax.twinx()
    ax2.set_ylim(ylim)
    ax2.set_ylabel(r"O$_3$ climatology (DU)", color=CLIM_COLOR)
    ax2.tick_params(axis='y', colors=CLIM_COLOR)

    if df is None or len(df["x"]) == 0:
        ax.set_title(panel_title, fontweight="bold")
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel("Final Warming Date")
        ax.set_ylabel(r"MA min O$_3$ (DU)")
        ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))
        return

    low_mask = df["is_low25"]
    bg_mask = ~low_mask

    # scatter points
    if np.any(bg_mask):
        ax.scatter(
            df["x"][bg_mask], df["y"][bg_mask],
            s=BG_SIZE, color=BG_COLOR, alpha=BG_ALPHA,
            edgecolors="none", zorder=2
        )

    if np.any(low_mask):
        ax.scatter(
            df["x"][low_mask], df["y"][low_mask],
            s=LOW25_SIZE, color=LOW25_COLOR, alpha=LOW25_ALPHA,
            edgecolors="k", linewidths=0.4, zorder=5
        )

    # extreme points
    legend_entries = []
    if MARK_EXTREMES:
        idx_ext = select_panel_extremes(
            df,
            topn=EXTREME_TOPN,
            extreme_by=EXTREME_BY,
            ascending=EXTREME_ASCENDING
        )
        legend_entries = overlay_panel_extremes(ax, df, idx_ext)

    # FWD mean ± sigma band + mean line
    mean_fwd = np.nanmean(df["x"])
    std_fwd  = np.nanstd(df["x"], ddof=0)
    

    # mean ± 1σ band
    ax.axvspan(mean_fwd - std_fwd, mean_fwd + std_fwd,
            color=FWD_BAND_COLOR, alpha=0.18, zorder=0)

    # mean line
    ax.axvline(mean_fwd, color=FWD_MEAN_COLOR, linestyle="-",
            linewidth=1.4, alpha=0.95, zorder=1)

    # ±1σ boundary lines
    ax.axvline(mean_fwd - std_fwd, color=FWD_MEAN_COLOR, linestyle="--",
            linewidth=1.0, alpha=0.9, zorder=1)
    ax.axvline(mean_fwd + std_fwd, color=FWD_MEAN_COLOR, linestyle="--",
            linewidth=1.0, alpha=0.9, zorder=1)

    # climatology curve on right y-axis
    if clim_curve is not None:
        ax2.plot(
            clim_curve["x"], clim_curve["y"],
            color=CLIM_COLOR, linestyle=CLIM_LINESTYLE,
            linewidth=CLIM_LINEWIDTH, alpha=CLIM_ALPHA, zorder=1
        )

    # statistics
    n_all = len(df["x"])
    if n_all >= 3:
        try:
            r_all, p_all = pearsonr(df["x"], df["y"])
            txt_all = f"ALL\nr = {r_all:.3f}\np = {p_all:.2e}\nN = {n_all}"
        except Exception:
            txt_all = f"ALL\nN = {n_all}"
    else:
        txt_all = f"ALL\nN = {n_all}"

    ax.text(
        0.04, 0.96, txt_all,
        transform=ax.transAxes, va="top", ha="left", fontsize=9.2, color="k",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor="0.7")
    )

    n_low = int(np.sum(low_mask))
    if n_low >= 3:
        try:
            r_low, p_low = pearsonr(df["x"][low_mask], df["y"][low_mask])
            txt_low = f"LOW25\nr = {r_low:.3f}\np = {p_low:.2e}\nN = {n_low}"
        except Exception:
            txt_low = f"LOW25\nN = {n_low}"
    else:
        txt_low = f"LOW25\nN = {n_low}"

    ax.text(
        0.96, 0.96, txt_low,
        transform=ax.transAxes, va="top", ha="right", fontsize=9.2, color=LOW25_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor=LOW25_COLOR)
    )

    # legend
    handles = []
    labels = []

    for i, label, c, part in legend_entries:
        if part == "BWCN":
            h = Line2D(
                [0], [0],
                marker="o",
                color="none",
                markeredgecolor=c,
                markerfacecolor="none",
                markeredgewidth=1.2,
                markersize=5.0,
                linewidth=0
            )
        else:
            h = Line2D(
                [0], [0],
                marker="o",
                color="none",
                markeredgecolor="k",
                markerfacecolor=c,
                markeredgewidth=0.6,
                markersize=5.0,
                linewidth=0
            )
        handles.append(h)
        labels.append(f"Top{i+1}: {label}")

    handles.extend([
        Line2D([0], [0], color=CLIM_COLOR, lw=CLIM_LINEWIDTH, ls=CLIM_LINESTYLE),
        Line2D([0], [0], color=FWD_MEAN_COLOR, lw=FWD_MEAN_LINEWIDTH, ls=FWD_MEAN_LINESTYLE),
        Patch(facecolor=FWD_BAND_COLOR, edgecolor="none", alpha=FWD_BAND_ALPHA),
    ])
    labels.extend([
        "O$_3$ climatology",
        f"Mean FWD: {doy_to_mmdd(mean_fwd, None)}",
        r"FWD $\pm 1\sigma$"
    ])

    ax.legend(
        handles, labels,
        loc="lower right",
        fontsize=4.8,
        frameon=False,
        handletextpad=0.35,
        borderpad=0.2,
        labelspacing=0.25
    )

    ax.set_title(panel_title, fontweight="bold")
    ax.set_xlabel("Final Warming Date")
    ax.set_ylabel(r"MA min O$_3$ (DU)")
    ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))


# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    print("Loading data...")

    # scatter data
    df_merra = build_scatter_df(
        FW_TXT_MERRA2,
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        os.path.join(DATA_BASE, "MERRA2", "low25pct_years_30_70hPa.txt"),
        part_name="MERRA2",
        is_bridge_case=False
    )

    df_bwcn = build_scatter_df(
        FW_TXT_BWCN,
        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "BWCN", "low25pct_years_30_70hPa.txt"),
        part_name="BWCN",
        is_bridge_case=False
    )

    df_b2000 = build_scatter_df(
        FW_TXT_B2000,
        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN", "low25pct_years_30_70hPa.txt"),
        part_name="B2000WCN",
        is_bridge_case=True
    )

    df_coupled = combine_dfs(df_bwcn, df_b2000)

    df_nocoupl = build_scatter_df(
        FW_TXT_NOCOUPL,
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"),
        part_name="B2000WCN_NOCOUPL",
        is_bridge_case=True
    )

    # climatology curves
    clim_merra = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years=None
    )

    clim_bwcn = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years=None
    )

    clim_b2000 = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years={BRIDGE_YEAR}
    )

    clim_coupled = average_climatology_curves([clim_bwcn, clim_b2000])

    clim_nocoupl = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years={BRIDGE_YEAR}
    )

    # global axes
    xlim, ylim = compute_global_axis_limits(
        [df_merra, df_coupled, df_nocoupl],
        clim_curves=None,
        use_manual=USE_MANUAL_AXIS_LIMITS,
        manual_xlim=MANUAL_XLIM,
        manual_ylim=MANUAL_YLIM
    )

    print("Global xlim =", xlim)
    print("Global ylim =", ylim)

    print("Plotting...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), dpi=FIG_DPI)
    plt.subplots_adjust(wspace=0.30)

    draw_one_panel(axes[0], df_merra,   clim_merra,   "MERRA2",             xlim, ylim)
    draw_one_panel(axes[1], df_coupled, clim_coupled, "B2000WCN (Coupled)", xlim, ylim)
    draw_one_panel(axes[2], df_nocoupl, clim_nocoupl, "B2000WCN NOCOUPL",   xlim, ylim)

    fig.suptitle("Scatter: MA min O$_3$ vs Final Warming Date (FWD)", fontsize=16, fontweight="bold", y=1.03)

    for ax in axes:
        for tick in ax.get_xticklabels():
            tick.set_rotation(45)

    save_path = os.path.join(OUT_DIR, "Scatter_FWD_vs_O3_MERRA2_B2000_NOCOUPL_extreme_climcurve_fwdband.png")
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    print(f"Plot saved successfully at:\n{save_path}")